# Recipe Generator — Notebook Monolitik (Tugas Besar IF5153)

Notebook ini menggabungkan **seluruh kode non-UI** repositori ke dalam satu file yang bisa
dibaca dan dijalankan end-to-end: preprocessing, ketiga pipeline, skrip data & pelatihan,
harness evaluasi/eksperimen, dan test suite. UI Streamlit (`app/`) tidak ikut digabung —
ia hanya mengimpor dari modul-modul yang sekarang didefinisikan di notebook ini.

**Pemetaan ke source** — setiap section merangkai file asli secara *verbatim* (hanya
import lintas-modul, guard CLI, dan `Path(__file__)` yang disesuaikan):

| Section | Sumber |
|---|---|
| 1. Preprocessing | `src/preprocessing/` |
| 2. Pipeline 1 | `src/pipeline1/` |
| 3. Pipeline 2 | `src/pipeline2/` |
| 4. Pipeline 3 | `src/pipeline3/` |
| 5. Skrip data & pelatihan | `scripts/` |
| 6. Evaluasi & eksperimen | `evaluation/` |
| 7. Test suite | `tests/` |
| 8. Demo end-to-end | — |

**Cara pakai**
1. Jalankan dengan kernel `.venv` repositori ini dari posisi mana pun (notebook
   otomatis mencari root repositori dari `Path.cwd()`).
2. *Warm path* (default): semua flag build `False` — notebook memuat artifact yang sudah
   ada di `artifacts/` dan `data/processed/` sehingga cepat.
3. *Cold path*: nyalakan flag `RUN_*` terkait untuk membangun ulang dari data mentah
   (download USDA/RecipeNLG, build korpus, encode dense, latih NER).
4. Testing dan eksperimen dijalankan langsung di notebook (section 6–7).

**Penyesuaian yang dilakukan saat penggabungan** (perilaku tidak berubah):
- Registry Pipeline 2 di-rename (`get_approach` -> `get_retrieval_approach`) agar tidak
  menimpa registry Pipeline 1 di satu namespace.
- `_split_compound` duplikat (identik) di `rule.py` dibuang; konstanta dengan nilai sama
  (`ROOT`, `REPO_ROOT`, `SAMPLE_CSV`, `TOKEN`, `MAX_LENGTH`, `DEFAULT_ARTIFACTS_DIR`)
  didefinisikan sekali.
- Konstanta nama-sama namun nilai berbeda di-rename per modul: `PROCESSED` ->
  `PROCESSED_WORDS`, `OUT` -> `REFERENCE_OUT_DIR`/`BASELINE_OUT`/`QUERIES_OUT`,
  `SAMPLE` -> `COVERAGE_SAMPLE`, `DEFAULT_CORPUS` -> `EVAL_SAMPLE_CORPUS`,
  `DEFAULT_OUT_DIR` -> `NER_DATA_OUT_DIR`.
- `main()` CLI menjadi fungsi `run_*` berparameter; fixture pytest diganti variabel biasa.


## 0. Setup — imports, konstanta, dan flag

Semua import eksternal dari seluruh modul di-hoist ke sini (verbatim, dedup). Konstanta
`ROOT`, `REPO_ROOT`, dan `SAMPLE_CSV` berasal dari definisi identik di berbagai file.


In [ ]:
# Stdlib (hoisted dari seluruh modul)
from collections import Counter
from collections import defaultdict
from dataclasses import dataclass
from dataclasses import dataclass, field
from dataclasses import replace
from fractions import Fraction
from functools import lru_cache
from pathlib import Path
from typing import Any
from typing import Any, Iterable, Mapping, Sequence
from typing import Any, Mapping
from typing import Any, Mapping, Sequence
from typing import Iterable
from typing import Iterable, Mapping
import ast
import csv
import difflib
import io
import json
import math
import pickle
import random
import re
import statistics
import sys
import time
import unittest
import urllib.request
import zipfile

# Third-party
from scipy.sparse import csr_matrix
from scipy.sparse import csr_matrix, diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize as sk_normalize
import numpy as np
import pandas as pd
import pytest
import tempfile  # pengganti fixture pytest tmp_path


In [ ]:
# Cari root repositori dari cwd (kernel Jupyter menjalankan notebook dari dir notebook-nya)
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), "root repositori tidak ditemukan dari cwd"
REPO_ROOT = ROOT                      # alias gaya penamaan scripts/tests
SAMPLE_CSV = ROOT / "data" / "samples" / "recipes_sample.csv"
print(f"repo root: {ROOT}")

In [ ]:
# ---- Flag cold path / evaluasi -------------------------------------------
RUN_DOWNLOAD_DATA = False     # scripts/download_raw_data.py    (USDA + sampel RecipeNLG)
RUN_BUILD_REFERENCE = False   # scripts/build_calorie_reference.py
RUN_BUILD_BASELINE = False    # scripts/build_calorie_baseline.py
RUN_BUILD_CORPUS = False      # src/preprocessing/build_corpus.py (RecipeNLG -> parquet)
RUN_BUILD_QUERIES = False     # scripts/build_synthetic_queries.py
RUN_CHECK_COVERAGE = False    # scripts/check_ingredient_coverage.py
RUN_GENERATE_NER_DATA = False # scripts/generate_ner_training_data.py
RUN_TRAIN_NER = False         # scripts/train_ner.py  (DistilBERT, puluhan menit di MPS/CPU)
RUN_FIT_TFIDF = False         # src/pipeline2/vectorizer.py main
RUN_ENCODE_DENSE = False      # scripts/encode_dense_corpus.py (sentence-transformers)
RUN_SIMULATE = False          # scripts/simulate_pipeline2.py
RUN_EVALUATIONS = True        # harness evaluasi section 6 (read-only, hitungan menit)

In [ ]:
# ---- Warm path: cek ketersediaan artifact --------------------------------
_expected = {
    "artifacts/ingredient_vocabulary.pkl":  "vocab Pipeline 1 (wajib untuk test & demo)",
    "data/processed/recipes.parquet":       "korpus terproses (wajib untuk retrieval)",
    "artifacts/tfidf_vectorizer.pkl":       "vektor TF-IDF (Pipeline 2)",
    "artifacts/recipe_vectors.pkl":         "matriks TF-IDF resep (Pipeline 2)",
    "artifacts/dense_vectors.npy":          "dense retrieval (opsional)",
    "artifacts/ner_model/config.json":      "model NER terlatih (opsional)",
}
for rel, note in _expected.items():
    p = ROOT / rel
    print(f"{'OK  ' if p.exists() else 'MISS'} {rel:<42} {note}")

## 1. Preprocessing — `src/preprocessing/`

Lapisan paling bawah: normalisasi ingredient, parsing kalimat kuantitas, estimasi kalori,
dan pembangunan korpus terproses dari RecipeNLG.


**`src/preprocessing/ingredient_normalizer.py`**

Canonical ingredient normalization shared by every pipeline.  Two collaborators live here on purpose:  * ``Vocabulary`` — the closed query-time vocabulary used by Pipeline 1. Only ingredients with a calorie treatment in ``data/processed/`` plus the curated alias table can be recognised; longest surface phrase wins. * ``IngredientNormalizer`` — the offline normalizer used by corpus preprocessing and Pipeline 2 (alias resolution, suffix-trim scanning, recipe vocabulary).  Both canonicalize ingredient strings through ``normalize_ingredient`` (fold + head-noun singularization). ``normalize_text`` stays a pure query-time fold and must not singularize: it feeds whole user queries, including trailing exclusion clauses such as "no cheese" that would corrupt under naive stemming. ``~`` is kept there because the calorie constraint grammar reads ``~500`` as an approximate target. Rules mirror ``scripts/ingredient_tables.py``; keep the two in sync through ``data/processed/ingredient_aliases.csv``.


In [ ]:
DEFAULT_PROCESSED_DIR = ROOT / "data" / "processed"
DATA_DIR = DEFAULT_PROCESSED_DIR
DEFAULT_VOCAB_PATH = ROOT / "artifacts" / "ingredient_vocabulary.pkl"

CALORIE_TABLES = ("calorie_baseline.csv", "supplement_calories.csv", "zero_calorie_ingredients.csv")
STAPLE_TABLE = "pantry_staples.csv"
ALIAS_TABLE = "ingredient_aliases.csv"

def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))

def normalize_text(text: str) -> str:
    """Lowercase, drop apostrophes, collapse everything that is not a word or ~."""
    text = text.lower().replace("\u2019", "'").replace("'", "")
    text = re.sub(r"[^a-z0-9~]+", " ", text)
    return text.strip()

def singularize(word: str) -> str:
    # suffix rules, not WordNet lemmatization; the closed vocabulary and
    # alias table hide the gap. Swap in NLTK when the vocabulary goes open-domain.
    if len(word) < 4 or word.endswith(("ss", "us", "is")):
        return word
    if word.endswith("ies"):
        return word[:-3] + "y"
    if word.endswith("oes"):
        return word[:-2]
    if word.endswith(("ses", "xes", "zes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s"):
        return word[:-1]
    return word

def normalize_ingredient(name: str) -> str:
    """Normalize an ingredient phrase; only the head noun is plural-folded."""
    tokens = normalize_text(name).split()
    if not tokens:
        return ""
    tokens[-1] = singularize(tokens[-1])
    return " ".join(tokens)

class Vocabulary:
    """Surface form -> canonical ingredient, longest match wins."""

    def __init__(self, surface_to_canonical: dict[str, str]) -> None:
        self.surface_to_canonical = surface_to_canonical
        self.max_tokens = max((len(s.split()) for s in surface_to_canonical), default=1)

    @classmethod
    def load(cls, data_dir: Path = DATA_DIR) -> "Vocabulary":
        canonical = set()
        for table in CALORIE_TABLES + (STAPLE_TABLE,):
            canonical |= {normalize_ingredient(row["ingredient"]) for row in read_csv_rows(data_dir / table)}
        aliases = {
            normalize_ingredient(row["alias"]): normalize_ingredient(row["canonical"])
            for row in read_csv_rows(data_dir / ALIAS_TABLE)
        }
        surface = {name: aliases.get(name, name) for name in canonical}
        surface.update(aliases)
        return cls(surface)

    def lookup(self, key: str) -> str | None:
        return self.surface_to_canonical.get(key)

    def canonical(self, surface: str) -> str | None:
        return self.lookup(normalize_ingredient(surface))

class IngredientNormalizer:
    """Offline alias/qualifier resolution for corpus preprocessing and Pipeline 2."""

    def __init__(self, processed_dir: Path | str = DEFAULT_PROCESSED_DIR) -> None:
        self.processed_dir = Path(processed_dir)
        self.baseline = {
            normalize_ingredient(r["ingredient"])
            for r in read_csv_rows(self.processed_dir / "calorie_baseline.csv")
        }
        self.supplement = {
            normalize_ingredient(r["ingredient"])
            for r in read_csv_rows(self.processed_dir / "supplement_calories.csv")
        }
        self.zero = {
            normalize_ingredient(r["ingredient"])
            for r in read_csv_rows(self.processed_dir / "zero_calorie_ingredients.csv")
        }
        self.aliases = {
            normalize_ingredient(r["alias"]): normalize_ingredient(r["canonical"])
            for r in read_csv_rows(self.processed_dir / ALIAS_TABLE)
        }
        self.known = self.baseline | self.supplement | self.zero

    def resolve(self, term: str) -> str | None:
        key = normalize_ingredient(term)
        if not key:
            return None
        key = self.aliases.get(key, key)
        if key in self.known:
            return key
        tokens = key.split()
        for start in range(1, len(tokens)):
            candidate = " ".join(tokens[start:])
            candidate = self.aliases.get(candidate, candidate)
            if candidate in self.known:
                return candidate
        return None

    def canonicalize(self, term: str, drop_qualifiers: bool = True) -> str:
        """Normalize, alias-map, and (optionally) drop leading qualifiers.

        Same fallback contract as :meth:`normalize_ingredient`: returns the
        alias-mapped normalized form when nothing known matches.
        """
        key = normalize_ingredient(term)
        if not key:
            return ""
        mapped = self.aliases.get(key, key)
        if not self.known or mapped in self.known or not drop_qualifiers:
            return mapped
        tokens = key.split()
        for start in range(1, len(tokens)):
            candidate = " ".join(tokens[start:])
            candidate_mapped = self.aliases.get(candidate, candidate)
            if candidate_mapped in self.known:
                return candidate_mapped
        return mapped

    def canonicalize_list(
        self,
        terms: Iterable[str],
        drop_qualifiers: bool = True,
    ) -> list[str]:
        """Deduplicated, order-preserving list of canonical ingredient names."""
        seen: dict[str, None] = {}
        for term in terms:
            canonical = self.canonicalize(term, drop_qualifiers=drop_qualifiers)
            if canonical:
                seen.setdefault(canonical, None)
        return list(seen)

    def normalize_ingredient(self, term: str) -> str:
        return self.canonicalize(term, drop_qualifiers=True)

    def extract_canonical_ingredients(
        self,
        raw_terms: list[str],
        known_only: bool = False,
    ) -> list[str]:
        seen: dict[str, None] = {}
        for term in raw_terms:
            if not isinstance(term, str):
                continue
            canonical = self.resolve(term) if known_only else self.normalize_ingredient(term)
            if canonical:
                seen.setdefault(canonical, None)
        return list(seen)

    def build_vocabulary(self, recipe_ingredient_lists: list[list[str]]) -> list[str]:
        vocab: set[str] = set(self.known)
        for ingredients in recipe_ingredient_lists:
            for item in ingredients:
                if item:
                    vocab.add(item)
        return sorted(vocab)

    @staticmethod
    def save_vocabulary(vocabulary: list[str], path: Path | str = DEFAULT_VOCAB_PATH) -> Path:
        out_path = Path(path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with out_path.open("wb") as fh:
            pickle.dump(vocabulary, fh)
        return out_path

    @staticmethod
    def load_vocabulary(path: Path | str = DEFAULT_VOCAB_PATH) -> list[str]:
        with Path(path).open("rb") as fh:
            return pickle.load(fh)


**`src/preprocessing/ingredient_parser.py`**


In [ ]:
UNICODE_FRACTIONS = {
    "\u00bc": " 1/4",
    "\u00bd": " 1/2",
    "\u00be": " 3/4",
    "\u2150": " 1/7",
    "\u2151": " 1/9",
    "\u2152": " 1/10",
    "\u2153": " 1/3",
    "\u2154": " 2/3",
    "\u2155": " 1/5",
    "\u2156": " 2/5",
    "\u2157": " 3/5",
    "\u2158": " 4/5",
    "\u2159": " 1/6",
    "\u215a": " 5/6",
    "\u215b": " 1/8",
    "\u215c": " 3/8",
    "\u215d": " 5/8",
    "\u215e": " 7/8",
}

UNIT_ALIASES = {
    "c": "cup",
    "cup": "cup",
    "cups": "cup",
    "tbsp": "tbsp",
    "tbs": "tbsp",
    "tb": "tbsp",
    "tablespoon": "tbsp",
    "tablespoons": "tbsp",
    "tsp": "tsp",
    "ts": "tsp",
    "teaspoon": "tsp",
    "teaspoons": "tsp",
    "fl oz": "fl_oz",
    "fluid ounce": "fl_oz",
    "fluid ounces": "fl_oz",
    "oz": "oz",
    "ounce": "oz",
    "ounces": "oz",
    "lb": "lb",
    "lbs": "lb",
    "pound": "lb",
    "pounds": "lb",
    "g": "g",
    "gm": "g",
    "gms": "g",
    "gram": "g",
    "grams": "g",
    "kg": "kg",
    "kilogram": "kg",
    "kilograms": "kg",
    "ml": "ml",
    "milliliter": "ml",
    "milliliters": "ml",
    "l": "l",
    "liter": "l",
    "liters": "l",
    "pt": "pt",
    "pts": "pt",
    "pint": "pt",
    "pints": "pt",
    "qt": "qt",
    "qts": "qt",
    "quart": "qt",
    "quarts": "qt",
    "gal": "gal",
    "gallon": "gal",
    "gallons": "gal",
    "stick": "stick",
    "sticks": "stick",
    "clove": "clove",
    "cloves": "clove",
    "slice": "slice",
    "slices": "slice",
    "piece": "piece",
    "pieces": "piece",
    "whole": "whole",
    "can": "can",
    "cans": "can",
    "jar": "jar",
    "jars": "jar",
    "package": "package",
    "packages": "package",
    "pkg": "pkg",
    "pkgs": "pkg",
    "box": "box",
    "boxes": "box",
    "bunch": "bunch",
    "bunches": "bunch",
    "head": "head",
    "heads": "head",
    "stalk": "stalk",
    "stalks": "stalk",
    "sprig": "sprig",
    "sprigs": "sprig",
    "dash": "dash",
    "dashes": "dash",
    "pinch": "pinch",
    "pinches": "pinch",
    "drop": "drop",
    "drops": "drop",
    "handful": "handful",
    "handfuls": "handful",
    "medium": "medium",
    "med": "medium",
    "large": "large",
    "lg": "large",
    "small": "small",
    "sm": "small",
    "bag": "package",
    "bags": "package",
    "carton": "package",
    "cartons": "package",
    "container": "package",
    "containers": "package",
    "pack": "package",
    "packs": "package",
}

PREP_WORDS = {
    "chopped",
    "diced",
    "minced",
    "sliced",
    "drained",
    "rinsed",
    "peeled",
    "seeded",
    "grated",
    "shredded",
    "crushed",
    "melted",
    "softened",
    "beaten",
    "sifted",
    "divided",
    "optional",
    "trimmed",
    "halved",
    "quartered",
    "cubed",
    "mashed",
    "cooked",
    "uncooked",
    "boiled",
    "toasted",
    "roasted",
    "freshly",
    "finely",
    "coarsely",
    "thinly",
    "lightly",
    "well",
    "about",
    "approximately",
    "approx",
}

NUMBER_PART = r"(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)"
QTY_RANGE_RE = re.compile(
    rf"^\s*(?P<first>{NUMBER_PART})(?:\s*(?:-|to|or)\s*(?P<second>{NUMBER_PART}))?\b",
    re.IGNORECASE,
)
PAREN_MEASURE_RE = re.compile(
    rf"\(\s*(?P<qty>{NUMBER_PART})\s*(?P<unit>[a-zA-Z\. ]+?)\s*\)",
    re.IGNORECASE,
)
SORTED_UNITS = sorted(UNIT_ALIASES.keys(), key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in SORTED_UNITS)
LEADING_UNIT_RE = re.compile(
    rf"^\s*(?P<unit>{UNIT_PATTERN})\.?(?:\s+of\b|\b)",
    re.IGNORECASE,
)
CONTAINER_UNITS = {"can", "jar", "package", "pkg", "box", "stick"}

def _replace_unicode_fractions(text: str) -> str:
    for symbol, replacement in UNICODE_FRACTIONS.items():
        if symbol in text:
            text = text.replace(symbol, replacement)
    return text

def _parse_number(token: str) -> float | None:
    token = token.strip()
    if not token:
        return None
    parts = token.split()
    total = 0.0
    try:
        for part in parts:
            if "/" in part:
                total += float(Fraction(part))
            else:
                total += float(part)
        return total
    except (ValueError, ZeroDivisionError):
        return None

def _leading_unit(text: str) -> tuple[str | None, str]:
    """Match a leading unit token; returns (unit, remaining text).

    Case matters for the bare abbreviations: uppercase ``T`` = tablespoon,
    lowercase ``t`` = teaspoon (RecipeNLG convention).
    """
    match = LEADING_UNIT_RE.match(text)
    if not match:
        return None, text
    raw_unit = re.sub(r"\s+", " ", match.group("unit").strip())
    if raw_unit == "T":
        unit = "tbsp"
    elif raw_unit == "t":
        unit = "tsp"
    else:
        unit = UNIT_ALIASES.get(raw_unit.lower())
    return unit, text[match.end():].strip()

def _clean_ingredient_name(text: str) -> str:
    text = re.sub(r"\([^)]*\)", " ", text)
    text = text.split(",")[0]
    text = re.sub(r"\bor\b.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bto taste\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-zA-Z0-9\s\-']", " ", text)
    tokens = [t.lower().strip("'-") for t in text.split()]
    while tokens and tokens[0] in {"of", "a", "an", "the"}:
        tokens.pop(0)
    filtered = [t for t in tokens if t and t not in PREP_WORDS]
    if filtered:
        return " ".join(filtered)
    return " ".join(t for t in tokens if t)

def parse_ingredient(raw: str) -> dict[str, float | str | None]:
    original = raw.strip()
    working = _replace_unicode_fractions(original)
    working = re.sub(r"\s+", " ", working).strip()

    quantity: float | None = None
    unit: str | None = None

    # Phrasing like "juice of 1/2 lemon" — the quantity attaches to the noun
    juice_match = re.match(rf"^juice\s+of\s+({NUMBER_PART})\s+(.+)$", working, flags=re.IGNORECASE)
    if juice_match:
        quantity = _parse_number(juice_match.group(1))
        unit = "whole"
        working = juice_match.group(2).split(",")[0].strip()

    qty_match = QTY_RANGE_RE.match(working)
    if qty_match:
        first_val = _parse_number(qty_match.group("first"))
        second_raw = qty_match.group("second")
        if first_val is not None and second_raw:
            second_val = _parse_number(second_raw)
            quantity = (first_val + second_val) / 2.0 if second_val is not None else first_val
        else:
            quantity = first_val
        working = working[qty_match.end():].strip()

    paren_qty: float | None = None
    paren_unit: str | None = None
    paren_match = PAREN_MEASURE_RE.search(working)
    if paren_match:
        candidate_unit = paren_match.group("unit").strip().rstrip(".").lower()
        candidate_unit = re.sub(r"\s+", " ", candidate_unit)
        if candidate_unit in UNIT_ALIASES:
            paren_qty = _parse_number(paren_match.group("qty"))
            paren_unit = UNIT_ALIASES[candidate_unit]
            working = (working[: paren_match.start()] + " " + working[paren_match.end():]).strip()

    unit, working = _leading_unit(working)

    if paren_qty is not None and paren_unit is not None:
        if unit in CONTAINER_UNITS or unit is None:
            multiplier = quantity if quantity is not None else 1.0
            quantity = multiplier * paren_qty
            unit = paren_unit

    if unit is None:
        second_unit, working = _leading_unit(working)
        if second_unit is not None:
            unit = second_unit

    ingredient_name = _clean_ingredient_name(working)
    if not ingredient_name:
        ingredient_name = _clean_ingredient_name(original)

    return {
        "raw": original,
        "quantity": quantity,
        "unit": unit,
        "ingredient": ingredient_name,
    }

def parse_ingredients(raw_list: list[str]) -> list[dict[str, float | str | None]]:
    return [parse_ingredient(item) for item in raw_list if isinstance(item, str) and item.strip()]


**`src/preprocessing/calorie_estimator.py`**

Recipe calorie estimator (PRD Sections 14-16) — the single one in the repo.  Merges the former ``CalorieMapper`` (dict API used by corpus preprocessing) and ``RecipeCalorieEstimator`` (typed API used by Pipeline 3, evaluation, UI) into one engine over the committed tables in ``data/processed/``:  - ``calorie_baseline.csv``         (deterministic canonical kcal/100g + fdc_id) - ``supplement_calories.csv``      (fallback kcal/100g) - ``zero_calorie_ingredients.csv`` (zero/negligible calorie items) - ``calorie_reference.csv``        (7,793 USDA SR Legacy foods for fallback matching) - ``unit_grams_reference.csv``     (per-fdc_id household unit -> grams) - ``unit_grams_fallback.csv``      (generic unit -> grams)  Lookup priority: zero -> baseline -> reference -> supplement -> fuzzy. Reference rows collapse to the median kcal per base ingredient (deterministic). Values returned are *Estimated Calories*, not authoritative nutritional labels (PRD Risk 2).


In [ ]:
# Scaled volume units folded to cups before the gram tables are consulted.
VOLUME_TO_CUPS = {
    "pt": 2.0,
    "qt": 4.0,
    "gal": 16.0,
}

# Bare-count defaults when no food-specific or fallback unit row exists.
DEFAULT_ITEM_GRAMS = {
    "small": 75.0,
    "medium": 110.0,
    "large": 150.0,
    "whole": 100.0,
}

_SERVINGS_PATTERNS = [
    re.compile(r"\bserves\s+(\d+)(?:\s*(?:-|to)\s*(\d+))?\b", re.IGNORECASE),
    re.compile(r"\bmakes\s+(\d+)(?:\s*(?:-|to)\s*(\d+))?\s+(?:servings?|portions?)\b", re.IGNORECASE),
    re.compile(r"\byields?\s*:?\s*(\d+)(?:\s*(?:-|to)\s*(\d+))?\s*(?:servings?|portions?)?\b", re.IGNORECASE),
    re.compile(r"\b(\d+)(?:\s*(?:-|to)\s*(\d+))?\s+servings?\b", re.IGNORECASE),
]

@dataclass
class ParsedIngredient:
    """Structured parse of a raw RecipeNLG ingredient string (PRD Section 14)."""

    raw: str
    quantity: float | None
    unit: str | None
    ingredient: str

def parse_ingredient_line(raw_line: str) -> ParsedIngredient:
    """Parse a raw ingredient line into the typed ``ParsedIngredient`` shape."""
    parsed = parse_ingredient(raw_line)
    return ParsedIngredient(
        raw=str(parsed["raw"]),
        quantity=parsed["quantity"],
        unit=parsed["unit"],
        ingredient=str(parsed["ingredient"] or ""),
    )

@dataclass
class IngredientCalorieItem:
    """Per-ingredient calorie estimation result."""

    raw: str
    quantity: float | None
    unit: str | None
    ingredient: str
    canonical: str | None
    grams: float | None
    kcal_per_100g: float | None
    calories: float | None
    match_type: str  # zero | baseline | reference | supplement | fuzzy | unmatched

@dataclass
class CalorieEstimate:
    """Estimated calorie summary for a recipe (PRD Section 16)."""

    total_calories: float | None
    calories_per_serving: float | None
    estimated_servings: int
    servings_source: str  # parsed | heuristic | single
    matched_count: int
    total_count: int
    items: list[IngredientCalorieItem] = field(default_factory=list)

    @property
    def match_ratio(self) -> float:
        return self.matched_count / self.total_count if self.total_count else 0.0

def extract_servings(directions: Iterable[str], title: str = "") -> int | None:
    """Extract stated serving count from recipe directions/title when present."""
    combined = " ".join([title, *directions])
    for pattern in _SERVINGS_PATTERNS:
        m = pattern.search(combined)
        if m:
            n1 = int(m.group(1))
            n2 = int(m.group(2)) if m.group(2) else n1
            servings = round((n1 + n2) / 2.0)
            if 1 <= servings <= 50:
                return servings
    return None

class RecipeCalorieEstimator:
    """Estimates recipe calories from raw ingredient lines using data/processed/ tables.

    Exposes both historical entry points:

    - typed:  :meth:`estimate_recipe` -> ``CalorieEstimate`` (Pipeline 3, UI, evaluation)
    - dict:   :meth:`estimate_recipe_calories` / :meth:`estimate_ingredient_calories`
      (corpus preprocessing)
    """

    def __init__(
        self,
        processed_dir: Path | None = None,
        normalizer: IngredientNormalizer | None = None,
        enable_fuzzy: bool = True,
        fuzzy_cutoff: float = 0.84,
    ) -> None:
        self.processed_dir = Path(processed_dir) if processed_dir is not None else DEFAULT_PROCESSED_DIR
        self.normalizer = normalizer or IngredientNormalizer(self.processed_dir)
        self.enable_fuzzy = enable_fuzzy
        self.fuzzy_cutoff = fuzzy_cutoff

        self.baseline: dict[str, tuple[float, str | None]] = {}
        self.reference: dict[str, tuple[float, str | None]] = {}
        self.supplement: dict[str, tuple[float, str | None]] = {}
        self.zero_calorie: set[str] = set()
        self.unit_grams_by_fdc: dict[tuple[str, str], float] = {}
        self.unit_grams_by_ingredient: dict[tuple[str, str], float] = {}
        self.unit_grams_fallback: dict[str, float] = {}

        self._fuzzy_keys: list[str] = []
        self._lookup_cache: dict[str, dict[str, float | str | None] | None] = {}
        self._load_tables()

    def _load_tables(self) -> None:
        processed = self.processed_dir
        path = processed / "calorie_baseline.csv"
        if path.exists():
            for row in read_csv_rows(path):
                key = normalize_ingredient(row.get("ingredient", ""))
                if key:
                    self.baseline[key] = (float(row["kcal_per_100g"]), row.get("fdc_id") or None)

        path = processed / "calorie_reference.csv"
        if path.exists():
            # One kcal per base ingredient: median over its USDA variants
            # (robust against outlier descriptions; deterministic).
            buckets: dict[str, list[tuple[float, str]]] = {}
            for row in read_csv_rows(path):
                key = normalize_ingredient(row.get("base_ingredient", ""))
                if not key:
                    continue
                try:
                    kcal = float(row["kcal_per_100g"])
                except (ValueError, KeyError):
                    continue
                buckets.setdefault(key, []).append((kcal, row.get("fdc_id") or None))
            for key, values in buckets.items():
                values.sort(key=lambda v: v[0])
                self.reference[key] = values[len(values) // 2]

        path = processed / "supplement_calories.csv"
        if path.exists():
            for row in read_csv_rows(path):
                key = normalize_ingredient(row.get("ingredient", ""))
                if key:
                    self.supplement[key] = (float(row["kcal_per_100g"]), None)

        path = processed / "zero_calorie_ingredients.csv"
        if path.exists():
            self.zero_calorie = {
                normalize_ingredient(row.get("ingredient", "")) for row in read_csv_rows(path)
            } - {""}

        path = processed / "unit_grams_reference.csv"
        if path.exists():
            for row in read_csv_rows(path):
                fdc_id = row.get("fdc_id", "")
                ing = normalize_ingredient(row.get("ingredient", ""))
                unit = row.get("unit", "").strip().lower()
                try:
                    grams = float(row["grams_per_unit"])
                except (ValueError, KeyError):
                    continue
                if fdc_id and unit:
                    self.unit_grams_by_fdc[(fdc_id, unit)] = grams
                if ing and unit:
                    self.unit_grams_by_ingredient.setdefault((ing, unit), grams)

        path = processed / "unit_grams_fallback.csv"
        if path.exists():
            for row in read_csv_rows(path):
                unit = row.get("unit", "").strip().lower()
                try:
                    grams = float(row["grams"])
                except (ValueError, KeyError):
                    continue
                if unit:
                    self.unit_grams_fallback[unit] = grams

        self._fuzzy_keys = sorted(set(self.baseline) | set(self.supplement) | set(self.reference))

    def lookup_calorie_per_100g(self, ingredient: str) -> dict[str, float | str | None] | None:
        """Match an ingredient phrase to kcal/100g; ``None`` when unmatched.

        Candidate keys: alias-mapped full form, raw normalized form, then
        qualifier-dropped suffixes (each alias-mapped too). Tables are consulted
        in priority order zero -> baseline -> reference -> supplement, then a
        fuzzy match across all three kcal tables.
        """
        norm = normalize_ingredient(ingredient)
        if not norm:
            return None
        if norm in self._lookup_cache:
            return self._lookup_cache[norm]
        result = self._lookup_uncached(norm)
        self._lookup_cache[norm] = result
        return result

    def _lookup_uncached(self, norm: str) -> dict[str, float | str | None] | None:
        aliases = self.normalizer.aliases
        candidates: list[str] = []
        for cand in (aliases.get(norm, norm), norm):
            if cand not in candidates:
                candidates.append(cand)
        tokens = norm.split()
        for start in range(1, len(tokens)):
            sub = " ".join(tokens[start:])
            for cand in (aliases.get(sub, sub), sub):
                if cand not in candidates:
                    candidates.append(cand)

        # Curated tables in ascending scope, then the generic USDA reference.
        for table, match_type in (
            (self.zero_calorie, "zero"),
            (self.baseline, "baseline"),
            (self.supplement, "supplement"),
            (self.reference, "reference"),
        ):
            for cand in candidates:
                if cand in table:
                    kcal, fdc_id = (0.0, None) if match_type == "zero" else table[cand]
                    return {
                        "canonical": cand,
                        "kcal_per_100g": kcal,
                        "fdc_id": fdc_id,
                        "match_type": match_type,
                    }

        if self.enable_fuzzy and self._fuzzy_keys:
            mapped = aliases.get(norm, norm)
            close = difflib.get_close_matches(mapped, self._fuzzy_keys, n=1, cutoff=self.fuzzy_cutoff)
            if close:
                matched = close[0]
                for table in (self.baseline, self.supplement, self.reference):
                    if matched in table:
                        kcal, fdc_id = table[matched]
                        return {
                            "canonical": matched,
                            "kcal_per_100g": kcal,
                            "fdc_id": fdc_id,
                            "match_type": "fuzzy",
                        }
        return None

    def to_grams(
        self,
        quantity: float | None,
        unit: str | None,
        canonical_ingredient: str,
        fdc_id: str | None = None,
    ) -> float | None:
        """Convert (quantity, unit) to grams: food-specific table -> fallback -> defaults."""
        if quantity is None and unit is None:
            return None
        qty = quantity if quantity is not None else 1.0
        if qty <= 0:
            return None

        if unit is None:
            if fdc_id and (fdc_id, "piece") in self.unit_grams_by_fdc:
                return qty * self.unit_grams_by_fdc[(fdc_id, "piece")]
            if (canonical_ingredient, "piece") in self.unit_grams_by_ingredient:
                return qty * self.unit_grams_by_ingredient[(canonical_ingredient, "piece")]
            return qty * DEFAULT_ITEM_GRAMS["whole"]

        norm_unit = unit.strip().lower()
        if norm_unit in VOLUME_TO_CUPS:
            qty *= VOLUME_TO_CUPS[norm_unit]
            norm_unit = "cup"

        if fdc_id and (fdc_id, norm_unit) in self.unit_grams_by_fdc:
            return qty * self.unit_grams_by_fdc[(fdc_id, norm_unit)]
        if (canonical_ingredient, norm_unit) in self.unit_grams_by_ingredient:
            return qty * self.unit_grams_by_ingredient[(canonical_ingredient, norm_unit)]
        if norm_unit in self.unit_grams_fallback:
            return qty * self.unit_grams_fallback[norm_unit]
        if norm_unit in DEFAULT_ITEM_GRAMS:
            return qty * DEFAULT_ITEM_GRAMS[norm_unit]
        return qty * self.unit_grams_fallback.get("whole", 100.0)

    def estimate_ingredient_calories(
        self,
        raw_ingredient: str,
        ner_hint: str | None = None,
    ) -> dict[str, float | str | None]:
        """Estimate calories for one raw ingredient line (dict API)."""
        parsed = parse_ingredient(raw_ingredient)
        base = {
            "raw": parsed["raw"],
            "quantity": parsed["quantity"],
            "unit": parsed["unit"],
            "ingredient": parsed["ingredient"],
        }
        lookup = self.lookup_calorie_per_100g(str(parsed["ingredient"] or ""))
        if lookup is None and ner_hint:
            lookup = self.lookup_calorie_per_100g(str(ner_hint))
        if lookup is None:
            return {
                **base,
                "canonical": None,
                "grams": None,
                "kcal_per_100g": None,
                "calories": None,
                "match_type": "unmatched",
            }

        canonical = str(lookup["canonical"])
        kcal = float(lookup["kcal_per_100g"])
        fdc_id = lookup["fdc_id"]
        match_type = str(lookup["match_type"])

        if match_type == "zero":
            return {
                **base,
                "canonical": canonical,
                "grams": 0.0,
                "kcal_per_100g": 0.0,
                "calories": 0.0,
                "match_type": "zero",
            }

        grams = self.to_grams(
            parsed["quantity"],
            parsed["unit"],
            canonical,
            fdc_id=str(fdc_id) if fdc_id else None,
        )
        if grams is None:
            grams = self.unit_grams_fallback.get("whole", 100.0)
        return {
            **base,
            "canonical": canonical,
            "grams": round(grams, 2),
            "kcal_per_100g": kcal,
            "calories": round((grams / 100.0) * kcal, 2),
            "match_type": match_type,
        }

    def estimate_recipe_calories(
        self,
        raw_ingredients: list[str],
        ner_ingredients: list[str] | None = None,
    ) -> dict[str, object]:
        """Estimate total calories + coverage for a recipe (dict API, corpus build)."""
        total_calories = 0.0
        matched_count = 0
        details: list[dict[str, float | str | None]] = []

        for idx, raw in enumerate(raw_ingredients):
            if not isinstance(raw, str) or not raw.strip():
                continue
            hint = ner_ingredients[idx] if ner_ingredients and idx < len(ner_ingredients) else None
            entry = self.estimate_ingredient_calories(raw, ner_hint=hint)
            details.append(entry)
            if entry["calories"] is not None:
                total_calories += float(entry["calories"])
                matched_count += 1

        total_count = len(details)
        coverage = round(matched_count / total_count, 4) if total_count > 0 else 0.0
        return {
            "total_calories": round(total_calories, 2),
            "matched_count": matched_count,
            "total_count": total_count,
            "calorie_coverage": coverage,
            "parsed_ingredients": details,
        }

    def estimate_recipe(
        self,
        raw_ingredients: Iterable[str],
        ner_ingredients: Iterable[str] | None = None,
        directions: Iterable[str] = (),
        title: str = "",
        default_servings: int = 4,
    ) -> CalorieEstimate:
        """Estimate total and per-serving calories for a recipe (typed API, PRD Section 16)."""
        raw_list = [r for r in raw_ingredients if isinstance(r, str) and r.strip()]
        ner_list = list(ner_ingredients) if ner_ingredients is not None else []

        items: list[IngredientCalorieItem] = []
        total_kcal = 0.0
        matched_count = 0

        for idx, raw_line in enumerate(raw_list):
            parsed = parse_ingredient(raw_line)
            quantity = parsed["quantity"]
            unit = parsed["unit"]
            name = str(parsed["ingredient"] or "")
            hint = ner_list[idx] if idx < len(ner_list) else None

            lookup = self.lookup_calorie_per_100g(name)
            if lookup is None and hint:
                lookup = self.lookup_calorie_per_100g(str(hint))

            if lookup is None:
                items.append(
                    IngredientCalorieItem(
                        raw=raw_line,
                        quantity=quantity,
                        unit=unit,
                        ingredient=name,
                        canonical=None,
                        grams=None,
                        kcal_per_100g=None,
                        calories=None,
                        match_type="unmatched",
                    )
                )
                continue

            canonical = str(lookup["canonical"])
            kcal = float(lookup["kcal_per_100g"])
            fdc_id = lookup["fdc_id"]
            match_type = str(lookup["match_type"])

            if match_type == "zero" and quantity is None:
                grams, item_kcal = 0.0, 0.0
            else:
                grams = self.to_grams(
                    quantity, unit, canonical, fdc_id=str(fdc_id) if fdc_id else None
                )
                if grams is None:
                    grams = 0.0
                item_kcal = (grams / 100.0) * kcal

            total_kcal += item_kcal
            matched_count += 1
            items.append(
                IngredientCalorieItem(
                    raw=raw_line,
                    quantity=quantity,
                    unit=unit,
                    ingredient=name,
                    canonical=canonical,
                    grams=round(grams, 2),
                    kcal_per_100g=kcal,
                    calories=round(item_kcal, 2),
                    match_type=match_type,
                )
            )

        if not raw_list or matched_count == 0:
            return CalorieEstimate(
                total_calories=None,
                calories_per_serving=None,
                estimated_servings=1,
                servings_source="single",
                matched_count=0,
                total_count=len(raw_list),
                items=items,
            )

        parsed_servings = extract_servings(directions, title=title)
        if parsed_servings is not None:
            servings = parsed_servings
            servings_source = "parsed"
        elif total_kcal > 850:
            servings = max(1, default_servings)
            servings_source = "heuristic"
        elif total_kcal > 500:
            servings = 2
            servings_source = "heuristic"
        else:
            servings = 1
            servings_source = "single"

        total_rounded = round(total_kcal, 1)
        return CalorieEstimate(
            total_calories=total_rounded,
            calories_per_serving=round(total_rounded / servings, 1),
            estimated_servings=servings,
            servings_source=servings_source,
            matched_count=matched_count,
            total_count=len(raw_list),
            items=items,
        )


**`src/preprocessing/build_corpus.py`**


In [ ]:
DEFAULT_CORPUS_CSV = ROOT / "data" / "samples" / "recipes_sample.csv"
DEFAULT_OUTPUT_PARQUET = DEFAULT_PROCESSED_DIR / "recipes.parquet"

def _parse_list_field(raw_value: str) -> list[str]:
    if not raw_value:
        return []
    try:
        parsed = ast.literal_eval(raw_value)
    except (ValueError, SyntaxError):
        return []
    if not isinstance(parsed, list):
        return []
    return [str(item) for item in parsed if isinstance(item, str) and item.strip()]

def build_processed_corpus(
    corpus_path: Path | str = DEFAULT_CORPUS_CSV,
    output_path: Path | str = DEFAULT_OUTPUT_PARQUET,
    vocab_path: Path | str = DEFAULT_VOCAB_PATH,
    processed_dir: Path | str = DEFAULT_PROCESSED_DIR,
    limit: int | None = None,
) -> pd.DataFrame:
    corpus_file = Path(corpus_path)
    out_file = Path(output_path)
    normalizer = IngredientNormalizer(processed_dir=processed_dir)
    calorie_estimator = RecipeCalorieEstimator(processed_dir=processed_dir, normalizer=normalizer)

    records: list[dict[str, object]] = []
    with corpus_file.open(encoding="utf-8", newline="") as fh:
        reader = csv.DictReader(fh)
        for index, row in enumerate(reader):
            if limit is not None and index >= limit:
                break
            raw_ingredients = _parse_list_field(row.get("ingredients", ""))
            ner_terms = _parse_list_field(row.get("NER", ""))
            directions = _parse_list_field(row.get("directions", ""))

            canonical = normalizer.extract_canonical_ingredients(ner_terms, known_only=False)
            calorie_info = calorie_estimator.estimate_recipe_calories(
                raw_ingredients,
                ner_ingredients=ner_terms,
            )

            if not canonical:
                parsed_names = [
                    str(entry["canonical"] or entry["ingredient"] or "")
                    for entry in calorie_info["parsed_ingredients"]
                ]
                canonical = normalizer.extract_canonical_ingredients(parsed_names, known_only=False)

            records.append(
                {
                    "recipe_id": index,
                    "title": (row.get("title") or "").strip(),
                    "ingredients": raw_ingredients,
                    "canonical_ingredients": canonical,
                    "canonical_text": " ".join(canonical),
                    "directions": directions,
                    "total_calories": float(calorie_info["total_calories"]),
                    "calorie_coverage": float(calorie_info["calorie_coverage"]),
                    "source": (row.get("source") or "").strip(),
                    "link": (row.get("link") or "").strip(),
                }
            )

    df = pd.DataFrame(records)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_file, index=False)

    vocabulary = normalizer.build_vocabulary(df["canonical_ingredients"].tolist())
    normalizer.save_vocabulary(vocabulary, path=vocab_path)
    return df

def run_build_corpus(corpus=DEFAULT_CORPUS_CSV, output=DEFAULT_OUTPUT_PARQUET, vocab=DEFAULT_VOCAB_PATH, limit=None) -> None:

    df = build_processed_corpus(
        corpus_path=corpus,
        output_path=output,
        vocab_path=vocab,
        limit=limit,
    )
    reliable_ratio = float((df["calorie_coverage"] >= 0.7).mean()) if len(df) else 0.0
    print(f"processed_recipes={len(df)} reliable_calorie_ratio={reliable_ratio:.4f}")


In [ ]:
if RUN_BUILD_CORPUS:
    run_build_corpus()          # RecipeNLG penuh -> data/processed/recipes.parquet
    # run_build_corpus(limit=10000)  # korpus kecil untuk eksperimen cepat

## 2. Pipeline 1 — Pemahaman Query & Ekstraksi Ingredient (`src/pipeline1/`)

Query bebas -> `StructuredQuery` (ingredient tersedia, eksklusi, target kalori).
Ekstraksi ingredient punya 4 pendekatan *pluggable*: rule-based, surface-exact, fuzzy
(difflib / char n-gram), dan NER.


**`src/pipeline1/constraint_extractor.py`**

Pipeline 1 — calorie and exclusion constraint extraction (PRD §12.5–12.6).  Both extractors work on text that has already gone through ``normalize_text`` (lowercase, punctuation collapsed to spaces, apostrophes removed), so ``don't want`` arrives as ``dont want``.


In [ ]:
_NUMBER = r"(\d+(?:\.\d+)?)"
_UNIT = r"(?:\s*(?:calories?|kcal|cal))?"

# Order matters: a range must win over its own endpoints, and an upper bound
# ("no more than 700") must win over the bare comparator ("more than 700").
CALORIE_RANGE = re.compile(
    # "&" and "," collapse to spaces in normalize_text, so the connector is optional.
    rf"\bbetween\s+{_NUMBER}\s+(?:(?:and|or|n)\s+)?{_NUMBER}{_UNIT}\b"
)
CALORIE_MAX = re.compile(
    rf"\b(?:no\s+more\s+than|not\s+more\s+than|at\s+most|up\s+to|less\s+than|under|below|max(?:imum)?)\s+{_NUMBER}{_UNIT}\b"
)
CALORIE_MIN = re.compile(
    rf"\b(?:at\s+least|no\s+less\s+than|greater\s+than|more\s+than|above|over|minimum|min)\s+{_NUMBER}{_UNIT}\b"
)
CALORIE_TARGET = re.compile(
    rf"(?<!\w)(?:(?:around|about|approximately|roughly|close\s+to|nearly)\s+|~\s*){_NUMBER}\s*{_UNIT}"
)

# "no more than" / "no less than" are calorie bounds, not exclusions; calorie
# spans are blanked before exclusion detection, so any remaining "no" is a cue.
EXCLUSION_CUE = re.compile(
    r"\b(?:without|excluding|exclude|avoiding|avoid|dont\s+want|do\s+not\s+want"
    r"|doesnt\s+want|does\s+not\s+want|didnt\s+want|did\s+not\s+want|no)\b"
)
CLAUSE_END = re.compile(r"[.,;!?]")

def _number(raw: str) -> int | float:
    return float(raw) if "." in raw else int(raw)

def extract_calorie_constraints(text: str) -> tuple[dict[str, int | float | None], list[tuple[int, int]]]:
    """Return the calorie fields and the character spans they occupy."""
    fields: dict[str, int | float | None] = {
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
    taken: list[tuple[int, int]] = []

    def free(match: re.Match) -> bool:
        return not any(match.start() < end and start < match.end() for start, end in taken)

    for match in CALORIE_RANGE.finditer(text):
        if free(match):
            taken.append(match.span())
            fields["calorie_min"], fields["calorie_max"] = _number(match[1]), _number(match[2])

    for field, pattern in (
        ("calorie_max", CALORIE_MAX),
        ("calorie_min", CALORIE_MIN),
        ("calorie_target", CALORIE_TARGET),
    ):
        for match in pattern.finditer(text):
            if fields[field] is None and free(match):
                taken.append(match.span())
                fields[field] = _number(match[1])
                break

    return fields, taken

def exclusion_spans(masked_text: str) -> list[tuple[int, int]]:
    """Character spans that follow an exclusion cue, up to the next cue/period.

    ``no milk or butter`` yields one span containing ``milk or butter``;
    ``no butter and no cream`` yields two.
    """
    cues = list(EXCLUSION_CUE.finditer(masked_text))
    spans: list[tuple[int, int]] = []
    for index, cue in enumerate(cues):
        end = len(masked_text)
        clause = CLAUSE_END.search(masked_text, cue.end())
        if clause:
            end = clause.start()
        if index + 1 < len(cues) and cues[index + 1].start() < end:
            end = cues[index + 1].start()
        spans.append((cue.end(), end))
    return spans


**`src/pipeline1/ingredient_extractor.py`**

Pipeline 1 — ingredient mention extraction over the closed vocabulary.


In [ ]:
TOKEN = re.compile(r"[a-z0-9]+")

def match_ingredients(text: str, vocab: Vocabulary) -> list[tuple[int, int, str]]:
    """Longest-match scan; returns ``(start, end, canonical)`` per hit in text order."""
    tokens = [(m.group(), m.start(), m.end()) for m in TOKEN.finditer(text)]
    hits: list[tuple[int, int, str]] = []
    index = 0
    while index < len(tokens):
        remaining = len(tokens) - index
        for size in range(min(vocab.max_tokens, remaining), 0, -1):
            window = [token for token, _, _ in tokens[index : index + size]]
            window[-1] = singularize(window[-1])
            canonical = vocab.lookup(" ".join(window))
            if canonical:
                hits.append((tokens[index][1], tokens[index + size - 1][2], canonical))
                index += size
                break
        else:
            token, start, end = tokens[index]
            joined = _split_compound(token, vocab)
            if joined:
                hits.append((start, end, joined))
            index += 1
    return hits

def _split_compound(token: str, vocab: Vocabulary) -> str | None:
    """``sweetpotato`` -> ``sweet potato``: a joined spelling the alias table
    does not spell out. Split only when both halves are vocabulary words."""
    folded = singularize(token)
    for cut in range(1, len(folded)):
        canonical = vocab.lookup(f"{folded[:cut]} {singularize(folded[cut:])}")
        if canonical:
            return canonical
    return None

def extract_ingredients(text: str, vocab: Vocabulary) -> list[str]:
    """Canonical ingredients mentioned in ``text``, first mention order, unique."""
    seen: dict[str, None] = {}
    for _, _, canonical in match_ingredients(text, vocab):
        seen.setdefault(canonical, None)
    return list(seen)

def canonicalize(name: str, vocab: Vocabulary) -> str | None:
    return vocab.canonical(normalize_ingredient(name))


**`src/pipeline1/query_parser.py`**

Pipeline 1 — Query & Constraint Understanding (PRD §12).  raw query -> normalize -> exclusion/calorie constraints -> ingredient scan -> Structured Query  The output matches the PRD contract exactly:  { "available_ingredients": ["chicken", ...], "excluded_ingredients": ["cheese", ...], "calorie_target": 500 | None, "calorie_min": None, "calorie_max": None }  An ingredient named in an exclusion clause is never reported as available, even when it is also listed earlier ("I have butter... without butter").


In [ ]:
@lru_cache(maxsize=1)
def load_vocabulary() -> Vocabulary:
    return Vocabulary.load()

def _blank(text: str, spans: list[tuple[int, int]]) -> str:
    chars = list(text)
    for start, end in spans:
        chars[start:end] = " " * (end - start)
    return "".join(chars)

@dataclass
class ConstrainedQuery:
    """Normalized query with calorie/exclusion spans masked out of the contexts.

    The single composition shared by ``parse_query`` (production) and
    ``runner.trace_parse_query`` (UI evidence), so the two cannot drift.
    """

    text: str
    calories: dict[str, int | None]
    calorie_spans: list[tuple[int, int]]
    exclusion_spans: list[tuple[int, int]]
    available_context: str  # exclusion clauses blanked; input to the ingredient scan
    exclusion_context: str  # only the exclusion clause text

def constrain_query(query: str) -> ConstrainedQuery:
    """Normalize a query and split it into masked contexts for ingredient matching."""
    text = normalize_text(query)
    calories, calorie_spans = extract_calorie_constraints(text)
    masked = _blank(text, calorie_spans)

    spans = exclusion_spans(masked)
    return ConstrainedQuery(
        text=text,
        calories=calories,
        calorie_spans=calorie_spans,
        exclusion_spans=spans,
        available_context=_blank(masked, spans),
        exclusion_context=" ".join(masked[start:end] for start, end in spans),
    )

def parse_query(
    query: str, vocab: Vocabulary | None = None, matcher=extract_ingredients
) -> dict:
    """Turn a natural-language pantry request into the structured query.

    ``matcher`` is the ingredient scan, ``(text, vocab) -> [canonical, ...]``;
    experiments swap it in without touching the constraint logic.
    """
    vocab = vocab or load_vocabulary()
    constrained = constrain_query(query)

    available = matcher(constrained.available_context, vocab)
    excluded = matcher(constrained.exclusion_context, vocab)

    return {
        "available_ingredients": [item for item in available if item not in excluded],
        "excluded_ingredients": excluded,
        **constrained.calories,
    }


**`src/pipeline1/approaches/base.py`**

Approach contract and evidence records for Pipeline 1 matching.


In [ ]:
@dataclass
class IngredientMatch:
    """One ingredient mention found in the (masked) matching text."""

    start: int
    end: int
    surface: str
    canonical: str
    method: str  # vocabulary | compound-split | difflib | char-ngram | ner
    score: float | None = None  # similarity or model confidence, when applicable

@dataclass
class ApproachTrace:
    """Intermediate evidence for one matching context; rendered by the UI."""

    approach: str
    text: str  # the exact text the approach matched on (already masked)
    matches: list[IngredientMatch] = field(default_factory=list)
    unmatched_tokens: list[str] = field(default_factory=list)
    uncanonicalized: list[dict[str, object]] = field(default_factory=list)  # NER-only OOV hits
    extras: dict[str, object] = field(default_factory=dict)  # approach-specific evidence

    @property
    def ingredients(self) -> list[str]:
        seen: dict[str, None] = {}
        for match in self.matches:
            seen.setdefault(match.canonical, None)
        return list(seen)

class IngredientApproach:
    """Strategy interface; subclasses are stateless unless they load a model."""

    name: str = ""
    description: str = ""

    def extract(self, text: str, vocab: Vocabulary) -> list[str]:
        return self.trace(text, vocab).ingredients

    def trace(self, text: str, vocab: Vocabulary) -> ApproachTrace:
        raise NotImplementedError


**`src/pipeline1/approaches/rule.py`**

Vocabulary-scanning approaches: production rule matcher and the exact-surface ablation.


In [ ]:
class RuleApproach(IngredientApproach):
    """Production longest-match scan: aliases, plural folding, compound splitting."""

    name = "rule"
    description = "Longest-match vocabulary scan with aliases, plural folding, and compound splitting (production baseline)."

    def trace(self, text: str, vocab: Vocabulary) -> ApproachTrace:
        tokens = [(m.group(), m.start(), m.end()) for m in TOKEN.finditer(text)]
        matches: list[IngredientMatch] = []
        index = 0
        while index < len(tokens):
            remaining = len(tokens) - index
            for size in range(min(vocab.max_tokens, remaining), 0, -1):
                window = [token for token, _, _ in tokens[index : index + size]]
                window[-1] = singularize(window[-1])
                canonical = vocab.lookup(" ".join(window))
                if canonical:
                    matches.append(
                        IngredientMatch(
                            start=tokens[index][1],
                            end=tokens[index + size - 1][2],
                            surface=" ".join(token for token, _, _ in tokens[index : index + size]),
                            canonical=canonical,
                            method="vocabulary",
                        )
                    )
                    index += size
                    break
            else:
                token, start, end = tokens[index]
                joined = _split_compound(token, vocab)
                if joined:
                    matches.append(
                        IngredientMatch(
                            start=start, end=end, surface=token, canonical=joined, method="compound-split"
                        )
                    )
                index += 1
        matched_chars = [(m.start, m.end) for m in matches]
        unmatched = [
            token
            for token, start, end in tokens
            if not any(start < hit_end and end > hit_start for hit_start, hit_end in matched_chars)
        ]
        return ApproachTrace(approach=self.name, text=text, matches=matches, unmatched_tokens=unmatched)

class SurfaceExactApproach(RuleApproach):
    """Ablation: exact vocabulary surfaces only — no plural folding or compound recovery."""

    name = "surface-exact"
    description = "Ablation: exact vocabulary surface matching; no plural folding, aliases, or compound splitting."

    def trace(self, text: str, vocab: Vocabulary) -> ApproachTrace:
        tokens = [(m.group(), m.start(), m.end()) for m in TOKEN.finditer(text)]
        matches: list[IngredientMatch] = []
        index = 0
        while index < len(tokens):
            remaining = len(tokens) - index
            for size in range(min(vocab.max_tokens, remaining), 0, -1):
                key = " ".join(token for token, _, _ in tokens[index : index + size])
                canonical = vocab.lookup(key)
                if canonical:
                    matches.append(
                        IngredientMatch(
                            start=tokens[index][1],
                            end=tokens[index + size - 1][2],
                            surface=key,
                            canonical=canonical,
                            method="vocabulary",
                        )
                    )
                    index += size
                    break
            else:
                index += 1
        matched_chars = [(m.start, m.end) for m in matches]
        unmatched = [
            token
            for token, start, end in tokens
            if not any(start < hit_end and end > hit_start for hit_start, hit_end in matched_chars)
        ]
        return ApproachTrace(approach=self.name, text=text, matches=matches, unmatched_tokens=unmatched)


**`src/pipeline1/approaches/fuzzy.py`**

Fuzzy-fallback approaches: production exact matches first, then similarity recovery.  Moved here from ``evaluation/experiment.py`` so the CLI harness, the tests, and the UI share one implementation. Both variants keep the production exact-match spans untouched and only try to rescue uncovered tokens.


In [ ]:
def _char_ngrams(value: str) -> Counter[str]:
    return Counter(value[i : i + n] for n in (2, 3, 4) for i in range(len(value) - n + 1))

@dataclass(frozen=True)
class _FuzzySpec:
    method: str  # difflib | char-ngram
    threshold: float

class FuzzyApproach(IngredientApproach):
    """Exact vocabulary scan, then similarity-match uncovered single tokens."""

    name = ""  # instance-level: difflib-0.82 / char-ngram-0.60
    description = ""

    difflib_name = "difflib-0.82"
    difflib_description = "Production matcher plus difflib.SequenceMatcher fallback for unmatched tokens (threshold 0.82)."
    ngram_name = "char-ngram-0.60"
    ngram_description = "Production matcher plus character 2-4-gram cosine fallback for unmatched tokens (threshold 0.60)."

    def __init__(self, spec: _FuzzySpec) -> None:
        self.spec = spec
        self.name = f"{spec.method}-{spec.threshold:.2f}"
        self.description = (
            self.difflib_description if spec.method == "difflib" else self.ngram_description
        )
        self._exact = RuleApproach()
        self._grams: dict[str, Counter[str]] | None = None
        self._norms: dict[str, float] | None = None

    @classmethod
    def for_name(cls, name: str) -> "FuzzyApproach":
        if name == cls.difflib_name:
            return cls(_FuzzySpec("difflib", 0.82))
        if name == cls.ngram_name:
            return cls(_FuzzySpec("char-ngram", 0.60))
        raise KeyError(f"unknown fuzzy approach: {name!r}")

    def _score_token(self, token: str, keys: list[tuple[str, str]]) -> tuple[float, str, str]:
        if self.spec.method == "difflib":
            scored = ((difflib.SequenceMatcher(None, token, key).ratio(), key, value) for key, value in keys)
            return max(scored, default=(0.0, "", ""))
        if self._grams is None:
            self._grams = {key: _char_ngrams(key) for key, _ in keys}
            self._norms = {
                key: math.sqrt(sum(n * n for n in vector.values())) for key, vector in self._grams.items()
            }
        left = _char_ngrams(token)
        left_norm = math.sqrt(sum(n * n for n in left.values()))
        scored = (
            (
                sum(count * self._grams[key].get(gram, 0) for gram, count in left.items())
                / (left_norm * self._norms[key])
                if left_norm
                else 0.0,
                key,
                value,
            )
            for key, value in keys
        )
        return max(scored, default=(0.0, "", ""))

    def trace(self, text: str, vocab: Vocabulary) -> ApproachTrace:
        exact = self._exact.trace(text, vocab)
        matches = list(exact.matches)
        covered = [(m.start, m.end) for m in matches]
        keys = self._single_word_keys(vocab)
        seen = {m.canonical: None for m in matches}
        unmatched: list[str] = []
        for token_match in TOKEN.finditer(text):
            start, end = token_match.span()
            if any(start < hit_end and end > hit_start for hit_start, hit_end in covered):
                continue
            token = singularize(token_match.group())
            unmatched.append(token)
            score, _, canonical = self._score_token(token, keys)
            if canonical and score >= self.spec.threshold and canonical not in seen:
                matches.append(
                    IngredientMatch(
                        start=start,
                        end=end,
                        surface=token_match.group(),
                        canonical=canonical,
                        method=f"fuzzy-{self.spec.method}",
                        score=round(score, 4),
                    )
                )
                seen[canonical] = None
        return ApproachTrace(
            approach=self.name,
            text=text,
            matches=matches,
            unmatched_tokens=unmatched,
            extras={"threshold": self.spec.threshold, "fallback_method": self.spec.method},
        )

    def _single_word_keys(self, vocab: Vocabulary) -> list[tuple[str, str]]:
        return [(key, value) for key, value in vocab.surface_to_canonical.items() if " " not in key]


**`src/pipeline1/approaches/ner.py`**

BERT-based token-classification approach for Pipeline 1 ingredient extraction.  Fine-tuned DistilBERT trained on synthetic pantry queries (see ``scripts/generate_ner_training_data.py`` and ``scripts/train_ner.py``). The model sees the same masked matching text the rule-based approaches see: calorie and exclusion spans are already blanked by the query parser.  Transformers/torch are imported lazily so the lexical approaches keep working without the deep-learning stack installed.


In [ ]:
DEFAULT_MODEL_DIR = ROOT / "artifacts" / "ner_model"
MAX_LENGTH = 96

def _pick_device(torch) -> "torch.device":
    """CUDA (NVIDIA) > MPS (Apple silicon) > CPU — portable by default."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class NerApproach(IngredientApproach):
    """Token classification (B-ING/I-ING/O) with vocabulary canonicalization."""

    name = "ner"
    description = "Fine-tuned DistilBERT token classification over the query, canonicalized through the closed vocabulary."

    def __init__(self, model_dir: Path | str = DEFAULT_MODEL_DIR) -> None:
        self.model_dir = Path(model_dir)
        self._tokenizer = None
        self._model = None
        self._device = None

    @property
    def available(self) -> bool:
        return (self.model_dir / "config.json").exists()

    def _load(self) -> None:
        if self._model is not None:
            return
        if not self.available:
            raise FileNotFoundError(
                f"NER model artifacts not found at {self.model_dir}; run scripts/train_ner.py first"
            )
        import torch
        from transformers import AutoModelForTokenClassification, AutoTokenizer

        self._tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        # eager attention is required for output_attentions (SDPA returns none)
        self._model = AutoModelForTokenClassification.from_pretrained(
            self.model_dir, attn_implementation="eager"
        )
        self._device = _pick_device(torch)
        self._model.to(self._device)
        self._model.eval()

    def _predict(self, text: str):
        """Tokenize + tag one text; returns (offsets, tag_ids, probabilities)."""
        import torch

        self._load()
        encoding = self._tokenizer(
            text,
            return_offsets_mapping=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        offsets = encoding.pop("offset_mapping")[0].tolist()
        with torch.no_grad():
            logits = self._model(
                input_ids=encoding["input_ids"].to(self._device),
                attention_mask=encoding["attention_mask"].to(self._device),
            ).logits[0]
        probs = torch.softmax(logits, dim=-1)
        tag_ids = probs.argmax(dim=-1).tolist()
        confidences = probs.max(dim=-1).values.tolist()
        return offsets, tag_ids, confidences

    def trace(self, text: str, vocab: Vocabulary) -> ApproachTrace:
        offsets, tag_ids, confidences = self._predict(text)
        id2label = self._model.config.id2label

        token_evidence: list[dict[str, object]] = []
        matches: list[IngredientMatch] = []
        uncanonicalized: list[dict[str, object]] = []
        unmatched: list[str] = []

        open_start = open_end = None
        open_probs: list[float] = []

        def close_span() -> None:
            nonlocal open_start, open_end, open_probs
            if open_start is None:
                return
            surface = text[open_start:open_end]
            canonical = vocab.canonical(normalize_ingredient(surface))
            confidence = sum(open_probs) / len(open_probs) if open_probs else None
            if canonical:
                matches.append(
                    IngredientMatch(
                        start=open_start,
                        end=open_end,
                        surface=surface,
                        canonical=canonical,
                        method="ner",
                        score=round(confidence, 4) if confidence is not None else None,
                    )
                )
            else:
                uncanonicalized.append(
                    {"surface": surface, "confidence": round(confidence, 4) if confidence else None}
                )
            open_start = open_end = None
            open_probs = []

        for (start, end), tag_id, conf in zip(offsets, tag_ids, confidences):
            label = id2label[tag_id]
            token = text[start:end]
            if end <= start:  # special tokens (CLS/SEP) carry (0, 0) offsets
                token_evidence.append({"token": token, "tag": label, "prob": round(conf, 4)})
                continue
            token_evidence.append(
                {"token": token, "tag": label, "prob": round(conf, 4), "start": start, "end": end}
            )
            if label.endswith("B-ING"):
                close_span()
                open_start, open_end, open_probs = start, end, [conf]
            elif label.endswith("I-ING") and open_start is not None:
                open_end = end
                open_probs.append(conf)
            elif label.endswith("I-ING"):
                open_start, open_end, open_probs = start, end, [conf]
            else:
                close_span()
                if token:
                    unmatched.append(token)
        close_span()

        seen: set[str] = set()
        ordered_matches: list[IngredientMatch] = []
        for match in matches:
            if match.canonical not in seen:
                ordered_matches.append(match)
                seen.add(match.canonical)
        return ApproachTrace(
            approach=self.name,
            text=text,
            matches=ordered_matches,
            unmatched_tokens=unmatched,
            uncanonicalized=uncanonicalized,
            extras={"tokens": token_evidence, "model_dir": str(self.model_dir)},
        )

    def attention(self, text: str) -> dict[str, object]:
        """Last-layer attention averaged over heads, for the UI heatmap."""
        import torch

        self._load()
        encoding = self._tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            return_offsets_mapping=True,
            return_tensors="pt",
        )
        offsets = encoding.pop("offset_mapping")[0].tolist()
        with torch.no_grad():
            outputs = self._model(
                input_ids=encoding["input_ids"].to(self._device),
                attention_mask=encoding["attention_mask"].to(self._device),
                output_attentions=True,
            )
        # last layer, mean over heads -> [seq, seq]
        matrix = outputs.attentions[-1][0].mean(dim=0).cpu().tolist()
        tokens = self._tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
        return {
            "tokens": tokens,
            "matrix": matrix,
            "offsets": offsets,
            "layers": len(outputs.attentions),
            "heads": self._model.config.num_attention_heads,
        }


**`src/pipeline1/approaches/runner.py`**

Trace-aware Pipeline 1 runner: same composition as ``query_parser.parse_query``, recording every intermediate step for the UI.


In [ ]:
@dataclass
class Pipeline1Trace:
    """Everything the UI needs to replay one query through Pipeline 1."""

    approach: str
    raw_query: str
    normalized_text: str
    calorie_constraints: dict[str, int | None]
    calorie_spans: list[tuple[int, int]]
    exclusion_spans: list[tuple[int, int]]
    available_context: str  # masked text handed to the ingredient matcher
    exclusion_context: str
    available_trace: ApproachTrace
    excluded_trace: ApproachTrace
    structured_query: dict[str, object] = field(default_factory=dict)

def trace_parse_query(
    query: str,
    approach: IngredientApproach,
    vocab: Vocabulary | None = None,
) -> Pipeline1Trace:
    """Run Pipeline 1 with the selected approach and capture per-step evidence.

    Uses the same ``constrain_query`` composition as ``query_parser.parse_query``;
    only the ingredient scan differs (approach traces instead of bare results).
    """
    vocab = vocab or load_vocabulary()
    constrained = constrain_query(query)

    available_trace = approach.trace(constrained.available_context, vocab)
    excluded_trace = approach.trace(constrained.exclusion_context, vocab)

    available = available_trace.ingredients
    excluded = excluded_trace.ingredients
    structured = {
        "available_ingredients": [item for item in available if item not in excluded],
        "excluded_ingredients": excluded,
        **constrained.calories,
    }
    return Pipeline1Trace(
        approach=approach.name,
        raw_query=query,
        normalized_text=constrained.text,
        calorie_constraints=constrained.calories,
        calorie_spans=constrained.calorie_spans,
        exclusion_spans=constrained.exclusion_spans,
        available_context=constrained.available_context,
        exclusion_context=constrained.exclusion_context,
        available_trace=available_trace,
        excluded_trace=excluded_trace,
        structured_query=structured,
    )


**`src/pipeline1/approaches/__init__.py`**

Pipeline 1 — pluggable ingredient-matching approaches.  Every approach answers the same contract so the query parser, the experiment harness, and the Streamlit UI can swap them freely:  * ``extract(text, vocab) -> list[str]`` — canonical ingredients, first-mention order (drop-in for ``parse_query``'s ``matcher`` argument); * ``trace(text, vocab) -> ApproachTrace`` — the same result plus the intermediate evidence (matched spans, provenance, scores) the UI renders.  The matching context is the *masked* query text produced by ``query_parser.parse_query``: calorie and exclusion spans are already blanked, so approaches never see constraint clauses.


In [ ]:
_NER_DEFAULT_DIR = ROOT / "artifacts" / "ner_model"

def _approach_factories() -> dict[str, type[IngredientApproach]]:
    return {
        RuleApproach.name: RuleApproach,
        SurfaceExactApproach.name: SurfaceExactApproach,
        FuzzyApproach.difflib_name: FuzzyApproach,
        FuzzyApproach.ngram_name: FuzzyApproach,
        NerApproach.name: NerApproach,
    }

def available_approaches() -> list[dict[str, object]]:
    """Registry metadata for UI pickers; ``available`` flags artifact-dependent models."""
    ner_ready = (_NER_DEFAULT_DIR / "config.json").exists()
    return [
        {"name": RuleApproach.name, "description": RuleApproach.description, "available": True},
        {"name": SurfaceExactApproach.name, "description": SurfaceExactApproach.description, "available": True},
        {"name": FuzzyApproach.difflib_name, "description": FuzzyApproach.difflib_description, "available": True},
        {"name": FuzzyApproach.ngram_name, "description": FuzzyApproach.ngram_description, "available": True},
        {"name": NerApproach.name, "description": NerApproach.description, "available": ner_ready},
    ]

def get_approach(name: str) -> IngredientApproach:
    factories = _approach_factories()
    if name not in factories:
        raise KeyError(f"unknown Pipeline 1 approach: {name!r}; known: {sorted(factories)}")
    if name == NerApproach.name:
        return NerApproach()
    if name in (FuzzyApproach.difflib_name, FuzzyApproach.ngram_name):
        return FuzzyApproach.for_name(name)
    return factories[name]()


## 3. Pipeline 2 — Retrieval Kandidat (`src/pipeline2/`)

TF-IDF (dengan down-weighting pantry staple) dan dense retrieval (sentence-transformers)
menghasilkan kandidat ber-skema identik untuk dikonsumsi reranker Pipeline 3.


**`src/pipeline2/vectorizer.py`**


In [ ]:
DEFAULT_ARTIFACTS_DIR = ROOT / "artifacts"
DEFAULT_VECTORIZER_PATH = DEFAULT_ARTIFACTS_DIR / "tfidf_vectorizer.pkl"
DEFAULT_VECTORS_PATH = DEFAULT_ARTIFACTS_DIR / "recipe_vectors.pkl"
DEFAULT_CORPUS_PARQUET = DEFAULT_PROCESSED_DIR / "recipes.parquet"
DEFAULT_STAPLES_PATH = DEFAULT_PROCESSED_DIR / "pantry_staples.csv"

def _load_staple_weights(path: Path | None) -> dict[str, float]:
    if path is None or not path.exists():
        return {}
    weights: dict[str, float] = {}
    with path.open(encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            key = normalize_ingredient(row["ingredient"])
            if key:
                weights[key] = float(row["weight"])
    return weights

def query_to_text(query: dict | Iterable[str] | str) -> str:
    if isinstance(query, dict):
        items = query.get("available_ingredients") or []
        return " ".join(normalize_ingredient(str(item)) for item in items if str(item).strip())
    if isinstance(query, str):
        return normalize_ingredient(query)
    return " ".join(normalize_ingredient(str(item)) for item in query if str(item).strip())

class RecipeVectorizer:
    def __init__(
        self,
        max_features: int | None = None,
        ngram_range: tuple[int, int] = (1, 1),
        min_df: int = 1,
        max_df: float = 1.0,
        sublinear_tf: bool = True,
        use_staple_downweight: bool = True,
        staples_path: Path | str | None = DEFAULT_STAPLES_PATH,
    ) -> None:
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.max_df = max_df
        self.sublinear_tf = sublinear_tf
        self.use_staple_downweight = use_staple_downweight
        self.staples_path = Path(staples_path) if staples_path is not None else None

        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=sublinear_tf,
            lowercase=True,
            token_pattern=r"(?u)\b[a-z0-9_]+\b",
        )
        self.staple_weights: dict[str, float] = (
            _load_staple_weights(self.staples_path) if self.use_staple_downweight else {}
        )
        self.feature_weights_: np.ndarray | None = None

    def _build_feature_weights(self) -> np.ndarray:
        feature_names = self.vectorizer.get_feature_names_out()
        weights = np.ones(len(feature_names), dtype=np.float32)
        if not self.use_staple_downweight or not self.staple_weights:
            return weights
        for idx, term in enumerate(feature_names):
            if term in self.staple_weights:
                weights[idx] = self.staple_weights[term]
        return weights

    def _apply_weights(self, matrix: csr_matrix) -> csr_matrix:
        if not self.use_staple_downweight or self.feature_weights_ is None:
            return matrix
        weighted = matrix @ diags(self.feature_weights_, format="csr")
        return sk_normalize(weighted, norm="l2", axis=1, copy=False)

    def fit_transform(self, documents: list[str]) -> csr_matrix:
        matrix = self.vectorizer.fit_transform(documents)
        self.feature_weights_ = self._build_feature_weights()
        return self._apply_weights(matrix)

    def transform_documents(self, documents: list[str]) -> csr_matrix:
        matrix = self.vectorizer.transform(documents)
        return self._apply_weights(matrix)

    def transform_query(self, query: dict | Iterable[str] | str) -> csr_matrix:
        text = query_to_text(query)
        matrix = self.vectorizer.transform([text])
        return self._apply_weights(matrix)

    def save(
        self,
        recipe_vectors: csr_matrix,
        vectorizer_path: Path | str = DEFAULT_VECTORIZER_PATH,
        vectors_path: Path | str = DEFAULT_VECTORS_PATH,
    ) -> tuple[Path, Path]:
        vec_file = Path(vectorizer_path)
        mat_file = Path(vectors_path)
        vec_file.parent.mkdir(parents=True, exist_ok=True)
        mat_file.parent.mkdir(parents=True, exist_ok=True)
        state = {
            "max_features": self.max_features,
            "ngram_range": self.ngram_range,
            "min_df": self.min_df,
            "max_df": self.max_df,
            "sublinear_tf": self.sublinear_tf,
            "use_staple_downweight": self.use_staple_downweight,
            "staple_weights": self.staple_weights,
            "feature_weights_": self.feature_weights_,
            "vectorizer": self.vectorizer,
        }
        with vec_file.open("wb") as fh:
            pickle.dump(state, fh)
        with mat_file.open("wb") as fh:
            pickle.dump(recipe_vectors, fh)
        return vec_file, mat_file

    @classmethod
    def load(
        cls,
        vectorizer_path: Path | str = DEFAULT_VECTORIZER_PATH,
        vectors_path: Path | str = DEFAULT_VECTORS_PATH,
    ) -> tuple[RecipeVectorizer, csr_matrix]:
        with Path(vectorizer_path).open("rb") as fh:
            state = pickle.load(fh)
        with Path(vectors_path).open("rb") as fh:
            recipe_vectors = pickle.load(fh)
        if isinstance(state, cls):
            return state, recipe_vectors
        instance = cls(
            max_features=state["max_features"],
            ngram_range=state["ngram_range"],
            min_df=state["min_df"],
            max_df=state["max_df"],
            sublinear_tf=state["sublinear_tf"],
            use_staple_downweight=state["use_staple_downweight"],
            staples_path=None,
        )
        instance.staple_weights = state["staple_weights"]
        instance.feature_weights_ = state["feature_weights_"]
        instance.vectorizer = state["vectorizer"]
        return instance, recipe_vectors

def fit_corpus_tfidf(
    corpus_parquet: Path | str = DEFAULT_CORPUS_PARQUET,
    vectorizer_path: Path | str = DEFAULT_VECTORIZER_PATH,
    vectors_path: Path | str = DEFAULT_VECTORS_PATH,
    max_features: int | None = None,
    ngram_range: tuple[int, int] = (1, 1),
    sublinear_tf: bool = True,
    use_staple_downweight: bool = True,
) -> tuple[RecipeVectorizer, csr_matrix]:
    df = pd.read_parquet(corpus_parquet)
    if "canonical_text" in df.columns:
        documents = df["canonical_text"].fillna("").astype(str).tolist()
    else:
        documents = [" ".join(items) for items in df["canonical_ingredients"].tolist()]

    vectorizer = RecipeVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=sublinear_tf,
        use_staple_downweight=use_staple_downweight,
    )
    recipe_vectors = vectorizer.fit_transform(documents)
    vectorizer.save(
        recipe_vectors=recipe_vectors,
        vectorizer_path=vectorizer_path,
        vectors_path=vectors_path,
    )
    return vectorizer, recipe_vectors

def run_fit_tfidf(corpus=DEFAULT_CORPUS_PARQUET, vectorizer_out=DEFAULT_VECTORIZER_PATH, vectors_out=DEFAULT_VECTORS_PATH, max_features=0, ngram_max=1, no_sublinear_tf=False, no_staple_downweight=False) -> None:

    max_features = None if max_features <= 0 else max_features
    _, vectors = fit_corpus_tfidf(
        corpus_parquet=corpus,
        vectorizer_path=vectorizer_out,
        vectors_path=vectors_out,
        max_features=max_features,
        ngram_range=(1, ngram_max),
        sublinear_tf=not no_sublinear_tf,
        use_staple_downweight=not no_staple_downweight,
    )
    print(f"fitted_shape={vectors.shape} nnz={vectors.nnz}")


In [ ]:
if RUN_FIT_TFIDF:
    run_fit_tfidf()             # -> artifacts/tfidf_vectorizer.pkl + recipe_vectors.pkl

**`src/pipeline2/retriever.py`**


In [ ]:
class CandidateRetriever:
    def __init__(
        self,
        vectorizer: RecipeVectorizer,
        recipe_vectors: csr_matrix,
        corpus: pd.DataFrame,
    ) -> None:
        self.vectorizer = vectorizer
        self.recipe_vectors = recipe_vectors
        self.corpus = corpus.reset_index(drop=True)
        self.recipe_ids = self.corpus["recipe_id"].to_numpy()

    @classmethod
    def from_artifacts(
        cls,
        corpus_path: Path | str = DEFAULT_CORPUS_PARQUET,
        vectorizer_path: Path | str = DEFAULT_VECTORIZER_PATH,
        vectors_path: Path | str = DEFAULT_VECTORS_PATH,
    ) -> CandidateRetriever:
        vectorizer, recipe_vectors = RecipeVectorizer.load(
            vectorizer_path=vectorizer_path,
            vectors_path=vectors_path,
        )
        corpus = pd.read_parquet(corpus_path)
        return cls(vectorizer=vectorizer, recipe_vectors=recipe_vectors, corpus=corpus)

    def score_all(self, structured_query: dict | Iterable[str] | str) -> np.ndarray:
        query_vector = self.vectorizer.transform_query(structured_query)
        if query_vector.nnz == 0:
            return np.zeros(self.recipe_vectors.shape[0], dtype=np.float32)
        scores = cosine_similarity(query_vector, self.recipe_vectors).ravel()
        return scores

    def retrieve_indices(
        self,
        structured_query: dict | Iterable[str] | str,
        top_n: int = 50,
    ) -> tuple[np.ndarray, np.ndarray]:
        scores = self.score_all(structured_query)
        total = len(scores)
        if total == 0 or top_n <= 0:
            return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
        k = min(top_n, total)
        if k < total:
            top_idx = np.argpartition(scores, -k)[-k:]
            ordered = top_idx[np.argsort(scores[top_idx])[::-1]]
        else:
            ordered = np.argsort(scores)[::-1]
        return ordered, scores[ordered]

    def retrieve(
        self,
        structured_query: dict | Iterable[str] | str,
        top_n: int = 50,
    ) -> list[dict[str, object]]:
        indices, scores = self.retrieve_indices(structured_query, top_n=top_n)
        candidates: list[dict[str, object]] = []
        for idx, score in zip(indices, scores):
            row = self.corpus.iloc[int(idx)]
            candidates.append(
                {
                    "recipe_id": int(row["recipe_id"]),
                    "title": str(row["title"]),
                    "ingredients": list(row["ingredients"]),
                    "canonical_ingredients": list(row["canonical_ingredients"]),
                    "directions": list(row["directions"]),
                    "total_calories": float(row["total_calories"]),
                    "calorie_coverage": float(row["calorie_coverage"]),
                    "tfidf_score": round(float(score), 6),
                }
            )
        return candidates


**`src/pipeline2/approaches/__init__.py`**

Pipeline 2 — pluggable retrieval approaches.  Both approaches answer one contract so the query parser output, the reranker, and the UI can swap them freely:  * ``retrieve(structured_query, top_n) -> list[dict]`` — candidate dicts with the exact schema ``pipeline3.models.CandidateRecipe.from_dict`` consumes (``recipe_id``, ``title``, ``ingredients``, ``canonical_ingredients``, ``directions``, ``total_calories``, ``calorie_coverage``, ``tfidf_score``); * ``trace(structured_query, top_n) -> RetrievalTrace`` — query terms, weights, and per-candidate scores for the UI.


In [ ]:
DEFAULT_DENSE_VECTORS_PATH = DEFAULT_ARTIFACTS_DIR / "dense_vectors.npy"
DEFAULT_DENSE_META_PATH = DEFAULT_ARTIFACTS_DIR / "dense_meta.json"
DEFAULT_MODEL_NAME = "all-MiniLM-L6-v2"

@dataclass
class RetrievalTrace:
    """Intermediate evidence for one retrieval call; rendered by the UI."""

    approach: str
    query_text: str
    score_type: str  # tfidf-cosine | dense-cosine
    query_terms: list[dict[str, float]] = field(default_factory=list)  # [{"term", "weight"}]
    candidates: list[dict[str, object]] = field(default_factory=list)
    extras: dict[str, object] = field(default_factory=dict)

class RetrievalApproach:
    """Strategy interface wrapping a concrete retriever."""

    name: str = ""
    description: str = ""

    def score_all(self, structured_query: dict | list[str] | str) -> np.ndarray:
        raise NotImplementedError

    def retrieve(self, structured_query: dict | list[str] | str, top_n: int = 50) -> list[dict]:
        raise NotImplementedError

    def trace(self, structured_query: dict | list[str] | str, top_n: int = 50) -> RetrievalTrace:
        raise NotImplementedError

def _sentence_transformers_available() -> bool:
    try:
        import sentence_transformers  # noqa: F401

        return True
    except ModuleNotFoundError:
        return False

def available_retrieval_approaches() -> list[dict[str, object]]:
    """Registry metadata for UI pickers; each ``available`` flags artifact readiness."""
    tfidf_ready = DEFAULT_VECTORIZER_PATH.exists() and DEFAULT_VECTORS_PATH.exists()
    dense_ready = (
        _sentence_transformers_available()
        and DEFAULT_DENSE_VECTORS_PATH.exists()
        and DEFAULT_DENSE_META_PATH.exists()
    )
    return [
        {"name": TfidfApproach.name, "description": TfidfApproach.description, "available": tfidf_ready},
        {"name": DenseApproach.name, "description": DenseApproach.description, "available": dense_ready},
    ]

def get_retrieval_approach(name: str) -> RetrievalApproach:
    if name == TfidfApproach.name:
        return TfidfApproach()
    if name == DenseApproach.name:
        return DenseApproach()
    raise KeyError(f"unknown Pipeline 2 approach: {name!r}")

class TfidfApproach(RetrievalApproach):
    """TF-IDF + cosine similarity over canonical ingredient text (production baseline)."""

    name = "tfidf"
    description = "TF-IDF unigram (full vocab, sublinear_tf) with pantry-staple down-weighting, cosine similarity."

    def __init__(self, retriever: CandidateRetriever | None = None) -> None:
        self._retriever = retriever

    @property
    def retriever(self) -> CandidateRetriever:
        if self._retriever is None:
            self._retriever = CandidateRetriever.from_artifacts()
        return self._retriever

    def score_all(self, structured_query: dict | list[str] | str) -> np.ndarray:
        return self.retriever.score_all(structured_query)

    @property
    def recipe_ids(self) -> np.ndarray:
        """Harness compatibility: evaluate_retriever reads recipe_ids directly."""
        return self.retriever.recipe_ids

    def retrieve_indices(self, structured_query: dict | list[str] | str, top_n: int = 50) -> tuple[np.ndarray, np.ndarray]:
        return self.retriever.retrieve_indices(structured_query, top_n=top_n)

    def retrieve(self, structured_query: dict | list[str] | str, top_n: int = 50) -> list[dict]:
        return self.retriever.retrieve(structured_query, top_n=top_n)

    def trace(self, structured_query: dict | list[str] | str, top_n: int = 50) -> RetrievalTrace:
        vectorizer: RecipeVectorizer = self.retriever.vectorizer
        query_vector = vectorizer.transform_query(structured_query)
        feature_names = vectorizer.vectorizer.get_feature_names_out()

        terms: list[dict[str, float]] = []
        if query_vector.nnz:
            row = query_vector.tocsr()[0]
            for idx, weight in zip(row.indices, row.data):
                terms.append({"term": str(feature_names[idx]), "weight": round(float(weight), 4)})
            terms.sort(key=lambda item: item["weight"], reverse=True)

        candidates = self.retrieve(structured_query, top_n=top_n)
        return RetrievalTrace(
            approach=self.name,
            query_text=query_to_text(structured_query),
            score_type="tfidf-cosine",
            query_terms=terms[:20],
            candidates=candidates,
            extras={
                "top_n": top_n,
                "matrix_shape": list(self.retriever.recipe_vectors.shape),
                "staple_downweight": vectorizer.use_staple_downweight,
            },
        )

class DenseApproach(RetrievalApproach):
    """Sentence-transformer embeddings + cosine similarity over the same corpus."""

    name = "dense"
    description = "Sentence-transformer (all-MiniLM-L6-v2) dense vectors over canonical ingredient text, cosine similarity."

    def __init__(self, model=None, vectors_path: Path = DEFAULT_DENSE_VECTORS_PATH, meta_path: Path = DEFAULT_DENSE_META_PATH) -> None:
        self._model = model
        self._vectors: np.ndarray | None = None
        self._corpus = None
        self._vectors_path = Path(vectors_path)
        self._meta_path = Path(meta_path)

    @property
    def model(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer

            self._model = SentenceTransformer(self._load_meta()["model"])
        return self._model

    def _load_meta(self) -> dict:
        import json

        if not self._meta_path.exists():
            raise FileNotFoundError(
                f"dense artifacts not found at {self._meta_path}; run scripts/encode_dense_corpus.py first"
            )
        return json.loads(self._meta_path.read_text(encoding="utf-8"))

    def _ensure_artifacts(self) -> None:
        if self._vectors is not None:
            return
        meta = self._load_meta()
        self._vectors = np.load(self._vectors_path)
        import pandas as pd

        self._corpus = pd.read_parquet(meta["corpus_path"]).reset_index(drop=True)

    def score_all(self, structured_query: dict | list[str] | str) -> np.ndarray:
        self._ensure_artifacts()
        query_text = query_to_text(structured_query)
        if not query_text.strip():
            return np.zeros(self._vectors.shape[0], dtype=np.float32)
        query_vec = self.model.encode(query_text, normalize_embeddings=True)
        return self._vectors @ np.asarray(query_vec, dtype=np.float32)

    @property
    def recipe_ids(self) -> np.ndarray:
        self._ensure_artifacts()
        return self._corpus["recipe_id"].to_numpy()

    def retrieve_indices(self, structured_query: dict | list[str] | str, top_n: int = 50) -> tuple[np.ndarray, np.ndarray]:
        """Same contract as ``CandidateRetriever.retrieve_indices`` (harness compatibility)."""
        self._ensure_artifacts()
        scores = self.score_all(structured_query)
        total = len(scores)
        if total == 0 or top_n <= 0:
            return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
        k = min(top_n, total)
        top_idx = np.argpartition(scores, -k)[-k:]
        ordered = top_idx[np.argsort(scores[top_idx])[::-1]]
        return ordered, scores[ordered]

    def retrieve(self, structured_query: dict | list[str] | str, top_n: int = 50) -> list[dict]:
        self._ensure_artifacts()
        scores = self.score_all(structured_query)
        total = len(scores)
        if total == 0 or top_n <= 0:
            return []
        k = min(top_n, total)
        top_idx = np.argpartition(scores, -k)[-k:]
        ordered = top_idx[np.argsort(scores[top_idx])[::-1]]
        candidates: list[dict[str, object]] = []
        for idx in ordered:
            row = self._corpus.iloc[int(idx)]
            candidates.append(
                {
                    "recipe_id": int(row["recipe_id"]),
                    "title": str(row["title"]),
                    "ingredients": list(row["ingredients"]),
                    "canonical_ingredients": list(row["canonical_ingredients"]),
                    "directions": list(row["directions"]),
                    "total_calories": float(row["total_calories"]),
                    "calorie_coverage": float(row["calorie_coverage"]),
                    "tfidf_score": round(float(scores[idx]), 6),
                }
            )
        return candidates

    def trace(self, structured_query: dict | list[str] | str, top_n: int = 50) -> RetrievalTrace:
        query_text = query_to_text(structured_query)
        candidates = self.retrieve(structured_query, top_n=top_n)
        return RetrievalTrace(
            approach=self.name,
            query_text=query_text,
            score_type="dense-cosine",
            query_terms=[{"term": word, "weight": 1.0} for word in query_text.split()],
            candidates=candidates,
            extras={
                "top_n": top_n,
                "vectors_shape": list(self._vectors.shape) if self._vectors is not None else None,
                "model": self._load_meta()["model"],
            },
        )


## 4. Pipeline 3 — Pantry-Aware Multi-Criteria Reranking (`src/pipeline3/`)

Kandidat Pipeline 2 dinilai ulang: coverage pantry, deteksi eksklusi, skor kalori,
digabung `MultiCriteriaReranker` menjadi rekomendasi akhir + kartu rekomendasi.


**`src/pipeline3/models.py`**

Data models and configuration schemas for Pipeline 3 (Multi-Criteria Reranking).  These dataclasses define the clean integration contract between: - Pipeline 1 (Query & Constraint Understanding) -> ``StructuredQuery`` - Pipeline 2 (Candidate Recipe Retrieval) -> ``CandidateRecipe`` - Pipeline 3 (Multi-Criteria Recipe Reranking) -> ``RecommendationResult``


In [ ]:
@dataclass(frozen=True)
class StructuredQuery:
    """Structured representation of a user's pantry query (PRD Section 12.1)."""

    available_ingredients: tuple[str, ...]
    excluded_ingredients: tuple[str, ...] = ()
    calorie_target: float | None = None
    calorie_min: float | None = None
    calorie_max: float | None = None
    raw_query: str | None = None

    def __init__(
        self,
        available_ingredients: list[str] | tuple[str, ...],
        excluded_ingredients: list[str] | tuple[str, ...] = (),
        calorie_target: float | None = None,
        calorie_min: float | None = None,
        calorie_max: float | None = None,
        raw_query: str | None = None,
    ) -> None:
        object.__setattr__(self, "available_ingredients", tuple(available_ingredients))
        object.__setattr__(self, "excluded_ingredients", tuple(excluded_ingredients))
        object.__setattr__(self, "calorie_target", float(calorie_target) if calorie_target is not None else None)
        object.__setattr__(self, "calorie_min", float(calorie_min) if calorie_min is not None else None)
        object.__setattr__(self, "calorie_max", float(calorie_max) if calorie_max is not None else None)
        object.__setattr__(self, "raw_query", raw_query)

    @property
    def has_calorie_constraint(self) -> bool:
        return (
            self.calorie_target is not None
            or self.calorie_min is not None
            or self.calorie_max is not None
        )

    @property
    def has_hard_calorie_bound(self) -> bool:
        return self.calorie_min is not None or self.calorie_max is not None

    @classmethod
    def from_dict(cls, data: Mapping[str, Any], raw_query: str | None = None) -> StructuredQuery:
        return cls(
            available_ingredients=list(data.get("available_ingredients") or []),
            excluded_ingredients=list(data.get("excluded_ingredients") or []),
            calorie_target=data.get("calorie_target"),
            calorie_min=data.get("calorie_min"),
            calorie_max=data.get("calorie_max"),
            raw_query=raw_query if raw_query is not None else data.get("raw_query"),
        )

@dataclass
class CandidateRecipe:
    """A candidate recipe passed from Pipeline 2 retrieval into Pipeline 3."""

    recipe_id: int | str
    title: str
    canonical_ingredients: list[str]
    tfidf_score: float = 0.0
    ingredients: list[str] = field(default_factory=list)
    directions: list[str] = field(default_factory=list)
    total_calories: float | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, data: Mapping[str, Any]) -> CandidateRecipe:
        """Build a ``CandidateRecipe`` from a Pipeline 2 retriever dict.

        Directly accepts the output schema of ``CandidateRetriever.retrieve()``
        (including top-level ``calorie_coverage``, ``source``, ``link``) and
        treats non-finite calories or unmapped 0-coverage recipes as ``None``.
        """
        metadata = dict(data.get("metadata") or {})
        calorie_coverage: float | None = None
        if data.get("calorie_coverage") is not None:
            try:
                cov_val = float(data["calorie_coverage"])
                if math.isfinite(cov_val):
                    calorie_coverage = cov_val
                    metadata.setdefault("calorie_coverage", cov_val)
            except (TypeError, ValueError):
                pass
        for extra_key in ("source", "link"):
            if extra_key in data and extra_key not in metadata:
                metadata[extra_key] = data[extra_key]

        raw_cal = data.get("total_calories")
        parsed_cal: float | None = None
        if raw_cal is not None:
            try:
                cal_val = float(raw_cal)
                if math.isfinite(cal_val):
                    # Corpus build emits 0.0 when calorie_coverage == 0.0
                    # (no ingredients matched); treat that as unavailable calories.
                    if not (cal_val == 0.0 and calorie_coverage == 0.0):
                        parsed_cal = cal_val
            except (TypeError, ValueError):
                parsed_cal = None

        raw_tfidf = data.get("tfidf_score", 0.0)
        try:
            tfidf_val = float(raw_tfidf)
            if not math.isfinite(tfidf_val):
                tfidf_val = 0.0
        except (TypeError, ValueError):
            tfidf_val = 0.0

        return cls(
            recipe_id=data["recipe_id"],
            title=str(data.get("title", "")),
            canonical_ingredients=list(data.get("canonical_ingredients") or []),
            tfidf_score=tfidf_val,
            ingredients=list(data.get("ingredients") or []),
            directions=list(data.get("directions") or []),
            total_calories=parsed_cal,
            metadata=metadata,
        )

@dataclass(frozen=True)
class RankingConfig:
    """Configurable weights and filter switches for Pipeline 3 (PRD Section 24).

    Initial weights follow PRD Section 24 and are experimental until validated
    through the ablation study (PRD Section 35 / BEAM note).
    """

    alpha: float = 0.30  # TF-IDF similarity
    beta: float = 0.35   # Ingredient coverage
    gamma: float = 0.20  # Calorie compatibility
    delta: float = 0.15  # Missing ingredient penalty

    use_pantry_staple_weights: bool = True
    use_weighted_missing_penalty: bool = True
    hard_calorie_filter: bool = True
    hard_exclusion_filter: bool = True
    enable_calorie_score: bool = True

    enable_fallback: bool = True
    fill_fallback_to_top_k: bool = True
    fallback_exclusion_penalty: float = 0.50

    unknown_calorie_score: float = 0.50
    no_calorie_constraint_score: float = 1.0
    reject_unknown_calorie_on_hard_bound: bool = True
    normalize_servings_from_directions: bool = False

    @property
    def max_positive_score(self) -> float:
        return self.alpha + self.beta + (self.gamma if self.enable_calorie_score else 0.0)

    @classmethod
    def ablation_model_a(cls) -> RankingConfig:
        """Model A (Baseline): TF-IDF cosine similarity only (PRD Section 35)."""
        return cls(
            alpha=1.0,
            beta=0.0,
            gamma=0.0,
            delta=0.0,
            use_pantry_staple_weights=False,
            use_weighted_missing_penalty=False,
            hard_calorie_filter=False,
            hard_exclusion_filter=False,
            enable_calorie_score=False,
            enable_fallback=False,
        )

    @classmethod
    def ablation_model_b(cls) -> RankingConfig:
        """Model B: TF-IDF + Ingredient Coverage (PRD Section 35)."""
        return cls(
            alpha=0.30,
            beta=0.35,
            gamma=0.0,
            delta=0.0,
            use_pantry_staple_weights=True,
            use_weighted_missing_penalty=False,
            hard_calorie_filter=False,
            hard_exclusion_filter=False,
            enable_calorie_score=False,
            enable_fallback=False,
        )

    @classmethod
    def ablation_model_c(cls) -> RankingConfig:
        """Model C: TF-IDF + Coverage + Missing Ingredient Penalty (PRD Section 35)."""
        return cls(
            alpha=0.30,
            beta=0.35,
            gamma=0.0,
            delta=0.15,
            use_pantry_staple_weights=True,
            use_weighted_missing_penalty=True,
            hard_calorie_filter=False,
            hard_exclusion_filter=False,
            enable_calorie_score=False,
            enable_fallback=False,
        )

    @classmethod
    def ablation_model_d(cls) -> RankingConfig:
        """Model D (Full Proposed): All criteria + Calorie & Exclusion filters (PRD Section 35)."""
        return cls()

    @classmethod
    def ablation_unweighted(cls) -> RankingConfig:
        """Full Proposed variant without pantry staple down-weighting (data/README.md check)."""
        return cls(
            use_pantry_staple_weights=False,
            use_weighted_missing_penalty=False,
        )

@dataclass
class CoverageBreakdown:
    """Detailed ingredient coverage and missing ingredient metrics (PRD Sections 19-21)."""

    coverage_score: float
    missing_penalty: float
    weighted_coverage: float
    unweighted_coverage: float
    weighted_missing_penalty: float
    unweighted_missing_penalty: float
    available_matched: list[str]
    missing_ingredients: list[str]
    missing_non_staples: list[str]
    missing_staples: list[str]
    recipe_ingredients: list[str]
    ingredient_weights: dict[str, float]

    @property
    def missing_count(self) -> int:
        return len(self.missing_ingredients)

    @property
    def missing_non_staple_count(self) -> int:
        return len(self.missing_non_staples)

@dataclass
class CalorieBreakdown:
    """Detailed calorie compatibility assessment (PRD Section 22)."""

    score: float
    estimated_calories: float | None
    calorie_distance: float | None
    passed_hard_filter: bool
    constraint_type: str  # "none" | "target" | "max" | "min" | "range" | "target+bounds"
    status_label: str

@dataclass
class ExclusionBreakdown:
    """Detailed excluded ingredient check (PRD Section 23)."""

    violated: bool
    violation_count: int
    violated_ingredients: list[str]
    matched_rules: dict[str, list[str]]  # excluded_term -> matched recipe ingredients

@dataclass
class RankedRecipe:
    """A scored, explainable recipe recommendation produced by Pipeline 3."""

    rank: int
    recipe: CandidateRecipe
    final_score: float
    overall_match_pct: int
    tfidf_score: float
    coverage: CoverageBreakdown
    calorie: CalorieBreakdown
    exclusion: ExclusionBreakdown
    is_fallback_candidate: bool = False
    relaxed_constraints: list[str] = field(default_factory=list)

@dataclass
class RecommendationResult:
    """Top-K reranking output including filtering telemetry and fallback status."""

    query: StructuredQuery
    recommendations: list[RankedRecipe]
    total_candidates: int
    filtered_out_exclusion: int = 0
    filtered_out_calorie: int = 0
    strict_match_count: int = 0
    is_fallback: bool = False
    fallback_message: str | None = None
    relaxed_constraints: list[str] = field(default_factory=list)


**`src/pipeline3/coverage.py`**

Ingredient coverage, missing ingredient penalty, and pantry staple weights (PRD §§19-21).


In [ ]:
class PantryStapleWeights:
    """Lookup table for pantry staple weights loaded from ``pantry_staples.csv`` (PRD §21).

    Common pantry staples (e.g. water=0.05, salt=0.15, pepper=0.15) receive lower
    weights so they do not disproportionately penalize recipes when omitted from
    the user's query. Ingredients absent from the table default to ``1.0``.
    """

    def __init__(
        self,
        weights: Mapping[str, float] | None = None,
        default_weight: float = 1.0,
        normalizer: IngredientNormalizer | None = None,
    ) -> None:
        self.default_weight = float(default_weight)
        self.normalizer = normalizer
        self.weights: dict[str, float] = {}
        if weights:
            for key, value in weights.items():
                norm = normalize_ingredient(key)
                if norm:
                    self.weights[norm] = float(value)
                if normalizer is not None:
                    canon = normalizer.canonicalize(key, drop_qualifiers=False)
                    if canon and canon not in self.weights:
                        self.weights[canon] = float(value)

    @classmethod
    def from_csv(
        cls,
        path: Path | None = None,
        normalizer: IngredientNormalizer | None = None,
        default_weight: float = 1.0,
    ) -> PantryStapleWeights:
        csv_path = Path(path) if path is not None else DEFAULT_PROCESSED_DIR / "pantry_staples.csv"
        weights: dict[str, float] = {}
        if csv_path.exists():
            for row in read_csv_rows(csv_path):
                ingredient = row.get("ingredient", "").strip()
                weight_str = row.get("weight", "").strip()
                if not ingredient or not weight_str:
                    continue
                try:
                    w = float(weight_str)
                except ValueError:
                    continue
                norm = normalize_ingredient(ingredient)
                if norm:
                    weights[norm] = w
        return cls(weights=weights, default_weight=default_weight, normalizer=normalizer)

    def get_weight(self, ingredient: str) -> float:
        norm = normalize_ingredient(ingredient)
        if not norm:
            return self.default_weight
        if norm in self.weights:
            return self.weights[norm]
        if self.normalizer is not None:
            canon = self.normalizer.canonicalize(norm, drop_qualifiers=False)
            if canon in self.weights:
                return self.weights[canon]
        return self.default_weight

    def is_staple(self, ingredient: str, threshold: float = 0.5) -> bool:
        """Return True if an ingredient's weight is below ``threshold``."""
        return self.get_weight(ingredient) < threshold

def compute_coverage(
    user_ingredients: Iterable[str],
    recipe_ingredients: Iterable[str],
    staple_weights: PantryStapleWeights | None = None,
    normalizer: IngredientNormalizer | None = None,
    use_weights: bool = True,
    use_weighted_missing_penalty: bool = True,
    staple_threshold: float = 0.5,
) -> CoverageBreakdown:
    """Compute ingredient coverage and missing ingredient penalty (PRD §§19-21).

    Unweighted formulas (PRD §§19-20):
        S_coverage_unweighted = |U ∩ R| / |R|
        P_missing_unweighted  = |R - U| / |R|

    Weighted formulas with pantry staples (PRD §21):
        S_coverage_weighted   = sum(w_i for i in U ∩ R) / sum(w_i for i in R)
        P_missing_weighted    = sum(w_i for i in R - U) / sum(w_i for i in R)
    """
    if normalizer is not None:
        user_list = normalizer.canonicalize_list(user_ingredients)
        recipe_list = normalizer.canonicalize_list(recipe_ingredients)
    else:
        user_list = list(dict.fromkeys(n for i in user_ingredients if (n := normalize_ingredient(i))))
        recipe_list = list(dict.fromkeys(n for i in recipe_ingredients if (n := normalize_ingredient(i))))

    user_set = set(user_list)

    if not recipe_list:
        return CoverageBreakdown(
            coverage_score=0.0,
            missing_penalty=0.0,
            weighted_coverage=0.0,
            unweighted_coverage=0.0,
            weighted_missing_penalty=0.0,
            unweighted_missing_penalty=0.0,
            available_matched=[],
            missing_ingredients=[],
            missing_non_staples=[],
            missing_staples=[],
            recipe_ingredients=[],
            ingredient_weights={},
        )

    available_matched = [ing for ing in recipe_list if ing in user_set]
    missing_ingredients = [ing for ing in recipe_list if ing not in user_set]

    unweighted_coverage = len(available_matched) / len(recipe_list)
    unweighted_missing_penalty = len(missing_ingredients) / len(recipe_list)

    weights_map: dict[str, float] = {}
    for ing in recipe_list:
        weights_map[ing] = staple_weights.get_weight(ing) if staple_weights is not None else 1.0

    total_weight = sum(weights_map.values())
    if total_weight > 0:
        matched_weight = sum(weights_map[ing] for ing in available_matched)
        missing_weight = sum(weights_map[ing] for ing in missing_ingredients)
        weighted_coverage = matched_weight / total_weight
        weighted_missing_penalty = missing_weight / total_weight
    else:
        weighted_coverage = unweighted_coverage
        weighted_missing_penalty = unweighted_missing_penalty

    missing_non_staples = [ing for ing in missing_ingredients if weights_map[ing] >= staple_threshold]
    missing_staples = [ing for ing in missing_ingredients if weights_map[ing] < staple_threshold]

    coverage_score = weighted_coverage if use_weights else unweighted_coverage
    missing_penalty = (
        weighted_missing_penalty
        if (use_weights and use_weighted_missing_penalty)
        else unweighted_missing_penalty
    )

    return CoverageBreakdown(
        coverage_score=coverage_score,
        missing_penalty=missing_penalty,
        weighted_coverage=weighted_coverage,
        unweighted_coverage=unweighted_coverage,
        weighted_missing_penalty=weighted_missing_penalty,
        unweighted_missing_penalty=unweighted_missing_penalty,
        available_matched=available_matched,
        missing_ingredients=missing_ingredients,
        missing_non_staples=missing_non_staples,
        missing_staples=missing_staples,
        recipe_ingredients=recipe_list,
        ingredient_weights=weights_map,
    )


**`src/pipeline3/exclusion.py`**

Excluded ingredient detection and filtering (PRD Section 23).  Handles both exact canonical matches and generic-category exclusions such as "no cheese" (matching cheddar cheese, mozzarella cheese, parmesan, cream cheese, etc.) or "no nuts" (matching almond, walnut, pecan, peanut, mixed nuts) while guarding common false-positive compound names (e.g., "peanut butter" is not dairy butter; "cream of tartar" is not dairy cream; "nutmeg" is not a nut).  Note (PRD Section 6): Excluded ingredients are treated as user preference constraints, not medical allergy guarantees.


In [ ]:
# Explicit family expansions for generic exclusion categories observed in PRD
# and data/test/annotated_queries.json.
CATEGORY_EXPANSIONS: dict[str, frozenset[str]] = {
    "cheese": frozenset({
        "cheese",
        "cheddar",
        "cheddar cheese",
        "mozzarella",
        "mozzarella cheese",
        "parmesan",
        "parmesan cheese",
        "parmigiano",
        "swiss cheese",
        "cream cheese",
        "cottage cheese",
        "ricotta",
        "ricotta cheese",
        "process cheese",
        "american cheese",
        "velveeta",
        "feta",
        "feta cheese",
        "brie",
        "gouda",
        "gruyere",
        "provolone",
        "monterey jack",
        "colby",
        "blue cheese",
        "goat cheese",
        "mascarpone",
        "gorgonzola",
        "paneer",
    }),
    "milk": frozenset({
        "milk",
        "whole milk",
        "skim milk",
        "low fat milk",
        "buttermilk",
        "condensed milk",
        "evaporated milk",
        "half and half",
    }),
    "cream": frozenset({
        "cream",
        "heavy cream",
        "whipping cream",
        "sour cream",
        "cream cheese",
        "half and half",
        "ice cream",
        "cream of chicken soup",
        "cream of mushroom soup",
        "cream of celery soup",
    }),
    "butter": frozenset({
        "butter",
        "unsalted butter",
        "salted butter",
        "sweet butter",
        "melted butter",
        "garlic butter",
        "clarified butter",
        "ghee",
    }),
    "nut": frozenset({
        "nut",
        "mixed nut",
        "mixed nuts",
        "almond",
        "walnut",
        "pecan",
        "cashew",
        "peanut",
        "pistachio",
        "hazelnut",
        "macadamia",
        "pine nut",
        "brazil nut",
        "chestnut",
        "peanut butter",
        "almond butter",
        "cashew butter",
        "almond milk",
    }),
    "peanut": frozenset({
        "peanut",
        "peanut butter",
        "peanut oil",
    }),
}

# Compound names that contain an excluded word lexically but belong to a
# different food class and must NOT trigger false-positive exclusions.
FALSE_POSITIVE_GUARDS: dict[str, frozenset[str]] = {
    "butter": frozenset({
        "peanut butter",
        "almond butter",
        "cashew butter",
        "apple butter",
        "cocoa butter",
        "shea butter",
        "sunflower butter",
        "cookie butter",
        "buttermilk",
        "butternut squash",
        "butterscotch",
        "butterscotch chip",
    }),
    "cream": frozenset({
        "cream of tartar",
        "coconut cream",
    }),
    "milk": frozenset({
        "almond milk",
        "coconut milk",
        "soy milk",
        "oat milk",
        "rice milk",
    }),
    "nut": frozenset({
        "nutmeg",
        "butternut squash",
        "water chestnut",
        "coconut",
        "coconut milk",
        "coconut oil",
        "coconut cream",
    }),
    "egg": frozenset({
        "eggplant",
    }),
    "pea": frozenset({
        "peanut",
        "peanut butter",
        "peach",
        "pear",
    }),
}

def _contains_whole_phrase(text: str, phrase: str) -> bool:
    if not text or not phrase:
        return False
    return bool(re.search(rf"(?:^| ){re.escape(phrase)}(?:$| )", text))

class ExclusionFilter:
    """Checks candidate recipes against user-specified excluded ingredients (PRD §23)."""

    def __init__(self, normalizer: IngredientNormalizer | None = None) -> None:
        self.normalizer = normalizer

    def _matches(self, excluded_raw: str, excluded_canon: str, candidate_forms: set[str]) -> bool:
        guards = FALSE_POSITIVE_GUARDS.get(excluded_raw, frozenset()) | FALSE_POSITIVE_GUARDS.get(
            excluded_canon, frozenset()
        )
        # If any form of the candidate is explicitly guarded, skip matching it.
        if candidate_forms & guards:
            return False

        # 1. Exact match against either raw-normalized or canonicalized excluded term
        if excluded_raw in candidate_forms or (excluded_canon and excluded_canon in candidate_forms):
            return True

        # 2. Category expansion (e.g., excluded="cheese" matches "mozzarella cheese")
        expanded = CATEGORY_EXPANSIONS.get(excluded_raw)
        if expanded:
            if candidate_forms & expanded:
                return True
            if excluded_raw in ("cheese", "cream", "milk", "butter"):
                for form in candidate_forms:
                    if _contains_whole_phrase(form, excluded_raw):
                        return True
            return False

        # 3. Safe whole-phrase subtype match (e.g., "mushroom" matches "cream of mushroom soup",
        #    "onion" matches "green onion", "garlic" matches "garlic powder")
        for form in candidate_forms:
            if _contains_whole_phrase(form, excluded_raw):
                return True
            if excluded_canon and excluded_canon != excluded_raw and _contains_whole_phrase(
                form, excluded_canon
            ):
                return True
        return False

    def check(
        self,
        excluded_ingredients: Iterable[str],
        recipe_ingredients: Iterable[str],
    ) -> ExclusionBreakdown:
        """Evaluate excluded ingredient violations for a single recipe."""
        excluded_list = [e for e in excluded_ingredients if normalize_ingredient(e)]
        if not excluded_list:
            return ExclusionBreakdown(
                violated=False,
                violation_count=0,
                violated_ingredients=[],
                matched_rules={},
            )

        # Precompute normalized + canonicalized forms per recipe ingredient
        recipe_entries: list[tuple[str, set[str]]] = []
        for ing in recipe_ingredients:
            raw_norm = normalize_ingredient(ing)
            if not raw_norm:
                continue
            forms = {raw_norm}
            display_name = raw_norm
            if self.normalizer is not None:
                canon = self.normalizer.canonicalize(raw_norm)
                if canon:
                    forms.add(canon)
                    display_name = canon
            recipe_entries.append((display_name, forms))

        matched_rules: dict[str, list[str]] = {}
        violated_set: dict[str, None] = {}

        for exc in excluded_list:
            exc_raw = normalize_ingredient(exc)
            if not exc_raw:
                continue
            # Keep exc_raw intact for category matching (e.g. "cheese" must stay
            # "cheese" even though ingredient_aliases maps cheese -> cheddar cheese).
            exc_canon = exc_raw
            if self.normalizer is not None and exc_raw not in CATEGORY_EXPANSIONS:
                exc_canon = self.normalizer.canonicalize(exc_raw, drop_qualifiers=False)

            hits: list[str] = []
            for display_name, forms in recipe_entries:
                if self._matches(exc_raw, exc_canon, forms):
                    if display_name not in hits:
                        hits.append(display_name)
                    violated_set.setdefault(display_name, None)

            if hits:
                matched_rules[exc_raw] = hits

        violated_ingredients = list(violated_set)
        return ExclusionBreakdown(
            violated=len(violated_ingredients) > 0,
            violation_count=len(violated_ingredients),
            violated_ingredients=violated_ingredients,
            matched_rules=matched_rules,
        )


**`src/pipeline3/calorie_score.py`**

Calorie compatibility scoring and hard-bound checking (PRD Section 22).


In [ ]:
def evaluate_calorie_compatibility(
    estimated_calories: float | None,
    calorie_target: float | None = None,
    calorie_min: float | None = None,
    calorie_max: float | None = None,
    unknown_calorie_score: float = 0.50,
    no_constraint_score: float = 1.0,
    reject_unknown_on_hard_bound: bool = True,
) -> CalorieBreakdown:
    """Evaluate how well a recipe's estimated calories match user calorie constraints.

    Supported constraint types (PRD §§12.6 & 22):
    - ``none``   : no target/min/max specified -> neutral score (``no_constraint_score``)
    - ``target`` : approximate target C_t -> ``max(0, 1 - |C_r - C_t| / C_t)`` (soft)
    - ``max``    : upper bound C_r <= C_max (hard filter + proximity decay if relaxed)
    - ``min``    : lower bound C_r >= C_min (hard filter + proximity decay if relaxed)
    - ``range``  : C_min <= C_r <= C_max (hard filter + proximity decay if relaxed)
    - ``target+bounds``: combined target proximity with explicit min/max hard bounds
    """
    has_target = calorie_target is not None and calorie_target > 0
    has_min = calorie_min is not None
    has_max = calorie_max is not None

    if not (has_target or has_min or has_max):
        return CalorieBreakdown(
            score=no_constraint_score,
            estimated_calories=estimated_calories,
            calorie_distance=None,
            passed_hard_filter=True,
            constraint_type="none",
            status_label="no calorie constraint",
        )

    if has_target and (has_min or has_max):
        constraint_type = "target+bounds"
    elif has_target:
        constraint_type = "target"
    elif has_min and has_max:
        constraint_type = "range"
    elif has_max:
        constraint_type = "max"
    else:
        constraint_type = "min"

    if estimated_calories is None:
        has_hard_bound = has_min or has_max
        passed = not (has_hard_bound and reject_unknown_on_hard_bound)
        return CalorieBreakdown(
            score=unknown_calorie_score,
            estimated_calories=None,
            calorie_distance=None,
            passed_hard_filter=passed,
            constraint_type=constraint_type,
            status_label="estimated calories unavailable",
        )

    cal = float(estimated_calories)

    # Check hard bounds when present
    passed_min = True if not has_min else cal >= float(calorie_min)
    passed_max = True if not has_max else cal <= float(calorie_max)
    passed_hard = passed_min and passed_max

    if constraint_type == "target":
        target = float(calorie_target)
        distance = abs(cal - target)
        score = max(0.0, 1.0 - distance / target)
        label = f"approximately {round(target)} kcal (est. {round(cal)} kcal)"
        return CalorieBreakdown(
            score=score,
            estimated_calories=cal,
            calorie_distance=distance,
            passed_hard_filter=True,
            constraint_type=constraint_type,
            status_label=label,
        )

    if constraint_type == "target+bounds":
        target = float(calorie_target)
        distance = abs(cal - target)
        score = max(0.0, 1.0 - distance / target)
        bounds_desc = []
        if has_min:
            bounds_desc.append(f"min {round(float(calorie_min))}")
        if has_max:
            bounds_desc.append(f"max {round(float(calorie_max))}")
        label = f"around {round(target)} kcal ({', '.join(bounds_desc)} kcal; est. {round(cal)} kcal)"
        return CalorieBreakdown(
            score=score,
            estimated_calories=cal,
            calorie_distance=distance,
            passed_hard_filter=passed_hard,
            constraint_type=constraint_type,
            status_label=label,
        )

    if constraint_type == "range":
        cmin = float(calorie_min)
        cmax = float(calorie_max)
        if cmin <= cal <= cmax:
            distance = 0.0
            score = 1.0
        elif cal < cmin:
            distance = cmin - cal
            ref = max((cmin + cmax) / 2.0, 1.0)
            score = max(0.0, 1.0 - distance / ref)
        else:
            distance = cal - cmax
            ref = max((cmin + cmax) / 2.0, 1.0)
            score = max(0.0, 1.0 - distance / ref)
        label = f"between {round(cmin)} and {round(cmax)} kcal (est. {round(cal)} kcal)"
        return CalorieBreakdown(
            score=score,
            estimated_calories=cal,
            calorie_distance=distance,
            passed_hard_filter=passed_hard,
            constraint_type=constraint_type,
            status_label=label,
        )

    if constraint_type == "max":
        cmax = float(calorie_max)
        if cal <= cmax:
            distance = 0.0
            score = 1.0
        else:
            distance = cal - cmax
            score = max(0.0, 1.0 - distance / max(cmax, 1.0))
        label = f"under {round(cmax)} kcal (est. {round(cal)} kcal)"
        return CalorieBreakdown(
            score=score,
            estimated_calories=cal,
            calorie_distance=distance,
            passed_hard_filter=passed_hard,
            constraint_type=constraint_type,
            status_label=label,
        )

    # constraint_type == "min"
    cmin = float(calorie_min)
    if cal >= cmin:
        distance = 0.0
        score = 1.0
    else:
        distance = cmin - cal
        score = max(0.0, 1.0 - distance / max(cmin, 1.0))
    label = f"at least {round(cmin)} kcal (est. {round(cal)} kcal)"
    return CalorieBreakdown(
        score=score,
        estimated_calories=cal,
        calorie_distance=distance,
        passed_hard_filter=passed_hard,
        constraint_type=constraint_type,
        status_label=label,
    )


**`src/pipeline3/reranker.py`**

Multi-criteria recipe reranker with constraint filtering and fallback (PRD §§24-25 & Risk 6).


In [ ]:
def _coerce_query(query: StructuredQuery | Mapping[str, Any]) -> StructuredQuery:
    if isinstance(query, StructuredQuery):
        return query
    return StructuredQuery.from_dict(query)

def _coerce_candidate(candidate: CandidateRecipe | Mapping[str, Any]) -> CandidateRecipe:
    if isinstance(candidate, CandidateRecipe):
        return candidate
    return CandidateRecipe.from_dict(candidate)

class MultiCriteriaReranker:
    """Pipeline 3 reranker combining TF-IDF, pantry coverage, calories, and exclusions.

    Final scoring formula (PRD Section 24):
        Score(r, q) = alpha * S_tfidf + beta * S_coverage + gamma * S_calorie - delta * P_missing
    """

    def __init__(
        self,
        config: RankingConfig | None = None,
        processed_dir: Path | None = None,
        normalizer: IngredientNormalizer | None = None,
        staple_weights: PantryStapleWeights | None = None,
        exclusion_filter: ExclusionFilter | None = None,
    ) -> None:
        self.config = config or RankingConfig()
        data_dir = Path(processed_dir) if processed_dir is not None else DEFAULT_PROCESSED_DIR
        self.normalizer = normalizer or IngredientNormalizer(data_dir)
        self.staple_weights = staple_weights or PantryStapleWeights.from_csv(
            data_dir / "pantry_staples.csv",
            normalizer=self.normalizer,
        )
        self.exclusion_filter = exclusion_filter or ExclusionFilter(normalizer=self.normalizer)

    def score_candidate(
        self,
        candidate: CandidateRecipe | Mapping[str, Any],
        query: StructuredQuery | Mapping[str, Any],
        config: RankingConfig | None = None,
        apply_exclusion_soft_penalty: bool = False,
    ) -> RankedRecipe:
        """Compute all component scores and the final weighted score for one candidate."""
        cfg = config or self.config
        cand = _coerce_candidate(candidate)
        sq = _coerce_query(query)

        cov = compute_coverage(
            user_ingredients=sq.available_ingredients,
            recipe_ingredients=cand.canonical_ingredients,
            staple_weights=self.staple_weights,
            normalizer=self.normalizer,
            use_weights=cfg.use_pantry_staple_weights,
            use_weighted_missing_penalty=cfg.use_weighted_missing_penalty,
        )

        effective_calories = cand.total_calories
        if (
            cfg.normalize_servings_from_directions
            and effective_calories is not None
            and (cand.directions or cand.title)
        ):
            servings = extract_servings(cand.directions, title=cand.title)
            if servings is not None and servings > 1:
                effective_calories = round(effective_calories / servings, 1)

        cal = evaluate_calorie_compatibility(
            estimated_calories=effective_calories,
            calorie_target=sq.calorie_target,
            calorie_min=sq.calorie_min,
            calorie_max=sq.calorie_max,
            unknown_calorie_score=cfg.unknown_calorie_score,
            no_constraint_score=cfg.no_calorie_constraint_score,
            reject_unknown_on_hard_bound=cfg.reject_unknown_calorie_on_hard_bound,
        )

        excl = self.exclusion_filter.check(
            excluded_ingredients=sq.excluded_ingredients,
            recipe_ingredients=cand.canonical_ingredients,
        )

        s_tfidf = max(0.0, min(1.0, float(cand.tfidf_score)))
        s_cov = cov.coverage_score
        s_cal = cal.score if cfg.enable_calorie_score else 0.0
        p_miss = cov.missing_penalty

        final_score = (
            cfg.alpha * s_tfidf
            + cfg.beta * s_cov
            + (cfg.gamma * s_cal if cfg.enable_calorie_score else 0.0)
            - cfg.delta * p_miss
        )

        if apply_exclusion_soft_penalty and excl.violated:
            final_score -= cfg.fallback_exclusion_penalty * excl.violation_count

        max_pos = cfg.max_positive_score
        if max_pos > 0:
            match_ratio = max(0.0, min(1.0, final_score / max_pos))
        else:
            match_ratio = 0.0
        overall_match_pct = round(match_ratio * 100)

        return RankedRecipe(
            rank=0,
            recipe=cand,
            final_score=round(final_score, 6),
            overall_match_pct=overall_match_pct,
            tfidf_score=s_tfidf,
            coverage=cov,
            calorie=cal,
            exclusion=excl,
        )

    @staticmethod
    def _sort_key(item: RankedRecipe) -> tuple:
        return (
            -item.final_score,
            -item.coverage.coverage_score,
            item.coverage.missing_count,
            -item.tfidf_score,
            str(item.recipe.recipe_id),
        )

    def rerank(
        self,
        candidates: Sequence[CandidateRecipe | Mapping[str, Any]],
        query: StructuredQuery | Mapping[str, Any],
        top_k: int = 3,
        config: RankingConfig | None = None,
    ) -> RecommendationResult:
        """Filter and rerank candidate recipes, applying Risk 6 fallback if needed.

        Accepts either typed ``CandidateRecipe`` / ``StructuredQuery`` objects
        or raw dicts directly from Pipeline 1 (``parse_query``) and Pipeline 2
        (``CandidateRetriever.retrieve``).
        """
        cfg = config or self.config
        sq = _coerce_query(query)
        coerced_candidates = [_coerce_candidate(c) for c in candidates]
        if top_k <= 0 or not coerced_candidates:
            return RecommendationResult(
                query=sq,
                recommendations=[],
                total_candidates=len(coerced_candidates),
            )

        scored_all = [self.score_candidate(c, sq, config=cfg) for c in coerced_candidates]

        strict_matches: list[RankedRecipe] = []
        calorie_rejected: list[RankedRecipe] = []
        exclusion_rejected: list[RankedRecipe] = []

        for item in scored_all:
            if cfg.hard_exclusion_filter and item.exclusion.violated:
                exclusion_rejected.append(item)
                continue
            if cfg.hard_calorie_filter and not item.calorie.passed_hard_filter:
                calorie_rejected.append(item)
                continue
            strict_matches.append(item)

        strict_matches.sort(key=self._sort_key)

        selected: list[RankedRecipe] = list(strict_matches[:top_k])
        is_fallback = False
        relaxed: list[str] = []
        fallback_message: str | None = None

        need_fallback = (
            cfg.enable_fallback
            and (
                len(strict_matches) == 0
                or (cfg.fill_fallback_to_top_k and len(strict_matches) < top_k)
            )
        )

        if need_fallback:
            remaining = top_k - len(selected)

            # Tier 1 fallback: relax calorie hard bounds while keeping exclusions strict
            if remaining > 0 and calorie_rejected:
                calorie_rejected.sort(key=self._sort_key)
                for item in calorie_rejected[:remaining]:
                    selected.append(
                        replace(
                            item,
                            is_fallback_candidate=True,
                            relaxed_constraints=["calorie_bounds"],
                        )
                    )
                if "calorie_bounds" not in relaxed:
                    relaxed.append("calorie_bounds")
                remaining = top_k - len(selected)

            # Tier 2 fallback: only if still short, relax exclusion filter with heavy penalty
            if remaining > 0 and exclusion_rejected:
                penalized = [
                    replace(
                        self.score_candidate(
                            item.recipe,
                            sq,
                            config=cfg,
                            apply_exclusion_soft_penalty=True,
                        ),
                        is_fallback_candidate=True,
                        relaxed_constraints=(
                            ["excluded_ingredients", "calorie_bounds"]
                            if (cfg.hard_calorie_filter and not item.calorie.passed_hard_filter)
                            else ["excluded_ingredients"]
                        ),
                    )
                    for item in exclusion_rejected
                ]
                penalized.sort(key=self._sort_key)
                for item in penalized[:remaining]:
                    selected.append(item)
                    for constraint_name in item.relaxed_constraints:
                        if constraint_name not in relaxed:
                            relaxed.append(constraint_name)

            if relaxed:
                is_fallback = True
                relaxed_str = ", ".join(relaxed)
                if len(strict_matches) == 0:
                    fallback_message = (
                        f"No exact recommendation found. Showing closest alternatives "
                        f"with relaxed constraints: {relaxed_str}."
                    )
                else:
                    fallback_message = (
                        f"Only {len(strict_matches)} exact match(es) found for Top-{top_k}. "
                        f"Remaining slots show closest alternatives with relaxed constraints: {relaxed_str}."
                    )

        ranked_final = [
            replace(item, rank=idx)
            for idx, item in enumerate(selected, start=1)
        ]

        return RecommendationResult(
            query=sq,
            recommendations=ranked_final,
            total_candidates=len(coerced_candidates),
            filtered_out_exclusion=len(exclusion_rejected),
            filtered_out_calorie=len(calorie_rejected),
            strict_match_count=len(strict_matches),
            is_fallback=is_fallback,
            fallback_message=fallback_message,
            relaxed_constraints=relaxed,
        )


**`src/pipeline3/recommender.py`**

Explainable recommendation entry point, formatting, and serialization (PRD §§26-27).


In [ ]:
def recommend(
    candidates: Sequence[CandidateRecipe | Mapping[str, Any]],
    query: StructuredQuery | Mapping[str, Any],
    top_k: int = 3,
    config: RankingConfig | None = None,
    reranker: MultiCriteriaReranker | None = None,
    processed_dir: Path | None = None,
) -> RecommendationResult:
    """High-level Pipeline 3 entry point accepting raw Pipeline 1 & 2 dicts or typed models."""
    engine = reranker or MultiCriteriaReranker(config=config, processed_dir=processed_dir)
    return engine.rerank(candidates=candidates, query=query, top_k=top_k, config=config)

def format_recommendation_card(
    ranked: RankedRecipe,
    excluded_ingredients: Iterable[str] = (),
    include_ingredients: bool = True,
    include_directions: bool = True,
) -> str:
    """Format a single ranked recipe as an explainable text card (PRD Section 26)."""
    recipe = ranked.recipe
    cov = ranked.coverage
    cal = ranked.calorie
    excl = ranked.exclusion

    cov_pct = round(cov.coverage_score * 100)
    cal_pct = round(cal.score * 100)
    cal_str = f"{round(cal.estimated_calories)} kcal" if cal.estimated_calories is not None else "N/A (not estimated)"

    header = f"#{ranked.rank} {recipe.title}"
    if ranked.is_fallback_candidate:
        header += f" [Closest Alternative — relaxed: {', '.join(ranked.relaxed_constraints)}]"

    lines = [
        header,
        f"Overall Match       : {ranked.overall_match_pct}% (score={ranked.final_score:.3f})",
        f"Ingredient Coverage : {cov_pct}% ({len(cov.available_matched)}/{len(cov.recipe_ingredients)} ingredients)",
        f"Estimated Calories  : {cal_str}",
    ]
    if cal.constraint_type != "none":
        lines.append(f"Calorie Match       : {cal_pct}%")

    lines.append("")
    lines.append("YOU HAVE:")
    if cov.available_matched:
        for ing in cov.available_matched:
            lines.append(f"  ✓ {ing}")
    else:
        lines.append("  (none matched)")

    lines.append("")
    lines.append("MISSING:")
    if cov.missing_ingredients:
        for ing in cov.missing_non_staples:
            lines.append(f"  ✗ {ing}")
        for ing in cov.missing_staples:
            lines.append(f"  ✗ {ing} (pantry staple)")
    else:
        lines.append("  ✓ nothing missing!")

    lines.append("")
    lines.append("CONSTRAINTS:")
    if cal.constraint_type == "none":
        lines.append("  ✓ no calorie constraint specified")
    elif cal.passed_hard_filter:
        lines.append(f"  ✓ {cal.status_label}")
    else:
        lines.append(f"  ✗ relaxed calorie bound: {cal.status_label}")

    excl_list = [e for e in excluded_ingredients if e]
    if excl.violated:
        lines.append(f"  ✗ excluded ingredient detected: {', '.join(excl.violated_ingredients)}")
    elif excl_list:
        lines.append(f"  ✓ no {', '.join(excl_list)} detected")
    else:
        lines.append("  ✓ no excluded ingredients detected")

    if include_ingredients and recipe.ingredients:
        lines.append("")
        lines.append("INGREDIENTS:")
        for raw_ing in recipe.ingredients:
            lines.append(f"  - {raw_ing}")

    if include_directions and recipe.directions:
        lines.append("")
        lines.append("DIRECTIONS:")
        for idx, step in enumerate(recipe.directions, start=1):
            lines.append(f"  {idx}. {step}")

    return "\n".join(lines)

def format_recommendations(
    result: RecommendationResult,
    include_ingredients: bool = True,
    include_directions: bool = True,
) -> str:
    """Render a full ``RecommendationResult`` into human-readable explainable text."""
    blocks: list[str] = []
    if result.is_fallback and result.fallback_message:
        blocks.append(f"[NOTICE] {result.fallback_message}")

    if not result.recommendations:
        blocks.append("No exact recommendation found.")
        return "\n\n".join(blocks)

    for ranked in result.recommendations:
        blocks.append(
            format_recommendation_card(
                ranked,
                excluded_ingredients=result.query.excluded_ingredients,
                include_ingredients=include_ingredients,
                include_directions=include_directions,
            )
        )
    return "\n\n" + ("\n\n" + "-" * 60 + "\n\n").join(blocks)

def to_dict(result: RecommendationResult) -> dict[str, Any]:
    """Serialize ``RecommendationResult`` into a JSON-friendly explainable dict."""
    return {
        "query": {
            "raw_query": result.query.raw_query,
            "available_ingredients": list(result.query.available_ingredients),
            "excluded_ingredients": list(result.query.excluded_ingredients),
            "calorie_target": result.query.calorie_target,
            "calorie_min": result.query.calorie_min,
            "calorie_max": result.query.calorie_max,
        },
        "total_candidates": result.total_candidates,
        "strict_match_count": result.strict_match_count,
        "filtered_out_exclusion": result.filtered_out_exclusion,
        "filtered_out_calorie": result.filtered_out_calorie,
        "is_fallback": result.is_fallback,
        "fallback_message": result.fallback_message,
        "relaxed_constraints": list(result.relaxed_constraints),
        "recommendations": [
            {
                "rank": item.rank,
                "recipe_id": item.recipe.recipe_id,
                "title": item.recipe.title,
                "final_score": item.final_score,
                "overall_match_pct": item.overall_match_pct,
                "tfidf_score": round(item.tfidf_score, 4),
                "coverage_score": round(item.coverage.coverage_score, 4),
                "weighted_coverage": round(item.coverage.weighted_coverage, 4),
                "unweighted_coverage": round(item.coverage.unweighted_coverage, 4),
                "missing_penalty": round(item.coverage.missing_penalty, 4),
                "available_matched": item.coverage.available_matched,
                "missing_ingredients": item.coverage.missing_ingredients,
                "missing_non_staples": item.coverage.missing_non_staples,
                "missing_staples": item.coverage.missing_staples,
                "estimated_calories": item.calorie.estimated_calories,
                "calorie_coverage": item.recipe.metadata.get("calorie_coverage"),
                "calorie_score": round(item.calorie.score, 4),
                "calorie_distance": item.calorie.calorie_distance,
                "calorie_status": item.calorie.status_label,
                "exclusion_violated": item.exclusion.violated,
                "violated_ingredients": item.exclusion.violated_ingredients,
                "is_fallback_candidate": item.is_fallback_candidate,
                "relaxed_constraints": item.relaxed_constraints,
                "ingredients": item.recipe.ingredients,
                "directions": item.recipe.directions,
            }
            for item in result.recommendations
        ],
    }


## 5. Skrip Data & Pelatihan — `scripts/`

Seluruh skrip CLI menjadi fungsi `run_*` (argparse -> parameter dengan default yang sama),
dipanggil lewat flag `RUN_*` di cell guard. Tidak ada yang jalan kecuali flag-nya dinyalakan.


**`scripts/ingredient_tables.py`**

Dataset tooling normalization — the single source lives in src/preprocessing.  Re-exports the same rules Pipeline 1-3 use (fold + head-noun singularization, alias resolution) so dataset tooling can no longer drift from pipeline code.


In [ ]:
PROCESSED = ROOT / "data" / "processed"

# alias dari src.preprocessing.ingredient_normalizer (sudah didefinisikan di section 1)
Resolver = IngredientNormalizer
normalize = normalize_ingredient
read_rows = read_csv_rows
# singularize sudah tersedia di namespace


**`scripts/download_raw_data.py`**

Fetch every external dataset YummyTummy needs (stdlib only).  Dataset inventory and rationale: data/README.md.  python scripts/download_raw_data.py --calories          # USDA FoodData Central (~7 MB) python scripts/download_raw_data.py --recipes sample    # ~7 MB random sample python scripts/download_raw_data.py --recipes full      # 2.3 GB full RecipeNLG corpus python scripts/download_raw_data.py --all               # calories + sample


In [ ]:
RAW_RECIPES = ROOT / "data" / "raw" / "recipenlg"
RAW_CALORIES = ROOT / "data" / "raw" / "calories"
SAMPLES = ROOT / "data" / "samples"

# RecipeNLG original columns: title, ingredients, directions, link, source, NER.
# HuggingFace mirrors are used because recipenlg.cs.put.poznan.pl has been
# unreachable since 2025 and Kaggle needs an authenticated API token.
RECIPE_MIRRORS = [
    "https://huggingface.co/datasets/Mahimas/recipenlg/resolve/main/RecipeNLG.csv",
    "https://huggingface.co/datasets/SandhyaKilari/RecipeNLG_dataset/resolve/main/RecipeNLG_dataset.csv",
]
RECIPE_COLUMNS = ["title", "ingredients", "directions", "link", "source", "NER"]

# USDA FoodData Central, public domain (CC0).
# SR Legacy only: 7,793 foods, the last complete generic-ingredient release.
# Foundation Foods switched to a per-sample schema without food_nutrient.csv.
FDC_FILES = {
    "sr_legacy": "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_sr_legacy_food_csv_2018-04.zip",
}

# The corpus is ordered by source, so a prefix sample would be single-source.
# With the full file on disk we take a uniform random sample; without it we fall
# back to equal-size byte windows spread across the whole file.
SAMPLE_ROWS = 10_000
SAMPLE_SEED = 42
SAMPLE_CHUNKS = 12
SAMPLE_CHUNK_BYTES = 800_000

def _download(url: str, dest: Path) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": "yummytummy-dataset-prep/1.0"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        total = int(resp.headers.get("Content-Length") or 0)
        seen = 0
        t0 = time.time()
        with dest.open("wb") as fh:
            while chunk := resp.read(1 << 20):
                fh.write(chunk)
                seen += len(chunk)
                if total:
                    print(f"\r  {dest.name}: {seen / total:6.1%} ({seen >> 20} / {total >> 20} MB)", end="")
                else:
                    print(f"\r  {dest.name}: {seen >> 20} MB", end="")
        print(f"\r  {dest.name}: {seen >> 20} MB in {time.time() - t0:.0f}s")

def _range(url: str, start: int, end: int) -> bytes:
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "yummytummy-dataset-prep/1.0",
            "Range": f"bytes={start}-{end}",
        },
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return resp.read()

def _corpus_size(url: str) -> int:
    req = urllib.request.Request(url, headers={"User-Agent": "yummytummy-dataset-prep/1.0", "Range": "bytes=0-0"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        content_range = resp.headers.get("Content-Range", "")
        if "/" in content_range:
            return int(content_range.rsplit("/", 1)[1])
        return int(resp.headers["Content-Length"])

def _parse_recipe_row(row: list[str], keep: list[int]) -> list[str] | None:
    """Return the six RecipeNLG columns, or None for a truncated/corrupt row."""
    if len(row) <= max(keep):
        return None
    try:
        ast.literal_eval(row[keep[1]])  # ingredients
        ast.literal_eval(row[keep[-1]])  # NER
    except (ValueError, SyntaxError, TypeError):
        return None
    return [row[i] for i in keep]

def _normalize_recipes(chunks: list[bytes], dest: Path) -> int:
    """Drop the unnamed pandas index column, keep the six RecipeNLG columns."""
    header: list[str] | None = None
    keep: list[int] = []
    written = 0
    with dest.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(RECIPE_COLUMNS)
        for index, raw in enumerate(chunks):
            rows = list(csv.reader(io.StringIO(raw.decode("utf-8", errors="ignore"), newline="")))
            if header is None:
                header = rows.pop(0)
                keep = [header.index(c) for c in RECIPE_COLUMNS]
            else:
                rows.pop(0)  # starts inside a previous record
            if not raw.endswith(b"\n"):
                rows.pop()  # last record is cut off
            for row in rows:
                record = _parse_recipe_row(row, keep)
                if record:
                    writer.writerow(record)
                    written += 1
    return written

def _write_sample(records: list[list[str]], dest: Path) -> int:
    with dest.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(RECIPE_COLUMNS)
        writer.writerows(records)
    return len(records)

def _sample_local(path: Path, count: int, seed: int) -> list[list[str]]:
    """Reservoir sample `count` valid records in corpus order (one pass over the file)."""
    rng = random.Random(seed)
    picked: list[tuple[int, list[str]]] = []
    keep: list[int] = []
    seen = 0
    with path.open(encoding="utf-8", newline="") as fh:
        reader = csv.reader(fh)
        header = next(reader)
        keep = [header.index(c) for c in RECIPE_COLUMNS]
        for row in reader:
            if len(row) <= max(keep):  # truncated row
                continue
            if len(picked) < count:
                picked.append((seen, row))
            else:
                slot = rng.randrange(seen + 1)
                if slot < count:
                    picked[slot] = (seen, row)
            seen += 1
    picked.sort(key=lambda item: item[0])
    records = []
    for _, row in picked:
        record = _parse_recipe_row(row, keep)
        if record:
            records.append(record)
    print(f"[info] {seen} corpus rows scanned, {len(records)} sampled")
    return records

def fetch_recipe_sample(force: bool = False) -> Path:
    dest = SAMPLES / "recipes_sample.csv"
    if dest.exists() and not force:
        print(f"[skip] {dest.relative_to(ROOT)} exists")
        return dest
    SAMPLES.mkdir(parents=True, exist_ok=True)
    full = RAW_RECIPES / "RecipeNLG.csv"
    if full.exists():
        print(f"[recipes] sampling {SAMPLE_ROWS} rows from local {full.relative_to(ROOT)}")
        n = _write_sample(_sample_local(full, SAMPLE_ROWS, SAMPLE_SEED), dest)
        print(f"[ok] {dest.relative_to(ROOT)}: {n} recipes")
        return dest
    last_err: Exception | None = None
    for url in RECIPE_MIRRORS:
        try:
            size = _corpus_size(url)
            print(f"[recipes] sampling {SAMPLE_CHUNKS} windows of {SAMPLE_CHUNK_BYTES >> 10} KB from {size / 1e9:.2f} GB")
            chunks = []
            for i in range(SAMPLE_CHUNKS):
                start = i * size // SAMPLE_CHUNKS
                blob = _range(url, start, min(start + SAMPLE_CHUNK_BYTES - 1, size - 1))
                if len(blob) < SAMPLE_CHUNK_BYTES // 2:
                    raise RuntimeError(f"short range response at byte {start}")
                chunks.append(blob)
            n = _normalize_recipes(chunks, dest)  # network fallback
            print(f"[ok] {dest.relative_to(ROOT)}: {n} recipes")
            return dest
        except Exception as err:  # noqa: BLE001 - try the next mirror
            last_err = err
            print(f"[warn] mirror failed: {err}")
    raise SystemExit(f"all RecipeNLG mirrors failed: {last_err}")

def fetch_recipe_full(force: bool = False) -> Path:
    dest = RAW_RECIPES / "RecipeNLG.csv"
    if dest.exists() and not force:
        print(f"[skip] {dest.relative_to(ROOT)} exists ({dest.stat().st_size >> 20} MB)")
        return dest
    RAW_RECIPES.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".csv.part")
    last_err: Exception | None = None
    for url in RECIPE_MIRRORS:
        try:
            print(f"[recipes] downloading 2.3 GB corpus from {url}")
            _download(url, tmp)
            tmp.replace(dest)
            with dest.open(encoding="utf-8", newline="") as fh:
                header = next(csv.reader(fh))
            missing = [c for c in RECIPE_COLUMNS if c not in header]
            if missing:
                raise SystemExit(f"{dest.name}: missing columns {missing}, got {header}")
            print(f"[ok] {dest.relative_to(ROOT)} ({dest.stat().st_size >> 20} MB)")
            return dest
        except SystemExit:
            raise
        except Exception as err:  # noqa: BLE001
            last_err = err
            print(f"[warn] mirror failed: {err}")
    tmp.unlink(missing_ok=True)
    raise SystemExit(f"all RecipeNLG mirrors failed: {last_err}")

def fetch_calories(force: bool = False) -> None:
    RAW_CALORIES.mkdir(parents=True, exist_ok=True)
    for name, url in FDC_FILES.items():
        target = RAW_CALORIES / name
        if target.is_dir() and any(target.glob("food.csv")) and not force:
            print(f"[skip] data/raw/calories/{name} exists")
            continue
        print(f"[calories] {name}")
        archive = RAW_CALORIES / f"{name}.zip"
        _download(url, archive)
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                rel = Path(member).parts[1:]  # strip the archive's top folder
                if not rel or member.endswith("/"):
                    continue
                out = target.joinpath(*rel)
                out.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, out.open("wb") as dst:
                    dst.write(src.read())
        archive.unlink()
        print(f"[ok] data/raw/calories/{name}")

def run_download_raw_data(recipes='none', calories=False, everything=False, force=False) -> None:

    if everything:
        calories, recipes = True, "sample"
    if calories:
        fetch_calories(force)
    if recipes == "sample":
        fetch_recipe_sample(force)
    elif recipes == "full":
        fetch_recipe_full(force)
    if not calories and recipes == "none":
        print("[notebook] tidak ada aksi: berikan parameter yang diinginkan")


In [ ]:
if RUN_DOWNLOAD_DATA:
    run_download_raw_data(everything=True)      # tabel USDA + sampel 10k resep
    # run_download_raw_data(recipes="full")     # RecipeNLG penuh (~2.2M resep)

**`scripts/build_calorie_reference.py`**

Build the calorie reference tables from USDA FoodData Central CSVs (stdlib only).  Input  : data/raw/calories/sr_legacy/     (download_raw_data.py --calories) Output : data/processed/calorie_reference.csv      one row per USDA food data/processed/unit_grams_reference.csv   (fdc_id, unit) -> grams  Every value is kcal per 100 g edible portion, taken from nutrient 1008 (Energy) with the Atwater factors (2047/2048) as fallback. FoodData Central is public domain (CC0); see data/README.md.


In [ ]:
RAW = ROOT / "data" / "raw" / "calories"
REFERENCE_OUT_DIR = ROOT / "data" / "processed"

# Same plural folding the pipelines use (stdlib-only module, keeps USDA keys aligned).

ENERGY_IDS = ("1008", "2047", "2048")  # Energy kcal, Atwater general, Atwater specific

# USDA `modifier` values -> unit tokens the ingredient parser emits.
UNIT_PATTERNS = [
    (r"^cup", "cup"),
    (r"^(tbsp|tablespoon)", "tbsp"),
    (r"^(tsp|teaspoon)", "tsp"),
    (r"^(fl oz|fluid ounce)", "fl_oz"),
    (r"^(oz|ounce)", "oz"),
    (r"^(lb|pound)", "lb"),
    (r"^(g|gram)", "g"),
    (r"^(ml|milliliter)", "ml"),
    (r"^slice", "slice"),
    (r"^clove", "clove"),
    (r"^(can|jar|package|pkg|stick|bar|bunch|head|stalk|piece|pinch|dash|drop|sprig)", None),
]

def canonical_unit(modifier: str) -> str | None:
    m = modifier.strip().lower()
    for pattern, unit in UNIT_PATTERNS:
        if re.match(pattern, m):
            return unit or m.split(",")[0].split()[0]
    return None

def base_ingredient(description: str) -> str:
    """'Butter, salted' -> 'butter'; 'Carrots, raw' -> 'carrot'."""
    head = description.split(",")[0]
    head = re.sub(r"\(.*?\)", " ", head)
    head = re.sub(r"[^a-z0-9 ]+", " ", head.lower())
    head = " ".join(head.split())
    if not head:
        return head
    tokens = head.split(" ")
    tokens[-1] = singularize(tokens[-1])
    return " ".join(tokens)

def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))

def load_energy() -> tuple[dict[str, tuple[float, str]], dict[str, dict]]:
    """fdc_id -> (kcal per 100 g, nutrient id used)."""
    rows: dict[str, list[tuple[int, float, str]]] = defaultdict(list)
    for source in sorted(RAW.iterdir()):
        if not (source / "food_nutrient.csv").exists():
            continue
        for row in read_csv(source / "food_nutrient.csv"):
            nid = row["nutrient_id"]
            if nid not in ENERGY_IDS or not row["amount"]:
                continue
            try:
                amount = float(row["amount"])
            except ValueError:
                continue
            rows[row["fdc_id"]].append((ENERGY_IDS.index(nid), amount, nid))
    best = {}
    for fdc_id, candidates in rows.items():
        candidates.sort()
        rank, amount, nid = candidates[0]
        best[fdc_id] = (amount, nid)
    return best

def load_foods() -> dict[str, dict]:
    foods = {}
    for source in sorted(RAW.iterdir()):
        if not (source / "food.csv").exists():
            continue
        for row in read_csv(source / "food.csv"):
            foods[row["fdc_id"]] = row
    return foods

def build_reference(foods: dict[str, dict], energy: dict) -> list[dict]:
    reference = []
    for fdc_id, row in foods.items():
        if fdc_id not in energy:
            continue
        kcal, nid = energy[fdc_id]
        reference.append(
            {
                "fdc_id": fdc_id,
                "description": row["description"].strip(),
                "base_ingredient": base_ingredient(row["description"]),
                "kcal_per_100g": round(kcal, 2),
                "energy_nutrient_id": nid,
                "data_type": row["data_type"],
                "food_category_id": row["food_category_id"],
            }
        )
    reference.sort(key=lambda r: (r["base_ingredient"], r["description"]))
    return reference

def build_unit_grams(reference: list[dict]) -> list[dict]:
    base_by_fdc = {row["fdc_id"]: row["base_ingredient"] for row in reference}
    buckets: dict[tuple[str, str, str], list[float]] = defaultdict(list)
    for source in sorted(RAW.iterdir()):
        path = source / "food_portion.csv"
        if not path.exists():
            continue
        for row in read_csv(path):
            ingredient = base_by_fdc.get(row["fdc_id"])
            unit = canonical_unit(row["modifier"] or row["portion_description"] or "")
            if not ingredient or not unit or not row["gram_weight"]:
                continue
            try:
                amount = float(row["amount"] or 1)
                grams = float(row["gram_weight"])
            except ValueError:
                continue
            if amount <= 0 or grams <= 0 or grams / amount > 5000:
                continue
            buckets[(row["fdc_id"], ingredient, unit)].append(grams / amount)
    out = []
    for (fdc_id, ingredient, unit), values in sorted(buckets.items(), key=lambda kv: (kv[0][1], kv[0][2], kv[0][0])):
        out.append(
            {
                "fdc_id": fdc_id,
                "ingredient": ingredient,
                "unit": unit,
                "grams_per_unit": round(statistics.median(values), 2),
                "n_portions": len(values),
                "source": "USDA FDC food_portion",
            }
        )
    return out

def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"[ok] {path.relative_to(ROOT)}: {len(rows)} rows")

def run_build_calorie_reference() -> None:
    if not RAW.exists() or not any(RAW.iterdir()):
        raise SystemExit("data/raw/calories is empty: run download_raw_data.py --calories")

    foods = load_foods()
    energy = load_energy()
    reference = build_reference(foods, energy)
    covered = len(reference) / max(len(foods), 1)
    print(f"[info] {len(foods)} USDA foods, {len(reference)} with an energy value ({covered:.1%})")

    write_csv(REFERENCE_OUT_DIR / "calorie_reference.csv", reference)
    write_csv(REFERENCE_OUT_DIR / "unit_grams_reference.csv", build_unit_grams(reference))


In [ ]:
if RUN_BUILD_REFERENCE:
    run_build_calorie_reference()

**`scripts/build_calorie_baseline.py`**

Build the curated ingredient -> kcal/100 g baseline used before fuzzy matching.  `calorie_reference.csv` is the raw USDA search space (7,793 foods, many variants per ingredient). This script pins one deliberate USDA description per canonical ingredient so the calorie mapper has a deterministic first hit; anything not listed here falls through to exact -> normalized -> fuzzy -> supplement.  Input  : data/processed/calorie_reference.csv Output : data/processed/calorie_baseline.csv


In [ ]:
REFERENCE = ROOT / "data" / "processed" / "calorie_reference.csv"
BASELINE_OUT = ROOT / "data" / "processed" / "calorie_baseline.csv"

# canonical ingredient -> tokens that must all appear in the USDA description.
# The first non-processed, shortest match wins, so keep tokens discriminating.
MAP: dict[str, tuple[str, ...]] = {
    # pantry / baking
    "salt": ("salt, table",),
    "sugar": ("sugars, granulated",),
    "brown sugar": ("sugars, brown",),
    "powdered sugar": ("sugars, powdered",),
    "flour": ("wheat flour", "all-purpose"),
    "whole wheat flour": ("wheat flour", "whole-grain"),
    "cornstarch": ("cornstarch",),
    "baking powder": ("leavening agents, baking powder", "double-acting"),
    "baking soda": ("leavening agents, baking soda",),
    "yeast": ("yeast, baker's",),
    "cornmeal": ("cornmeal, whole-grain, yellow",),
    "oat": ("cereals, oats", "regular and quick", "dry"),
    "bread crumb": ("bread, crumbs", "plain"),
    "cracker crumb": ("crackers, saltines",),
    "gelatin": ("gelatins, dry powder",),
    "cocoa": ("cocoa, dry powder, unsweetened",),
    "chocolate chip": ("candies, semisweet chocolate",),
    "vanilla": ("vanilla extract",),
    "honey": ("honey",),
    "molasses": ("molasses",),
    "maple syrup": ("syrup, maple",),
    "peanut butter": ("peanut butter, smooth",),
    # fats and dairy
    "butter": ("butter, salted",),
    "margarine": ("margarine, regular",),
    "shortening": ("shortening, household",),
    "oil": ("oil, soybean, salad or cooking",),
    "olive oil": ("oil, olive",),
    "vegetable oil": ("oil, soybean, salad or cooking",),
    "milk": ("milk, whole, 3.25%",),
    "evaporated milk": ("milk, evaporated, 2%",),
    "condensed milk": ("milk, canned, condensed, sweetened",),
    "buttermilk": ("milk, buttermilk, fluid, cultured, lowfat",),
    "cream": ("cream, fluid, heavy whipping",),
    "heavy cream": ("cream, fluid, heavy whipping",),
    "half and half": ("cream, fluid, half and half",),
    "sour cream": ("cream, sour, cultured",),
    "yogurt": ("yogurt, plain, whole milk",),
    "cream cheese": ("cheese, cream",),
    "cheddar cheese": ("cheese, cheddar",),
    "parmesan cheese": ("cheese, parmesan, hard",),
    "mozzarella cheese": ("cheese, mozzarella, whole milk",),
    "swiss cheese": ("cheese, swiss",),
    "ricotta cheese": ("cheese, ricotta, whole milk",),
    "cottage cheese": ("cheese, cottage, creamed", "curd"),
    "egg": ("egg, whole, raw, fresh",),
    "egg white": ("egg, white, raw, fresh",),
    "egg yolk": ("egg, yolk, raw, fresh",),
    # vegetables
    "onion": ("onions, raw",),
    "green onion": ("onions, spring or scallions",),
    "garlic": ("garlic, raw",),
    "potato": ("potatoes, flesh and skin, raw",),
    "sweet potato": ("sweet potato, raw",),
    "carrot": ("carrots, raw",),
    "celery": ("celery, raw",),
    "tomato": ("tomatoes, red, ripe, raw",),
    "tomato paste": ("tomato products, canned, paste",),
    "tomato sauce": ("tomato products, canned, sauce",),
    "green pepper": ("peppers, sweet, green, raw",),
    "red pepper": ("peppers, sweet, red, raw",),
    "bell pepper": ("peppers, sweet, green, raw",),
    "jalapeno": ("peppers, jalapeno, raw",),
    "mushroom": ("mushrooms, white, raw",),
    "spinach": ("spinach, raw",),
    "lettuce": ("lettuce, iceberg",),
    "cabbage": ("cabbage, raw",),
    "broccoli": ("broccoli, raw",),
    "cauliflower": ("cauliflower, raw",),
    "zucchini": ("squash, summer, zucchini", "raw"),
    "eggplant": ("eggplant, raw",),
    "arugula": ("arugula, raw",),
    "asparagus": ("asparagus, raw",),
    "lime": ("limes, raw",),
    "shallot": ("shallots, raw",),
    "mint": ("peppermint, fresh",),
    "caper": ("capers, canned",),
    "tarragon": ("spices, tarragon, dried",),
    "coffee": ("beverages, coffee, brewed, breakfast blend",),
    "mango": ("mangos, raw",),
    "raspberry": ("raspberries, raw",),
    "butternut squash": ("squash, winter, butternut, raw",),
    "summer squash": ("squash, summer, all varieties, raw",),
    "rhubarb": ("rhubarb, raw",),
    "hazelnut": ("nuts, hazelnuts or filberts",),
    "beet": ("beets, raw",),
    "pumpkin puree": ("pumpkin, canned",),
    "bean sprout": ("mung beans, mature seeds, sprouted, raw",),
    "couscous": ("couscous, dry",),
    "radish": ("radishes, raw",),
    "pork tenderloin": ("pork, fresh, loin, tenderloin", "cooked, broiled"),
    "beer": ("alcoholic beverage, beer, regular, all",),
    "sake": ("alcoholic beverage, rice (sake)",),
    "whiskey": ("alcoholic beverage, distilled, all", "80 proof"),
    "pretzel": ("snacks, pretzels, hard, plain",),
    "cucumber": ("cucumber, with peel, raw",),
    "green bean": ("beans, snap, green, raw",),
    "pea": ("peas, green, raw",),
    "corn": ("corn, sweet, yellow, raw",),
    "pumpkin": ("pumpkin, raw",),
    "avocado": ("avocados, raw, all commercial varieties",),
    "olive": ("olives, ripe, canned",),
    "ginger": ("ginger root, raw",),
    "pineapple": ("pineapple, raw, all varieties",),
    "peach": ("peaches, yellow, raw",),
    "cherry": ("cherries, sweet, raw",),
    "date": ("dates, medjool",),
    "tomato juice": ("tomato juice, canned",),
    "horseradish": ("horseradish, prepared",),
    "sauerkraut": ("sauerkraut, canned",),
    "applesauce": ("applesauce, canned, unsweetened",),
    "apple juice": ("apple juice, canned or bottled, unsweetened",),
    "potato chip": ("snacks, potato chips, plain, salted",),
    "mixed vegetable": ("vegetables, mixed, canned",),
    "cranberry sauce": ("cranberry sauce, canned, sweetened",),
    "red wine": ("alcoholic beverage, wine, table, red",),
    "chocolate cake": ("cake, chocolate, prepared from recipe without frosting",),
    "apple": ("apples, raw, with skin",),
    "banana": ("bananas, raw",),
    "orange": ("oranges, raw, all commercial varieties",),
    "lemon": ("lemons, raw, without peel",),
    "lemon juice": ("lemon juice, raw",),
    "lime juice": ("lime juice, raw",),
    "strawberry": ("strawberries, raw",),
    "blueberry": ("blueberries, raw",),
    "raisin": ("raisins, dark, seedless",),
    "coconut": ("nuts, coconut meat, raw",),
    # proteins
    "chicken": ("chicken, broilers or fryers, meat and skin, raw",),
    "chicken breast": ("chicken, broilers or fryers, breast", "raw"),
    "chicken thigh": ("chicken, broilers or fryers, thigh", "raw"),
    "ground beef": ("beef, ground, 80% lean meat / 20% fat", "raw"),
    "beef": ("beef, chuck, arm pot roast", "raw"),
    "pork": ("pork, fresh, loin, blade", "raw"),
    "bacon": ("pork, cured, bacon, unprepared",),
    "ham": ("ham, sliced, regular",),
    "sausage": ("pork sausage, link/patty, unprepared",),
    "turkey": ("turkey, whole, meat and skin, raw",),
    "shrimp": ("crustaceans, shrimp, raw",),
    "crab": ("crustaceans, crab, blue, raw",),
    "tuna": ("fish, tuna, light, canned in water", "drained"),
    "salmon": ("fish, salmon, atlantic, farmed, raw",),
    "cod": ("fish, cod, atlantic, raw",),
    "tilapia": ("fish, tilapia, raw",),
    "anchovy": ("fish, anchovy, european, raw",),
    "sardine": ("fish, sardine, atlantic, canned in oil",),
    "tofu": ("tofu, raw, firm",),
    "lentil": ("lentils, raw",),
    "chickpea": ("chickpeas", "mature seeds, raw"),
    "kidney bean": ("beans, kidney, all types", "raw"),
    "black bean": ("beans, black, mature seeds, raw",),
    "pinto bean": ("beans, pinto, mature seeds, raw",),
    # grains and pasta
    "rice": ("rice, white, long-grain, regular, raw",),
    "brown rice": ("rice, brown, long-grain, raw",),
    "pasta": ("pasta, dry, enriched",),
    "spaghetti": ("pasta, dry, enriched",),
    "macaroni": ("pasta, dry, enriched",),
    "noodle": ("noodles, egg, dry, enriched",),
    "quinoa": ("quinoa, uncooked",),
    "bread": ("bread, white, commercially prepared",),
    "tortilla": ("tortillas, ready-to-bake or -fry, flour",),
    "oatmeal": ("cereals, oats, regular and quick", "dry"),
    # nuts and seeds
    "almond": ("nuts, almonds",),
    "walnut": ("nuts, walnuts, english",),
    "pecan": ("nuts, pecans",),
    "cashew": ("nuts, cashew nuts, raw",),
    "peanut": ("peanuts, all types, raw",),
    "pistachio": ("nuts, pistachio nuts, raw",),
    "sunflower seed": ("seeds, sunflower seed kernels, dried",),
    "pumpkin seed": ("seeds, pumpkin and squash seed kernels, dried",),
    "sesame seed": ("seeds, sesame seeds, whole, dried",),
    "poppy seed": ("spices, poppy seed",),
    "flaxseed": ("seeds, flaxseed",),
    "chia seed": ("seeds, chia seeds, dried",),
    # condiments and seasoning
    "pepper": ("spices, pepper, black",),
    "salt and pepper": ("spices, pepper, black",),
    "cinnamon": ("spices, cinnamon, ground",),
    "nutmeg": ("spices, nutmeg, ground",),
    "paprika": ("spices, paprika",),
    "cumin": ("spices, cumin seed",),
    "oregano": ("spices, oregano, dried",),
    "thyme": ("spices, thyme, dried",),
    "rosemary": ("spices, rosemary, dried",),
    "basil": ("spices, basil, dried",),
    "parsley": ("parsley, fresh",),
    "cilantro": ("coriander (cilantro) leaves, raw",),
    "bay leaf": ("spices, bay leaf",),
    "chili powder": ("spices, chili powder",),
    "cayenne": ("spices, pepper, red or cayenne",),
    "garlic powder": ("spices, garlic powder",),
    "onion powder": ("spices, onion powder",),
    "mustard": ("mustard, prepared, yellow",),
    "mayonnaise": ("mayonnaise, regular",),
    "ketchup": ("catsup",),
    "vinegar": ("vinegar, distilled",),
    "soy sauce": ("soy sauce made from soy and wheat", "shoyu"),
    "worcestershire sauce": ("sauce, worcestershire",),
    "hot sauce": ("sauce, ready-to-serve, pepper or hot",),
    "salsa": ("sauce, salsa, ready-to-serve",),
    "chicken broth": ("soup, chicken broth, ready-to-serve",),
    "beef broth": ("soup, beef broth or bouillon canned, ready-to-serve",),
    "water": ("water, tap",),
}

EXCLUDE = (
    "meatless", "imitation", "substitute", "low-sodium", "low sodium", "low fat",
    "fat free", "reduced", "toasted", "sharp", "babyfood", "fast food", "restaurant",
)

PROCESSED_WORDS = (
    "dehydrated", "dried", "canned", "frozen", "cooked", "powder", "syrup",
    "sweetened", "juice", "concentrate", "fried", "babyfood", "restaurant",
    "fast food", "candies", "cereal", "snack", "mix", "prepared", "pasteurized",
)

def load_reference() -> list[dict]:
    with REFERENCE.open(encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))

def pick(reference: list[dict], tokens: tuple[str, ...]) -> dict | None:
    hits = [
        row
        for row in reference
        if all(token in row["description"].lower() for token in tokens)
        and not any(word in row["description"].lower() for word in EXCLUDE)
    ]
    if not hits:
        return None
    hits.sort(key=lambda r: (any(w in r["description"].lower() for w in PROCESSED_WORDS), len(r["description"])))
    return hits[0]

def run_build_calorie_baseline() -> None:
    if not REFERENCE.exists():
        raise SystemExit("missing data/processed/calorie_reference.csv: run build_calorie_reference.py")
    reference = load_reference()
    rows, unresolved = [], []
    for ingredient, tokens in MAP.items():
        match = pick(reference, tokens)
        if match is None:
            unresolved.append((ingredient, tokens))
            continue
        rows.append(
            {
                "ingredient": ingredient,
                "kcal_per_100g": match["kcal_per_100g"],
                "matched_description": match["description"],
                "fdc_id": match["fdc_id"],
                "source": "USDA FDC SR Legacy",
            }
        )
    rows.sort(key=lambda r: r["ingredient"])
    with BASELINE_OUT.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"[ok] {BASELINE_OUT.relative_to(ROOT)}: {len(rows)} ingredients")
    for ingredient, tokens in unresolved:
        print(f"[warn] no USDA match for {ingredient!r} ({' '.join(tokens)})", file=sys.stderr)


In [ ]:
if RUN_BUILD_BASELINE:
    run_build_calorie_baseline()

**`scripts/build_synthetic_queries.py`**

Generate the Pipeline 2 evaluation set from the recipe corpus (PRD section 32).  For each sampled recipe, a random subset of its canonical ingredients becomes a simulated pantry. The source recipe is the relevance target: retrieval succeeds when it appears in the candidate list.  python scripts/build_synthetic_queries.py python scripts/build_synthetic_queries.py --corpus data/raw/recipenlg/RecipeNLG.csv --queries 500  Deterministic for a fixed --seed and corpus. Output: data/test/synthetic_retrieval_queries.json


In [ ]:
ROOT = PROCESSED.parents[1]
SAMPLE = ROOT / "data" / "samples" / "recipes_sample.csv"
QUERIES_OUT = ROOT / "data" / "test" / "synthetic_retrieval_queries.json"

def canonical_ingredients(ner_field: str, resolver: Resolver) -> list[str]:
    try:
        terms = ast.literal_eval(ner_field)
    except (ValueError, SyntaxError):
        return []
    seen: dict[str, None] = {}
    for term in terms:
        canonical = resolver.resolve(term)
        if canonical:
            seen.setdefault(canonical, None)
    return list(seen)

def run_build_synthetic_queries(corpus=SAMPLE, queries=200, pantry_fraction=0.6, seed=42, min_ingredients=4) -> None:

    corpus = corpus.resolve()
    if not corpus.exists():
        raise SystemExit(f"corpus not found: {corpus}\nRun scripts/download_raw_data.py --recipes sample")
    resolver = Resolver()
    rng = random.Random(seed)

    # Reservoir sample so the full 2.2M-recipe corpus needs O(--queries) memory.
    chosen: list[tuple[int, str, list[str]]] = []
    eligible = 0
    with corpus.open(encoding="utf-8", newline="") as fh:
        for index, row in enumerate(csv.DictReader(fh)):
            ingredients = canonical_ingredients(row["NER"], resolver)
            if len(ingredients) < min_ingredients:
                continue
            candidate = (index, row["title"], ingredients)
            if len(chosen) < queries:
                chosen.append(candidate)
            else:
                slot = rng.randrange(eligible + 1)
                if slot < queries:
                    chosen[slot] = candidate
            eligible += 1

    if not chosen:
        raise SystemExit("no recipes with enough resolvable ingredients")
    chosen.sort(key=lambda c: c[0])
    print(f"[info] {eligible} eligible recipes, {len(chosen)} sampled")

    queries = []
    for index, title, ingredients in chosen:
        count = max(3, round(len(ingredients) * pantry_fraction))
        pantry = rng.sample(ingredients, min(count, len(ingredients) - 1))
        held_out = [i for i in ingredients if i not in pantry]
        queries.append(
            {
                "id": f"syn{len(queries) + 1:04d}",
                "recipe_id": index,
                "title": title,
                "query": "I have " + ", ".join(pantry[:-1]) + (" and " if len(pantry) > 1 else "") + pantry[-1] + ".",
                "pantry": pantry,
                "held_out": held_out,
                "required": ingredients,
            }
        )

    payload = {
        "version": "1.0.0",
        "description": (
            "Synthetic retrieval queries for Pipeline 2 evaluation. recipe_id is the 0-based row "
            "index in the source corpus; required is the full canonical ingredient set of that recipe."
        ),
        "generator": {
            "corpus": str(corpus.relative_to(ROOT)),
            "queries": len(queries),
            "pantry_fraction": pantry_fraction,
            "seed": seed,
            "min_ingredients": min_ingredients,
        },
        "queries": queries,
    }
    QUERIES_OUT.parent.mkdir(parents=True, exist_ok=True)
    QUERIES_OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"[ok] {QUERIES_OUT.relative_to(ROOT)}: {len(queries)} queries from {corpus.name}")


In [ ]:
if RUN_BUILD_QUERIES:
    run_build_synthetic_queries()

**`scripts/check_ingredient_coverage.py`**

Validate the curated ingredient tables and report corpus coverage.  Checks (exit code 1 on failure): * every alias canonical exists in the calorie baseline, supplement, or zero list * every pantry-staple ingredient has a calorie treatment * no duplicate aliases after normalization * every annotated-query label resolves to the ingredient vocabulary  Prints how many RecipeNLG `NER` mentions resolve to a calorie source and the most frequent unresolved terms (the next alias/supplement candidates).  python scripts/check_ingredient_coverage.py python scripts/check_ingredient_coverage.py --corpus data/raw/recipenlg/RecipeNLG.csv


In [ ]:
COVERAGE_SAMPLE = PROCESSED.parent / "samples" / "recipes_sample.csv"

def run_check_ingredient_coverage(corpus=COVERAGE_SAMPLE, top=25) -> None:

    resolver = Resolver()
    failures: list[str] = []

    alias_rows = read_rows(PROCESSED / "ingredient_aliases.csv")
    for alias, count in Counter(normalize(r["alias"]) for r in alias_rows).items():
        if count > 1:
            failures.append(f"duplicate alias: {alias}")
    for row in alias_rows:
        if normalize(row["canonical"]) not in resolver.known:
            failures.append(f"alias {row['alias']!r} -> unknown canonical {row['canonical']!r}")
    for row in read_rows(PROCESSED / "pantry_staples.csv"):
        if resolver.resolve(row["ingredient"]) is None:
            failures.append(f"pantry staple {row['ingredient']!r} has no calorie treatment")

    annotations = json.loads((PROCESSED.parent / "test" / "annotated_queries.json").read_text(encoding="utf-8"))
    for query in annotations["queries"]:
        for field in ("available_ingredients", "excluded_ingredients"):
            for ingredient in query["expected"][field]:
                if resolver.resolve(ingredient) is None:
                    failures.append(f"{query['id']}: labeled {field} {ingredient!r} is outside the vocabulary")

    unresolved: Counter[str] = Counter()
    total = hit = 0
    if corpus.exists():
        with corpus.open(encoding="utf-8", newline="") as fh:
            for row in csv.DictReader(fh):
                try:
                    terms = ast.literal_eval(row["NER"])
                except (ValueError, SyntaxError):
                    continue
                for term in terms:
                    total += 1
                    if resolver.resolve(term):
                        hit += 1
                    else:
                        unresolved[term.lower().strip()] += 1
    else:
        print(f"[warn] corpus not found: {corpus}", file=sys.stderr)

    print(
        f"baseline={len(resolver.baseline)} supplement={len(resolver.supplement)} "
        f"zero={len(resolver.zero)} aliases={len(resolver.aliases)}"
    )
    if total:
        print(f"corpus coverage: {hit}/{total} ingredient mentions resolved ({hit / total:.1%})")
        print(f"distinct unresolved terms: {len(unresolved)}")
        for term, count in unresolved.most_common(top):
            print(f"  {count:>5}  {term}")

    if failures:
        print("\nFAIL", file=sys.stderr)
        for line in failures:
            print(f"  - {line}", file=sys.stderr)
        raise SystemExit(1)
    print("\nOK: curated tables are internally consistent")


In [ ]:
if RUN_CHECK_COVERAGE:
    run_check_ingredient_coverage()             # exit non-zero jika tabel inkonsisten

**`scripts/generate_ner_training_data.py`**

Generate synthetic pantry queries with ingredient spans for NER training.  Recipes from the RecipeNLG sample (``NER`` field) supply real ingredient names; templates turn them into natural-language queries with calorie/exclusion clauses. The matching input is the *masked* context Pipeline 1 actually feeds to its matcher (calorie and exclusion spans blanked by the same production functions), so train-time and inference-time text distributions match.  Output: JSONL rows ``{"text", "spans", "canonical", "query"}`` where ``spans`` are character offsets of ingredient words inside the masked ``text``. Labels are aligned to model subtokens at training time.  Usage: python scripts/generate_ner_training_data.py --count 15000 --seed 42


In [ ]:
NER_DATA_OUT_DIR = ROOT / "artifacts" / "ner_training"

CLAUSE_TEMPLATES = [
    "I have {ings}.",
    "I've got {ings}.",
    "I have {ings} in my pantry.",
    "I have {ings} on hand.",
    "i have {ings} at home.",
    "Currently I have {ings}.",
]
CONSTRAINT_TEMPLATES = [
    "",
    "Around {n} calories.",
    "around {n} calories",
    "About {n} kcal.",
    "~{n} kcal",
    "Under {n} calories.",
    "under {n} calories",
    "Max {n} kcal.",
    "less than {n} calories",
    "Between {a} and {b} calories.",
    "At least {a} calories.",
    "No {x}.",
    "no {x}",
    "Without {x}.",
    "No {x} or {y}.",
    "without {x} and {y}",
    "Around {n} calories, no {x}.",
    "Under {n} calories without {x}.",
]

def _join_names(names: list[str], rng: random.Random) -> str:
    if len(names) == 1:
        return names[0]
    if len(names) == 2:
        return f"{names[0]} {rng.choice(['and', 'plus', 'with'])} {names[1]}"
    return ", ".join(names[:-1]) + f" {rng.choice(['and', 'plus'])} {names[-1]}"

def _maybe_pluralize(name: str, rng: random.Random) -> str:
    if rng.random() < 0.30 and not name.endswith("s"):
        head, _, rest = name.rpartition(" ")
        head = head + "s" if not rest else head
        return f"{head} {rest}".strip() if rest else head
    return name

def _maybe_typo(word: str, rng: random.Random) -> str:
    if len(word) >= 4 and rng.random() < 0.10:
        cut = rng.randrange(1, len(word) - 1)
        return word[:cut] + word[cut + 1] + word[cut] + word[cut + 2 :]
    return word

def _surface(name: str, rng: random.Random) -> str:
    """Render a canonical ingredient as a plausible query surface form."""
    words = [_maybe_typo(w, rng) for w in _maybe_pluralize(name, rng).split()]
    return " ".join(words)

def _canonical_pool(vocab: Vocabulary, recipes: list[list[str]]) -> list[str]:
    seen: dict[str, None] = {}
    for names in recipes:
        for name in names:
            canonical = vocab.canonical(normalize_ingredient(name))
            if canonical:
                seen.setdefault(canonical, None)
    return sorted(seen)

def _word_spans(clause: str, names: list[str]) -> list[tuple[int, int]]:
    """Char spans of every expected ingredient word inside the normalized clause."""
    expected = [word for name in names for word in name.split()]
    spans: list[tuple[int, int]] = []
    cursor = 0
    for word in expected:
        match = re.compile(rf"(?<![a-z0-9]){re.escape(word)}(?![a-z0-9])").search(clause, cursor)
        if match is None:  # typo augmentation changed the surface; search all remaining words
            match = re.compile(rf"(?<![a-z0-9]){re.escape(word)}(?![a-z0-9])").search(clause)
            if match is None:
                raise ValueError(f"ingredient word {word!r} not found in clause {clause!r}")
        spans.append((match.start(), match.end()))
        cursor = match.end()
    return spans

def _ingredient_spans(clause: str, names: list[str]) -> list[list[int]]:
    """One char span per ingredient PHRASE (multi-word names stay one entity).

    Per-ingredient grouping is essential: after normalization, consecutive
    different ingredients are also separated by a single space, so a gap-based
    merge would fuse ``chicken potato`` into one entity and corrupt BIO labels.
    """
    word_spans = _word_spans(clause, names)
    phrases: list[list[int]] = []
    cursor = 0
    for name in names:
        width = len(name.split())
        first, last = word_spans[cursor], word_spans[cursor + width - 1]
        phrases.append([first[0], last[1]])
        cursor += width
    return phrases

def generate_one(names: list[str], excluded_pool: list[str], rng: random.Random, vocab: Vocabulary) -> dict:
    surfaces = [_surface(name, rng) for name in names]
    clause = rng.choice(CLAUSE_TEMPLATES).format(ings=_join_names(surfaces, rng))

    excluded = [n for n in rng.sample(excluded_pool, k=min(2, len(excluded_pool))) if n not in names]
    template = rng.choice(CONSTRAINT_TEMPLATES)
    constraint = template.format(
        n=rng.choice(range(200, 800, 25)),
        a=rng.choice(range(250, 500, 50)),
        b=rng.choice(range(500, 800, 50)),
        x=excluded[0] if excluded else "cheese",
        y=excluded[1] if len(excluded) > 1 else "milk",
    )
    query = f"{clause} {constraint}".strip()

    # Production masking, verbatim from query_parser.parse_query.
    text = normalize_text(query)
    _, calorie_spans = extract_calorie_constraints(text)
    masked = _blank(text, calorie_spans)
    spans = exclusion_spans(masked)
    available_context = _blank(masked, spans)

    clause_text = normalize_text(clause)
    canonical = [vocab.canonical(normalize_ingredient(s)) or s for s in surfaces]
    return {
        "text": available_context,
        "spans": _ingredient_spans(clause_text, surfaces),
        "canonical": canonical,
        "query": query,
    }

def run_generate_ner_training_data(count=15000, dev_rows=500, seed=42, corpus=SAMPLE_CSV, output_dir=NER_DATA_OUT_DIR) -> None:

    rng = random.Random(seed)
    vocab = load_vocabulary()

    recipes: list[list[str]] = []
    with corpus.open(encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            names = ast.literal_eval(row["NER"])
            if isinstance(names, list):
                recipes.append([str(n).strip().lower() for n in names if str(n).strip()])

    pool = _canonical_pool(vocab, recipes)
    print(f"canonical ingredient pool: {len(pool)} | recipes: {len(recipes)}")

    rows: list[dict] = []
    attempts = 0
    while len(rows) < count + dev_rows and attempts < (count + dev_rows) * 3:
        attempts += 1
        recipe = rng.choice(recipes)
        usable = [n for n in recipe if vocab.canonical(normalize_ingredient(n))]
        if len(usable) < 2:
            continue
        k = rng.randint(2, min(6, len(usable)))
        names = rng.sample(usable, k=k)
        try:
            rows.append(generate_one(names, pool, rng, vocab))
        except ValueError:
            continue

    rng.shuffle(rows)
    dev, train = rows[: dev_rows], rows[dev_rows :]

    output_dir.mkdir(parents=True, exist_ok=True)
    for split, split_rows in (("train", train), ("dev", dev)):
        out_path = output_dir / f"{split}.jsonl"
        with out_path.open("w", encoding="utf-8") as fh:
            for row in split_rows:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"{out_path}: {len(split_rows)} rows")

    sample = train[0]
    print("sample text:", sample["text"])
    print("sample spans:", sample["spans"], "->", [sample["text"][s:e] for s, e in sample["spans"]])


In [ ]:
if RUN_GENERATE_NER_DATA:
    run_generate_ner_training_data()            # -> artifacts/ner_training/{train,dev}.jsonl

**`scripts/train_ner.py`**

Fine-tune DistilBERT for pantry-ingredient token classification (Pipeline 1 NER approach).  Trains on the synthetic masked-context queries produced by ``scripts/generate_ner_training_data.py`` (labels: O / B-ING / I-ING). Best checkpoint by dev span-F1 is saved to ``artifacts/ner_model/`` for the ``NerApproach`` registry entry and the Streamlit UI.  Defaults target Apple-silicon MPS; falls back to CPU automatically.  Usage: python scripts/train_ner.py                       # full run (15k rows, 3 epochs) python scripts/train_ner.py --max-samples 200     # smoke run


In [ ]:
DEFAULT_TRAIN = REPO_ROOT / "artifacts" / "ner_training" / "train.jsonl"
DEFAULT_DEV = REPO_ROOT / "artifacts" / "ner_training" / "dev.jsonl"
DEFAULT_OUTPUT = REPO_ROOT / "artifacts" / "ner_model"

LABELS = ["O", "B-ING", "I-ING"]

def _align_labels(text: str, spans: list[list[int]], offsets: list[tuple[int, int]]) -> list[int]:
    """BIO labels per subtoken; -100 for special tokens.

    ``spans`` from the generator are per-ingredient PHRASE spans (multi-word
    names arrive as one span), so no gap-based merging happens here: merging on
    single-space gaps would fuse consecutive distinct ingredients after text
    normalization and corrupt the labels.
    """
    labels: list[int] = []
    for start, end in offsets:
        if end <= start:
            labels.append(-100)
            continue
        entity = next((m for m in spans if start < m[1] and end > m[0]), None)
        if entity is None:
            labels.append(0)
        elif start == entity[0]:
            labels.append(1)  # B-ING
        else:
            labels.append(2)  # I-ING
    return labels

def _load_rows(path: Path, limit: int | None) -> list[dict]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if limit:
        rows = rows[:limit]
    return rows

def _encode(rows: list[dict], tokenizer) -> list[dict[str, list[int]]]:
    encoded = []
    for row in rows:
        item = tokenizer(
            row["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            return_offsets_mapping=True,
        )
        labels = _align_labels(row["text"], row["spans"], item["offset_mapping"])
        encoded.append(
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
                "labels": labels,
                "spans": row["spans"],
            }
        )
    return encoded

def _collate(batch: list[dict[str, list[int]]], tokenizer):
    model_inputs = tokenizer.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    width = model_inputs["input_ids"].shape[1]
    labels = np.full((len(batch), width), -100, dtype=np.int64)
    for i, example in enumerate(batch):
        labels[i, : len(example["labels"])] = example["labels"]
    model_inputs["labels"] = __import__("torch").from_numpy(labels)
    spans = [example["spans"] for example in batch]
    return model_inputs, spans

def _decode_spans(label_ids: list[int], offsets: list[tuple[int, int]]) -> list[list[int]]:
    """BIO decode over real (non-special) subtoken offsets -> char spans."""
    spans: list[list[int]] = []
    open_start: int | None = None
    open_end = 0
    for label, (start, end) in zip(label_ids, offsets):
        if end <= start:
            continue
        if label == 1:  # B-ING
            if open_start is not None:
                spans.append([open_start, open_end])
            open_start, open_end = start, end
        elif label == 2:  # I-ING
            if open_start is None:
                open_start, open_end = start, end
            else:
                open_end = end
        else:
            if open_start is not None:
                spans.append([open_start, open_end])
                open_start = None
    if open_start is not None:
        spans.append([open_start, open_end])
    return spans

def evaluate(model, rows: list[dict[str, list[int]]], tokenizer, device, batch_size: int) -> dict[str, float]:
    import torch

    model.eval()
    token_tp = token_fp = token_fn = 0
    span_tp = span_fp = span_fn = 0
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            batch, _ = _collate(rows[i : i + batch_size], tokenizer)
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = logits.argmax(dim=-1).cpu()

            for j, example in enumerate(rows[i : i + batch_size]):
                offsets = example["offsets"]
                pred_ids = preds[j][: len(offsets)].tolist()
                pred_labels = [p for p, off in zip(pred_ids, offsets) if off[1] > off[0]]
                pred_spans = _decode_spans(
                    pred_labels,
                    [off for off in offsets if off[1] > off[0]],
                )
                gold_labels = [l for l, off in zip(example["labels"], offsets) if off[1] > off[0] and l != -100]
                gold_spans = example["spans"]

                token_tp += sum(1 for p, g in zip(pred_labels, gold_labels) if p != 0 and g != 0 and p == g)
                token_fp += sum(1 for p, g in zip(pred_labels, gold_labels) if p != 0 and g == 0)
                token_fn += sum(1 for p, g in zip(pred_labels, gold_labels) if p == 0 and g != 0)

                pred_set = {tuple(s) for s in pred_spans}
                gold_set = {tuple(s) for s in gold_spans}
                span_tp += len(pred_set & gold_set)
                span_fp += len(pred_set - gold_set)
                span_fn += len(gold_set - pred_set)

    def _f1(tp: int, fp: int, fn: int) -> float:
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "token_f1": _f1(token_tp, token_fp, token_fn),
        "span_f1": _f1(span_tp, span_fp, span_fn),
    }

def run_train_ner(train=DEFAULT_TRAIN, dev=DEFAULT_DEV, output_dir=DEFAULT_OUTPUT, model_name='distilbert-base-uncased', epochs=3, batch_size=32, lr=5e-05, max_samples=None, seed=42) -> None:

    import torch
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = _pick_device(torch)
    print(f"device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    id2label = {0: "O", 1: "B-ING", 2: "I-ING"}
    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
    ).to(device)

    train_rows = _load_rows(train, max_samples)
    dev_rows = _load_rows(dev, None)
    print(f"train rows: {len(train_rows)} | dev rows: {len(dev_rows)}")

    # Keep per-example offsets so evaluation can decode predicted spans.
    train_encoded = _encode(train_rows, tokenizer)
    dev_encoded = _encode(dev_rows, tokenizer)
    for example, row in zip(dev_encoded, dev_rows):
        enc = tokenizer(row["text"], truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True)
        example["offsets"] = enc["offset_mapping"]

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = (len(train_encoded) // batch_size + 1) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_span_f1 = -1.0
    order = list(range(len(train_encoded)))
    for epoch in range(epochs):
        random.Random(seed + epoch).shuffle(order)
        model.train()
        running = 0.0
        for step, start in enumerate(range(0, len(order), batch_size)):
            batch_idx = order[start : start + batch_size]
            batch, _ = _collate([train_encoded[i] for i in batch_idx], tokenizer)
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running += loss.item()
            if step % 50 == 0:
                print(f"epoch {epoch + 1}/{epochs} step {step}: loss={loss.item():.4f}", flush=True)
        metrics = evaluate(model, dev_encoded, tokenizer, device, batch_size)
        print(
            f"epoch {epoch + 1} done | train_loss={running / (len(order) / batch_size + 1):.4f} "
            f"| dev token_f1={metrics['token_f1']:.4f} span_f1={metrics['span_f1']:.4f}",
            flush=True,
        )
        if metrics["span_f1"] > best_span_f1:
            best_span_f1 = metrics["span_f1"]
            output_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            (output_dir / "eval_report.json").write_text(
                json.dumps({"epoch": epoch + 1, **metrics}, indent=2), encoding="utf-8"
            )
            print(f"  saved new best (span_f1={best_span_f1:.4f}) to {output_dir}")

    print(f"done; best dev span_f1={best_span_f1:.4f}")


In [ ]:
if RUN_TRAIN_NER:
    run_train_ner()                             # -> artifacts/ner_model/ (best by dev span_f1)

**`scripts/simulate_pipeline2.py`**


In [ ]:
DUMMY_QUERIES = [
    {
        "available_ingredients": ["chicken", "potato", "carrot", "onion", "garlic"],
        "excluded_ingredients": ["cheese"],
        "calorie_target": 500,
        "calorie_min": None,
        "calorie_max": None,
    },
    {
        "available_ingredients": ["egg", "milk", "butter", "flour", "sugar"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": 600,
    },
    {
        "available_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
        "excluded_ingredients": ["milk"],
        "calorie_target": None,
        "calorie_min": 400,
        "calorie_max": 700,
    },
]

def run_simulate_pipeline2() -> None:
    retriever = CandidateRetriever.from_artifacts()
    results = []
    for query in DUMMY_QUERIES:
        candidates = retriever.retrieve(query, top_n=5)
        results.append(
            {
                "dummy_structured_query": query,
                "top_candidates": [
                    {
                        "recipe_id": c["recipe_id"],
                        "title": c["title"],
                        "tfidf_score": c["tfidf_score"],
                        "total_calories": c["total_calories"],
                        "canonical_ingredients": c["canonical_ingredients"],
                    }
                    for c in candidates
                ],
            }
        )
    print(json.dumps(results, indent=2))


In [ ]:
if RUN_SIMULATE:
    run_simulate_pipeline2()

**`scripts/encode_dense_corpus.py`**

Encode the processed recipe corpus with a sentence-transformer (Pipeline 2 dense approach).  Reads ``data/processed/recipes.parquet`` (same corpus as the TF-IDF approach), encodes each recipe's canonical ingredient text, and stores L2-normalized vectors plus a metadata sidecar under ``artifacts/``.  Usage: python scripts/encode_dense_corpus.py                       # default model + corpus python scripts/encode_dense_corpus.py --model all-MiniLM-L6-v2


In [ ]:
DEFAULT_CORPUS = REPO_ROOT / "data" / "processed" / "recipes.parquet"
DEFAULT_OUT_DIR = REPO_ROOT / "artifacts"

def encode_corpus(corpus_path: Path, model_name: str, out_dir: Path, batch_size: int = 128) -> None:
    import pandas as pd
    from sentence_transformers import SentenceTransformer

    corpus = pd.read_parquet(corpus_path)

    def to_text(row) -> str:
        text = str(row.get("canonical_text") or "").strip()
        if not text:
            text = " ".join(row["canonical_ingredients"])
        return text

    texts = [to_text(row) for _, row in corpus.iterrows()]
    print(f"encoding {len(texts)} recipes with {model_name}")

    model = SentenceTransformer(model_name)
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "dense_vectors.npy", vectors)
    meta = {
        "model": model_name,
        "corpus_path": str(corpus_path),
        "column": "canonical_text",
        "normalized": True,
        "dim": int(vectors.shape[1]),
        "n_recipes": int(vectors.shape[0]),
    }
    (out_dir / "dense_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"saved {vectors.shape} vectors to {out_dir / 'dense_vectors.npy'}")
    print(json.dumps(meta, indent=2))

def run_encode_dense_corpus(corpus=DEFAULT_CORPUS, model='all-MiniLM-L6-v2', out_dir=DEFAULT_OUT_DIR, batch_size=128) -> None:
    encode_corpus(corpus, model, out_dir, batch_size=batch_size)


In [ ]:
if RUN_ENCODE_DENSE:
    run_encode_dense_corpus()                   # -> artifacts/dense_vectors.npy + dense_meta.json

## 6. Evaluasi & Eksperimen — `evaluation/`

Harness P1 (ekstraksi & perbandingan pendekatan), P2 (retrieval + analisis error),
dan P3 (ranking/ablation). Hasil historis: `artifacts/p2_experiment_results.json`,
`evaluation/results_pipeline3.{json,csv}`.


**`evaluation/evaluate_extraction.py`**

Pipeline 1 evaluation against data/test/annotated_queries.json (PRD §31).  python -m evaluation.evaluate_extraction python -m evaluation.evaluate_extraction --verbose


In [ ]:
QUERIES = ROOT / "data" / "test" / "annotated_queries.json"

def _scores(pairs: list[tuple[set[str], set[str]]]) -> tuple[float, float, float, int, int, int]:
    tp = sum(len(pred & gold) for pred, gold in pairs)
    fp = sum(len(pred - gold) for pred, gold in pairs)
    fn = sum(len(gold - pred) for pred, gold in pairs)
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1, tp, fp, fn

def run_evaluate_extraction(queries=QUERIES, verbose=False) -> None:

    cases = json.loads(queries.read_text(encoding="utf-8"))["queries"]
    available: list[tuple[set[str], set[str]]] = []
    excluded: list[tuple[set[str], set[str]]] = []
    calorie_hits = {"calorie_target": 0, "calorie_min": 0, "calorie_max": 0}
    exact = 0
    mismatches = []

    for case in cases:
        expected = case["expected"]
        predicted = parse_query(case["query"])
        available.append((set(predicted["available_ingredients"]), set(expected["available_ingredients"])))
        excluded.append((set(predicted["excluded_ingredients"]), set(expected["excluded_ingredients"])))
        for field in calorie_hits:
            calorie_hits[field] += predicted[field] == expected[field]
        if predicted == expected:
            exact += 1
        else:
            mismatches.append((case["id"], case["query"], expected, predicted))

    for label, pairs in (("available_ingredients", available), ("excluded_ingredients", excluded)):
        precision, recall, f1, tp, fp, fn = _scores(pairs)
        print(f"{label:<24} P={precision:.3f} R={recall:.3f} F1={f1:.3f}  (tp={tp} fp={fp} fn={fn})")

    total = len(cases)
    for field, hits in calorie_hits.items():
        print(f"{field:<24} accuracy={hits / total:.3f} ({hits}/{total})")
    print(f"{'structured query exact':<24} {exact}/{total} ({exact / total:.1%})")

    if verbose:
        for query_id, query, expected, predicted in mismatches:
            print(f"\n{query_id} | {query}")
            for field in expected:
                if expected[field] != predicted[field]:
                    print(f"  {field}: expected {expected[field]} got {predicted[field]}")


**`evaluation/evaluate_retrieval.py`**


In [ ]:
DEFAULT_QUERIES_PATH = ROOT / "data" / "test" / "synthetic_retrieval_queries.json"

def generate_in_memory_queries(
    corpus: pd.DataFrame,
    num_queries: int = 200,
    pantry_fraction: float = 0.6,
    min_ingredients: int = 4,
    seed: int = 42,
) -> list[dict[str, object]]:
    rng = random.Random(seed)
    eligible = [
        (int(row["recipe_id"]), str(row["title"]), list(row["canonical_ingredients"]))
        for _, row in corpus.iterrows()
        if len(row["canonical_ingredients"]) >= min_ingredients
    ]
    if not eligible:
        return []
    sampled = rng.sample(eligible, min(num_queries, len(eligible)))
    queries: list[dict[str, object]] = []
    for idx, (recipe_id, title, ingredients) in enumerate(sampled, start=1):
        count = max(3, round(len(ingredients) * pantry_fraction))
        pantry = rng.sample(ingredients, min(count, len(ingredients) - 1))
        held_out = [item for item in ingredients if item not in pantry]
        queries.append(
            {
                "id": f"sim{idx:04d}",
                "recipe_id": recipe_id,
                "title": title,
                "pantry": pantry,
                "held_out": held_out,
                "required": ingredients,
            }
        )
    return queries

def rank_target(retriever: CandidateRetriever, item: dict[str, object], max_k: int) -> int | None:
    """1-based rank of the target recipe in the retriever's top-``max_k``, or ``None``.

    Shared by ``evaluate_retriever`` and the Pipeline 2 experiment harness so the
    retrieval loop is written once.
    """
    structured_query = {
        "available_ingredients": list(item["pantry"]),
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
    indices, _ = retriever.retrieve_indices(structured_query, top_n=max_k)
    for pos, idx in enumerate(indices, start=1):
        if int(retriever.recipe_ids[idx]) == int(item["recipe_id"]):
            return pos
    return None

def evaluate_retriever(
    retriever: CandidateRetriever,
    queries: list[dict[str, object]],
    ks: tuple[int, ...] = (1, 5, 10, 20, 50),
) -> dict[str, float]:
    max_k = max(ks)
    hits_at_k = {k: 0 for k in ks}
    precision_at_k = {k: 0.0 for k in ks}
    reciprocal_ranks: list[float] = []

    valid_ids = set(int(rid) for rid in retriever.recipe_ids)
    evaluated = 0

    for item in queries:
        if int(item["recipe_id"]) not in valid_ids:
            continue
        evaluated += 1
        rank = rank_target(retriever, item, max_k)
        if rank is not None:
            reciprocal_ranks.append(1.0 / rank)
            for k in ks:
                if rank <= k:
                    hits_at_k[k] += 1
                    precision_at_k[k] += 1.0 / k
        else:
            reciprocal_ranks.append(0.0)

    if evaluated == 0:
        return {"evaluated_queries": 0.0, "mrr": 0.0}

    metrics: dict[str, float] = {"evaluated_queries": float(evaluated)}
    for k in ks:
        metrics[f"recall@{k}"] = round(hits_at_k[k] / evaluated, 4)
        metrics[f"precision@{k}"] = round(precision_at_k[k] / evaluated, 4)
    metrics["mrr"] = round(sum(reciprocal_ranks) / evaluated, 4)
    return metrics

def run_evaluate_retrieval(corpus=DEFAULT_CORPUS_PARQUET, vectorizer=DEFAULT_VECTORIZER_PATH, vectors=DEFAULT_VECTORS_PATH, queries_path=DEFAULT_QUERIES_PATH, sample_queries=200) -> None:

    retriever = CandidateRetriever.from_artifacts(
        corpus_path=corpus,
        vectorizer_path=vectorizer,
        vectors_path=vectors,
    )

    queries: list[dict[str, object]] = []
    if queries_path.exists():
        payload = json.loads(queries_path.read_text(encoding="utf-8"))
        queries = payload.get("queries", [])

    metrics = evaluate_retriever(retriever, queries)
    if metrics.get("evaluated_queries", 0.0) == 0.0:
        queries = generate_in_memory_queries(retriever.corpus, num_queries=sample_queries)
        metrics = evaluate_retriever(retriever, queries)

    print(json.dumps(metrics, indent=2))


**`evaluation/experiment.py`**

Compare Pipeline 1 ingredient-matching approaches and report their errors.  Run ``python -m evaluation.experiment``; add ``--verbose`` for query-level diffs and ``--output-json PATH`` to persist the metrics table (used by the Streamlit comparison tab). All variants share the production calorie/exclusion parser so this experiment isolates ingredient matching. Approaches come from the ``src.pipeline1.approaches`` registry (single source of truth); no LLM is used.


In [ ]:
DATASETS = {
    "annotated": ROOT / "data/test/annotated_queries.json",
    "stress": ROOT / "data/test/stress_queries.json",
}
CALORIE_FIELDS = ("calorie_target", "calorie_min", "calorie_max")
FIELDS = ("available_ingredients", "excluded_ingredients", *CALORIE_FIELDS)

def _errors(expected: dict, predicted: dict) -> list[tuple[str, str]]:
    errors = []
    for field, category in (("available_ingredients", "ingredient"), ("excluded_ingredients", "exclusion")):
        gold, got = set(expected[field]), set(predicted[field])
        errors.extend((f"{category}_miss", item) for item in sorted(gold - got))
        errors.extend((f"{category}_extra", item) for item in sorted(got - gold))

    swapped_fields = set()
    for field in CALORIE_FIELDS:
        value = expected[field]
        if value is not None and predicted[field] != value:
            other = next((name for name in CALORIE_FIELDS if name != field and predicted[name] == value), None)
            if other:
                errors.append(("calorie_field", f"{field}->{other}"))
                swapped_fields.add(other)
            elif predicted[field] is None:
                errors.append(("calorie_miss", field))
            else:
                errors.append(("calorie_value", field))
    for field in CALORIE_FIELDS:
        if field not in swapped_fields and expected[field] is None and predicted[field] is not None:
            errors.append(("calorie_extra", field))
    return errors

def _f1(pairs: list[tuple[set[str], set[str]]]) -> float:
    tp = sum(len(pred & gold) for pred, gold in pairs)
    fp = sum(len(pred - gold) for pred, gold in pairs)
    fn = sum(len(gold - pred) for pred, gold in pairs)
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def _approach_names() -> list[str]:
    """All registry approaches with loadable artifacts, in registry order."""
    return [entry["name"] for entry in available_approaches() if entry["available"]]

def run_experiment(
    approach_names: list[str] | None = None,
    datasets: dict[str, Path] | None = None,
    vocab: Vocabulary | None = None,
) -> dict[str, dict]:
    """Evaluate each approach over the annotated + stress fixtures.

    Returns ``{approach: {total, exact, exact_pct, available_f1, excluded_f1,
    calorie_accuracy, errors, failures}}`` — the single source both the CLI
    table and the Streamlit comparison tab render.
    """
    datasets = datasets or DATASETS
    vocab = vocab or load_vocabulary()
    names = approach_names or _approach_names()
    cases = [(name, item) for name, path in datasets.items() for item in json.loads(path.read_text(encoding="utf-8"))["queries"]]

    results: dict[str, dict] = {}
    for name in names:
        approach = get_approach(name)
        exact = 0
        available: list[tuple[set[str], set[str]]] = []
        excluded: list[tuple[set[str], set[str]]] = []
        calorie_hits = 0
        taxonomy: Counter[str] = Counter()
        failures = []
        for dataset, case in cases:
            expected = case["expected"]
            predicted = parse_query(case["query"], vocab, approach.extract)
            errs = _errors(expected, predicted)
            taxonomy.update(category for category, _ in errs)
            exact += predicted == expected
            available.append((set(predicted["available_ingredients"]), set(expected["available_ingredients"])))
            excluded.append((set(predicted["excluded_ingredients"]), set(expected["excluded_ingredients"])))
            calorie_hits += sum(predicted[field] == expected[field] for field in CALORIE_FIELDS)
            if errs:
                failures.append(
                    {
                        "dataset": dataset,
                        "id": case["id"],
                        "query": case["query"],
                        "expected": expected,
                        "predicted": predicted,
                        "errors": errs,
                    }
                )
        total = len(cases)
        results[name] = {
            "total": total,
            "exact": exact,
            "exact_pct": exact / total if total else 0.0,
            "available_f1": round(_f1(available), 4),
            "excluded_f1": round(_f1(excluded), 4),
            "calorie_accuracy": round(calorie_hits / (total * 3), 4) if total else 0.0,
            "errors": dict(taxonomy),
            "failures": failures,
        }
    return results

def format_results_table(results: dict[str, dict]) -> str:
    """ASCII-aligned table, same layout the report historically used."""
    lines = [
        "Approach         Exact     Available F1  Excluded F1  Calorie accuracy",
        "---------------- --------- ------------- ------------ ----------------",
    ]
    for name, metrics in results.items():
        lines.append(
            f"{name:<16} {metrics['exact']:>3}/{metrics['total']:<5}  {metrics['available_f1']:>11.3f}  "
            f"{metrics['excluded_f1']:>10.3f}  {metrics['calorie_accuracy']:>15.3f}"
        )
        errors = ", ".join(f"{key}={value}" for key, value in sorted(metrics["errors"].items())) or "none"
        lines.append(f"  errors: {errors}")
    return "\n".join(lines)

def run_p1_experiment(verbose=False, output_json=None) -> None:

    results = run_experiment()
    print(f"Pipeline 1: {results[next(iter(results))]['total']} queries (annotated + stress)")
    print(format_results_table(results))

    if verbose:
        for name, metrics in results.items():
            for failure in metrics["failures"]:
                print(f"  {name} {failure['dataset']}/{failure['id']}: {failure['query']}")
                print(f"    errors={failure['errors']}; expected={failure['expected']}; predicted={failure['predicted']}")

    if output_json:
        output_json.parent.mkdir(parents=True, exist_ok=True)
        output_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"[info] wrote {output_json}")


**`evaluation/experiment_pipeline2.py`**


In [ ]:
DEFAULT_OUT_JSON = ROOT / "artifacts" / "p2_experiment_results.json"
DEFAULT_OUT_CHART = ROOT / "artifacts" / "p2_error_analysis.png"

def _load_staples(path: Path = DEFAULT_STAPLES_PATH) -> set[str]:
    if not path.exists():
        return set()
    with path.open(encoding="utf-8", newline="") as fh:
        return {normalize_ingredient(r["ingredient"]) for r in csv.DictReader(fh)}

def _build_approaches(corpus: pd.DataFrame) -> list[tuple[str, str, object, set[str] | None]]:
    documents = corpus["canonical_text"].fillna("").astype(str).tolist()
    specs = [
        (
            "tfidf-raw",
            "TF-IDF baseline (max_features=1000, ngram 1-2) tanpa down-weighting pantry staple",
            dict(max_features=1000, ngram_range=(1, 2), sublinear_tf=False, use_staple_downweight=False),
        ),
        (
            "tfidf-staple",
            "TF-IDF produksi (max_features=1000, ngram 1-2) dengan pantry staple down-weighting",
            dict(max_features=1000, ngram_range=(1, 2), sublinear_tf=False, use_staple_downweight=True),
        ),
        (
            "tfidf-unigram",
            "TF-IDF unigram saja (max_features=1000, ngram 1-1) dengan staple down-weighting",
            dict(max_features=1000, ngram_range=(1, 1), sublinear_tf=False, use_staple_downweight=True),
        ),
        (
            "tfidf-unigram-full",
            "TF-IDF unigram kosakata penuh (max_features=None, ngram 1-1, sublinear_tf) + staple down-weight",
            dict(max_features=None, ngram_range=(1, 1), sublinear_tf=True, use_staple_downweight=True),
        ),
    ]

    built: list[tuple[str, str, object, set[str] | None]] = []
    for name, desc, kwargs in specs:
        vec = RecipeVectorizer(**kwargs)
        mat = vec.fit_transform(documents)
        retriever = CandidateRetriever(vectorizer=vec, recipe_vectors=mat, corpus=corpus)
        vocab_terms = set(vec.vectorizer.get_feature_names_out())
        built.append((name, desc, retriever, vocab_terms))

    if DEFAULT_DENSE_VECTORS_PATH.exists() and DEFAULT_DENSE_META_PATH.exists():
        try:
            dense = DenseApproach()
            _ = dense.model
            built.append(
                (
                    "dense-minilm",
                    "Embedding semantik lokal all-MiniLM-L6-v2 (384-dim, L2-normalized) + cosine similarity",
                    dense,
                    None,
                )
            )
        except Exception:
            pass

    return built

def _classify_miss(
    item: dict[str, object],
    vocab_terms: set[str] | None,
    staples: set[str],
) -> str:
    pantry = [str(x) for x in item["pantry"]]
    required = [str(x) for x in item["required"]]

    if vocab_terms is not None:
        query_tokens = query_to_text(pantry).split()
        if any(tok not in vocab_terms for tok in query_tokens):
            return "miss_oov"

    staple_hits = sum(1 for ing in pantry if normalize_ingredient(ing) in staples)
    if pantry and (staple_hits / len(pantry)) >= 0.5:
        return "miss_staple"

    if len(required) >= 8:
        return "miss_long_recipe"

    return "miss_overlap"

def run_p2_experiment(
    corpus_path: Path = DEFAULT_CORPUS_PARQUET,
    num_queries: int = 100,
    pantry_fraction: float = 0.6,
    seed: int = 42,
    miss_cutoff: int = 10,
    ks: tuple[int, ...] = (1, 5, 10, 20, 50),
) -> list[dict[str, object]]:
    corpus = pd.read_parquet(corpus_path).reset_index(drop=True)
    queries = generate_in_memory_queries(
        corpus,
        num_queries=num_queries,
        pantry_fraction=pantry_fraction,
        seed=seed,
    )
    staples = _load_staples()
    approaches = _build_approaches(corpus)
    max_k = max(ks)
    results: list[dict[str, object]] = []

    for name, desc, retriever, vocab_terms in approaches:
        hits = {k: 0 for k in ks}
        rr_list: list[float] = []
        errors = {
            "miss_oov": 0,
            "miss_staple": 0,
            "miss_long_recipe": 0,
            "miss_overlap": 0,
        }
        t0 = time.perf_counter()
        for item in queries:
            rank = rank_target(retriever, item, max_k)

            if rank is not None:
                rr_list.append(1.0 / rank)
                for k in ks:
                    if rank <= k:
                        hits[k] += 1
            else:
                rr_list.append(0.0)

            if rank is None or rank > miss_cutoff:
                cause = _classify_miss(item, vocab_terms, staples)
                errors[cause] += 1

        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        n = max(len(queries), 1)
        row: dict[str, object] = {
            "approach": name,
            "description": desc,
            "queries": len(queries),
            "recall@1": round(hits[1] / n, 4),
            "recall@5": round(hits[5] / n, 4),
            "recall@10": round(hits[10] / n, 4),
            "recall@20": round(hits[20] / n, 4),
            "recall@50": round(hits[50] / n, 4),
            "mrr": round(sum(rr_list) / n, 4),
            "avg_ms": round(elapsed_ms / n, 2),
            "total_misses": sum(errors.values()),
            "errors": errors,
        }
        results.append(row)

    return results

MISS_CATEGORIES = [
    ("miss_oov", "Vocab terpotong (OOV)", "#175935"),
    ("miss_staple", "Dominasi pantry staple", "#2D8659"),
    ("miss_long_recipe", "Resep panjang (>=8 bahan)", "#D6702B"),
    ("miss_overlap", "Kompetitor / overlap leksikal", "#547EB2"),
]

def save_error_chart(
    results: list[dict[str, object]],
    out_path: Path = DEFAULT_OUT_CHART,
    miss_cutoff: int = 10,
) -> Path:
    """Stacked bar chart of retrieval miss causes per approach (matplotlib)."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    sorted_rows = sorted(results, key=lambda r: int(r["total_misses"]), reverse=True)
    labels = [str(r["approach"]) for r in sorted_rows]
    fig, ax = plt.subplots(figsize=(10, max(4.0, 0.62 * len(labels) + 2.5)))

    left = [0] * len(sorted_rows)
    for key, label, color in MISS_CATEGORIES:
        values = [int(r["errors"].get(key, 0)) for r in sorted_rows]
        ax.barh(labels, values, left=left, color=color, label=label)
        left = [l + v for l, v in zip(left, values)]

    ax.invert_yaxis()
    ax.set_title("Dari mana miss retrieval berasal? — Pipeline 2")
    ax.set_xlabel(f"jumlah miss (target gagal masuk Top-{miss_cutoff})")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

def run_p2_experiment_cli(corpus=DEFAULT_CORPUS_PARQUET, queries=100, miss_cutoff=10, out_json=DEFAULT_OUT_JSON, out_chart=DEFAULT_OUT_CHART) -> None:

    results = run_p2_experiment(
        corpus_path=corpus,
        num_queries=queries,
        miss_cutoff=miss_cutoff,
    )
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
    save_error_chart(results, out_path=out_chart, miss_cutoff=miss_cutoff)
    print(json.dumps(results, indent=2))


**`evaluation/evaluate_ranking.py`**

Standalone evaluation & ablation study for Pipeline 3 (PRD Sections 33-35).  Runs independently without waiting for Pipeline 1 & Pipeline 2 by: 1. Reading ground-truth structured queries from ``data/test/annotated_queries.json`` (decoupling Pipeline 3 evaluation from Pipeline 1 parsing errors). 2. Running a self-contained local TF-IDF candidate retriever on ``data/samples/recipes_sample.csv`` (matching PRD §17: TfidfVectorizer with ``max_features=1000, ngram_range=(1, 2)`` + cosine similarity, Top-N=50). 3. Estimating recipe calories on-demand via ``RecipeCalorieEstimator`` using the committed tables in ``data/processed/``. 4. Comparing Baseline vs Proposed and running the full PRD §35 Ablation Study: - Model A: TF-IDF only (Baseline) - Model B: TF-IDF + Ingredient Coverage - Model C: TF-IDF + Ingredient Coverage + Missing Ingredient Penalty - Model D (Full Proposed): TF-IDF + Coverage + Missing Penalty + Calorie Score + Exclusion Filter - Model D (Unweighted): Full Proposed without pantry_staples.csv weights  Usage: python evaluation/evaluate_ranking.py python evaluation/evaluate_ranking.py --top-k 3 --candidate-n 50 --examples 3 python evaluation/evaluate_ranking.py --output-json evaluation/results_pipeline3.json


In [ ]:
EVAL_SAMPLE_CORPUS = REPO_ROOT / "data" / "samples" / "recipes_sample.csv"
DEFAULT_QUERIES = REPO_ROOT / "data" / "test" / "annotated_queries.json"
DEFAULT_PROCESSED = REPO_ROOT / "data" / "processed"

@dataclass
class RawRecipeRow:
    recipe_id: int
    title: str
    raw_ingredients: list[str]
    ner_ingredients: list[str]
    canonical_ingredients: list[str]
    directions: list[str]
    ingredient_doc: str

class LocalTfidfRetriever:
    """Self-contained PRD §17 retriever for standalone Pipeline 3 evaluation."""

    def __init__(
        self,
        corpus_path: Path,
        normalizer: IngredientNormalizer,
        max_features: int = 1000,
    ) -> None:
        self.normalizer = normalizer
        self.recipes: list[RawRecipeRow] = []

        with corpus_path.open(encoding="utf-8", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh)):
                try:
                    ner = ast.literal_eval(row["NER"])
                    raw_ings = ast.literal_eval(row["ingredients"])
                    directions = ast.literal_eval(row["directions"])
                except (ValueError, SyntaxError):
                    continue
                canonical = normalizer.canonicalize_list(ner)
                if not canonical:
                    continue
                # Join canonical ingredients into space-separated document (PRD §17.2)
                doc = " ".join(canonical)
                self.recipes.append(
                    RawRecipeRow(
                        recipe_id=idx,
                        title=row["title"].strip(),
                        raw_ingredients=raw_ings,
                        ner_ingredients=ner,
                        canonical_ingredients=canonical,
                        directions=directions,
                        ingredient_doc=doc,
                    )
                )

        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([r.ingredient_doc for r in self.recipes])

    def retrieve(
        self,
        query: StructuredQuery,
        estimator: RecipeCalorieEstimator,
        calorie_cache: dict[int, float | None],
        top_n: int = 50,
        calorie_mode: str = "per_serving",
    ) -> list[CandidateRecipe]:
        query_terms = self.normalizer.canonicalize_list(query.available_ingredients)
        query_doc = " ".join(query_terms)
        if not query_doc:
            return []

        q_vec = self.vectorizer.transform([query_doc])
        sims = cosine_similarity(q_vec, self.matrix).ravel()
        k = min(top_n, len(self.recipes))
        if k <= 0:
            return []

        top_indices = np.argpartition(sims, -k)[-k:]
        top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]

        candidates: list[CandidateRecipe] = []
        for pos in top_indices:
            sim_score = float(sims[pos])
            if sim_score <= 0.0:
                continue
            row = self.recipes[int(pos)]
            if row.recipe_id not in calorie_cache:
                est = estimator.estimate_recipe(
                    row.raw_ingredients,
                    row.ner_ingredients,
                    row.directions,
                    title=row.title,
                )
                calorie_cache[row.recipe_id] = (
                    est.calories_per_serving if calorie_mode == "per_serving" else est.total_calories
                )
            candidates.append(
                CandidateRecipe(
                    recipe_id=row.recipe_id,
                    title=row.title,
                    canonical_ingredients=row.canonical_ingredients,
                    tfidf_score=sim_score,
                    ingredients=row.raw_ingredients,
                    directions=row.directions,
                    total_calories=calorie_cache[row.recipe_id],
                )
            )
        return candidates

def evaluate_model(
    model_name: str,
    config: RankingConfig,
    reranker: MultiCriteriaReranker,
    query_candidates: list[tuple[str, StructuredQuery, list[CandidateRecipe]]],
    top_k: int,
    mrr_coverage_threshold: float = 0.50,
) -> dict[str, Any]:
    """Compute PRD §§33-34 metrics across all evaluated queries for a single config."""
    weighted_covs: list[float] = []
    unweighted_covs: list[float] = []
    missing_totals: list[int] = []
    missing_non_staples: list[int] = []
    calorie_distances: list[float] = []
    constraint_violations = 0
    exclusion_violations = 0
    calorie_bound_violations = 0
    total_recommended = 0
    total_with_exclusion_query = 0
    total_with_calorie_bound_query = 0
    total_with_any_constraint_query = 0
    reciprocal_ranks: list[float] = []
    fallback_queries = 0

    for _qid, query, candidates in query_candidates:
        result = reranker.rerank(candidates, query, top_k=top_k, config=config)
        if result.is_fallback:
            fallback_queries += 1

        recs = result.recommendations
        first_relevant_rr = 0.0

        for rec in recs:
            total_recommended += 1
            weighted_covs.append(rec.coverage.weighted_coverage)
            unweighted_covs.append(rec.coverage.unweighted_coverage)
            missing_totals.append(rec.coverage.missing_count)
            missing_non_staples.append(rec.coverage.missing_non_staple_count)

            if query.has_calorie_constraint and rec.calorie.calorie_distance is not None:
                calorie_distances.append(rec.calorie.calorie_distance)

            excl_bad = bool(query.excluded_ingredients) and rec.exclusion.violated
            cal_bad = query.has_hard_calorie_bound and (not rec.calorie.passed_hard_filter)
            any_bad = excl_bad or cal_bad

            if query.excluded_ingredients:
                total_with_exclusion_query += 1
                if excl_bad:
                    exclusion_violations += 1

            if query.has_hard_calorie_bound:
                total_with_calorie_bound_query += 1
                if cal_bad:
                    calorie_bound_violations += 1

            if query.excluded_ingredients or query.has_hard_calorie_bound:
                total_with_any_constraint_query += 1
                if any_bad:
                    constraint_violations += 1

            # Practical cookability relevance for MRR:
            # no exclusion violation, passes hard calorie bound, and weighted coverage >= threshold
            if first_relevant_rr == 0.0 and (not any_bad) and rec.coverage.weighted_coverage >= mrr_coverage_threshold:
                first_relevant_rr = 1.0 / rec.rank

        reciprocal_ranks.append(first_relevant_rr)

    def _mean(values: list[float] | list[int]) -> float:
        return float(sum(values) / len(values)) if values else 0.0

    return {
        "model": model_name,
        "top_k": top_k,
        "num_queries": len(query_candidates),
        "weighted_coverage_at_k": round(_mean(weighted_covs), 4),
        "unweighted_coverage_at_k": round(_mean(unweighted_covs), 4),
        "avg_missing_at_k": round(_mean(missing_totals), 3),
        "avg_missing_non_staple_at_k": round(_mean(missing_non_staples), 3),
        "calorie_distance": round(_mean(calorie_distances), 1),
        "constraint_violation_rate": round(
            constraint_violations / total_with_any_constraint_query, 4
        ) if total_with_any_constraint_query else 0.0,
        "exclusion_violation_rate": round(
            exclusion_violations / total_with_exclusion_query, 4
        ) if total_with_exclusion_query else 0.0,
        "calorie_bound_violation_rate": round(
            calorie_bound_violations / total_with_calorie_bound_query, 4
        ) if total_with_calorie_bound_query else 0.0,
        "mrr": round(_mean(reciprocal_ranks), 4),
        "fallback_rate": round(fallback_queries / len(query_candidates), 4) if query_candidates else 0.0,
        "total_recommended": total_recommended,
    }

def format_markdown_table(rows: list[dict[str, Any]], top_k: int) -> str:
    headers = [
        "Model",
        f"Weighted Cov@{top_k} ↑",
        f"Unweighted Cov@{top_k} ↑",
        f"Missing@{top_k} ↓",
        f"Non-Staple Miss@{top_k} ↓",
        "Calorie Dist (kcal) ↓",
        "Constraint Viol ↓",
        "Excl Viol ↓",
        "Cal Viol ↓",
        "MRR ↑",
        "Fallback",
    ]
    sep = ["---"] + ["---:"] * (len(headers) - 1)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(sep) + " |"]
    for r in rows:
        lines.append(
            "| "
            + " | ".join([
                r["model"],
                f"{r['weighted_coverage_at_k']:.1%}",
                f"{r['unweighted_coverage_at_k']:.1%}",
                f"{r['avg_missing_at_k']:.2f}",
                f"{r['avg_missing_non_staple_at_k']:.2f}",
                f"{r['calorie_distance']:.1f}",
                f"{r['constraint_violation_rate']:.1%}",
                f"{r['exclusion_violation_rate']:.1%}",
                f"{r['calorie_bound_violation_rate']:.1%}",
                f"{r['mrr']:.3f}",
                f"{r['fallback_rate']:.1%}",
            ])
            + " |"
        )
    return "\n".join(lines)

def run_evaluate_ranking(corpus=EVAL_SAMPLE_CORPUS, queries=DEFAULT_QUERIES, processed_dir=DEFAULT_PROCESSED, top_k=3, candidate_n=50, calorie_mode='per_serving', examples=2, output_json=None, output_csv=None) -> None:

    t0 = time.perf_counter()
    normalizer = IngredientNormalizer(processed_dir)
    staple_weights = PantryStapleWeights.from_csv(
        processed_dir / "pantry_staples.csv",
        normalizer=normalizer,
    )
    estimator = RecipeCalorieEstimator(processed_dir=processed_dir, normalizer=normalizer)
    reranker = MultiCriteriaReranker(
        processed_dir=processed_dir,
        normalizer=normalizer,
        staple_weights=staple_weights,
    )

    print(f"[info] Loading corpus: {corpus}")
    retriever = LocalTfidfRetriever(corpus, normalizer=normalizer)
    print(f"[info] Indexed {len(retriever.recipes):,} recipes with TF-IDF ({time.perf_counter() - t0:.2f}s)")

    raw_queries = json.loads(queries.read_text(encoding="utf-8"))["queries"]
    calorie_cache: dict[int, float | None] = {}
    query_candidates: list[tuple[str, StructuredQuery, list[CandidateRecipe]]] = []

    t_ret = time.perf_counter()
    for entry in raw_queries:
        sq = StructuredQuery.from_dict(entry["expected"], raw_query=entry["query"])
        cands = retriever.retrieve(
            sq,
            estimator=estimator,
            calorie_cache=calorie_cache,
            top_n=candidate_n,
            calorie_mode=calorie_mode,
        )
        query_candidates.append((entry["id"], sq, cands))

    estimated_non_null = sum(1 for v in calorie_cache.values() if v is not None)
    print(
        f"[info] Retrieved Top-{candidate_n} candidates for {len(query_candidates)} queries; "
        f"estimated calories for {estimated_non_null}/{len(calorie_cache)} unique candidate recipes "
        f"({estimated_non_null / max(len(calorie_cache), 1):.1%}) in {time.perf_counter() - t_ret:.2f}s"
    )

    # PRD Section 35 Ablation Study + Pantry Staple Weight validation
    ablation_models: list[tuple[str, RankingConfig]] = [
        ("Model A: TF-IDF Only (Baseline)", RankingConfig.ablation_model_a()),
        ("Model B: TF-IDF + Coverage", RankingConfig.ablation_model_b()),
        ("Model C: TF-IDF + Cov + Missing Pen.", RankingConfig.ablation_model_c()),
        ("Model D: Full Proposed (Weighted)", RankingConfig.ablation_model_d()),
        ("Model D (Unweighted Staples)", RankingConfig.ablation_unweighted()),
    ]

    results = [
        evaluate_model(name, cfg, reranker, query_candidates, top_k=top_k)
        for name, cfg in ablation_models
    ]

    print("\n=== Pipeline 3 Evaluation & Ablation Study (PRD §§33-35) ===")
    print(
        f"Corpus: {corpus.name} | Queries: {queries.name} ({len(query_candidates)}) | "
        f"Top-N={candidate_n} -> Top-K={top_k} | Calorie mode: {calorie_mode}"
    )
    print("Note: MRR measures first Top-K recipe with 0 constraint violations & weighted coverage >= 50%.\n")
    print(format_markdown_table(results, top_k=top_k))

    # Print qualitative Baseline vs Proposed examples for combined-constraint queries
    if examples > 0:
        demo_ids = ("q041", "q042", "q044", "q045", "q056", "q060")
        selected_demos = [qc for qc in query_candidates if qc[0] in demo_ids][: examples]
        if not selected_demos:
            selected_demos = query_candidates[: examples]

        print("\n=== Qualitative Comparison: Baseline (Model A) vs Proposed (Model D) ===")
        for qid, sq, cands in selected_demos:
            base_res = reranker.rerank(cands, sq, top_k=1, config=RankingConfig.ablation_model_a())
            prop_res = reranker.rerank(cands, sq, top_k=1, config=RankingConfig.ablation_model_d())
            print(f"\n[{qid}] Query: {sq.raw_query}")
            print("  -- Baseline #1 (TF-IDF Only) --")
            if base_res.recommendations:
                print(
                    format_recommendation_card(
                        base_res.recommendations[0],
                        excluded_ingredients=sq.excluded_ingredients,
                        include_ingredients=False,
                        include_directions=False,
                    )
                )
            print("\n  -- Proposed #1 (Full Multi-Criteria Reranker) --")
            if prop_res.is_fallback and prop_res.fallback_message:
                print(f"  [NOTICE] {prop_res.fallback_message}")
            if prop_res.recommendations:
                print(
                    format_recommendation_card(
                        prop_res.recommendations[0],
                        excluded_ingredients=sq.excluded_ingredients,
                        include_ingredients=False,
                        include_directions=False,
                    )
                )

    if output_json is not None:
        output_json.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "corpus": str(corpus),
            "queries": str(queries),
            "top_k": top_k,
            "candidate_n": candidate_n,
            "calorie_mode": calorie_mode,
            "unique_candidates_evaluated": len(calorie_cache),
            "calorie_estimation_coverage": round(estimated_non_null / max(len(calorie_cache), 1), 4),
            "ablation_results": results,
        }
        output_json.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        print(f"\n[ok] Saved JSON report -> {output_json}")

    if output_csv is not None:
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        with output_csv.open("w", encoding="utf-8", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(results[0].keys()))
            writer.writeheader()
            writer.writerows(results)
        print(f"[ok] Saved CSV table -> {output_csv}")


In [ ]:
if RUN_EVALUATIONS:
    run_evaluate_extraction()                   # P/R/F1 ekstraksi pada 60 query teranotasi
    run_p1_experiment()                         # perbandingan rule/exact/difflib/n-gram
    run_evaluate_retrieval()                    # recall/MRR retrieval TF-IDF
    run_p2_experiment_cli()                     # eksperimen P2 + grafik analisis error
    run_evaluate_ranking()                      # evaluasi ranking Pipeline 3

## 7. Test Suite — `tests/`

Seluruh test regresi repositori, dijalankan langsung di notebook. Fixture pytest
(`vocab`, `tmp_path`) diganti variabel/`tempfile` biasa; `test_pipeline3` tetap
`unittest.TestCase` dan dijalankan lewat runner yang sama.


**`tests/test_preprocessing.py`**


In [ ]:
def test_singularize_and_normalize_text() -> None:
    assert singularize("potatoes") == "potato"
    assert singularize("tomatoes") == "tomato"
    assert singularize("eggs") == "egg"
    assert singularize("berries") == "berry"
    # normalize_text is a pure query-time fold (no stemming); only
    # normalize_ingredient folds the head noun.
    assert normalize_text("Fresh Green Onions!") == "fresh green onions"
    assert normalize_ingredient("Fresh Green Onions!") == "fresh green onion"

def test_ingredient_normalizer_aliases_and_qualifiers() -> None:
    normalizer = IngredientNormalizer()
    assert normalizer.resolve("scallions") == "green onion"
    assert normalizer.resolve("catsup") == "ketchup"
    assert normalizer.resolve("granulated sugar") == "sugar"
    assert normalizer.resolve("unsalted butter") == "butter"
    assert normalizer.extract_canonical_ingredients(
        ["chickens", "potatoes", "scallions", "green onions", "salt"]
    ) == ["chicken", "potato", "green onion", "salt"]

def test_parse_ingredient_quantities_units_and_containers() -> None:
    p1 = parse_ingredient("2 tbsp butter")
    assert p1["quantity"] == 2.0
    assert p1["unit"] == "tbsp"
    assert p1["ingredient"] == "butter"

    p2 = parse_ingredient("1 1/2 c. sugar")
    assert p2["quantity"] == 1.5
    assert p2["unit"] == "cup"
    assert p2["ingredient"] == "sugar"

    p3 = parse_ingredient("1 (16 oz.) can crushed pineapple, drained")
    assert p3["quantity"] == 16.0
    assert p3["unit"] == "oz"
    assert p3["ingredient"] == "pineapple"

    p4 = parse_ingredient("200 g chicken breast")
    assert p4["quantity"] == 200.0
    assert p4["unit"] == "g"
    assert p4["ingredient"] == "chicken breast"

    p5 = parse_ingredient("salt and pepper")
    assert p5["quantity"] is None
    assert p5["unit"] is None
    assert p5["ingredient"] == "salt and pepper"

    parsed_many = parse_ingredients(["1/2 cup milk", "3 medium apples, chopped"])
    assert len(parsed_many) == 2
    assert parsed_many[0]["quantity"] == 0.5
    assert parsed_many[1]["unit"] == "medium"

def test_calorie_estimator_estimation() -> None:
    mapper = RecipeCalorieEstimator()
    butter_lookup = mapper.lookup_calorie_per_100g("unsalted butter")
    assert butter_lookup is not None
    assert butter_lookup["canonical"] == "butter"
    assert float(butter_lookup["kcal_per_100g"]) > 500.0

    water_lookup = mapper.lookup_calorie_per_100g("cold water")
    assert water_lookup is not None
    assert water_lookup["match_type"] == "zero"
    assert float(water_lookup["kcal_per_100g"]) == 0.0

    recipe_est = mapper.estimate_recipe_calories(
        [
            "2 tbsp butter",
            "1/2 cup milk",
            "200 g chicken breast",
            "1 tsp salt",
        ],
        ner_ingredients=["butter", "milk", "chicken breast", "salt"],
    )
    assert float(recipe_est["total_calories"]) > 300.0
    assert float(recipe_est["calorie_coverage"]) == 1.0

def test_build_processed_corpus_subset():
    tmp_path = Path(tempfile.mkdtemp())
    out_parquet = tmp_path / "recipes.parquet"
    out_vocab = tmp_path / "ingredient_vocabulary.pkl"
    df = build_processed_corpus(
        corpus_path=SAMPLE_CSV,
        output_path=out_parquet,
        vocab_path=out_vocab,
        limit=25,
    )
    assert len(df) == 25
    assert out_parquet.exists()
    assert out_vocab.exists()
    assert {
        "recipe_id",
        "title",
        "ingredients",
        "canonical_ingredients",
        "canonical_text",
        "directions",
        "total_calories",
        "calorie_coverage",
    }.issubset(df.columns)
    vocab = IngredientNormalizer.load_vocabulary(out_vocab)
    assert len(vocab) > 100


**`tests/test_pipeline1.py`**

Pipeline 1 regression guards for behaviour the annotated set only implies.


In [ ]:
def test_exclusion_clause_removes_earlier_availability():
    result = parse_query("I have potatoes, butter and garlic. Without milk or butter.")
    assert result["available_ingredients"] == ["potato", "garlic"]
    assert result["excluded_ingredients"] == ["milk", "butter"]

def test_upper_bound_is_not_an_exclusion():
    result = parse_query("I have beef and noodles. No more than 700 calories.")
    assert result["excluded_ingredients"] == []
    assert result["calorie_max"] == 700
    assert result["available_ingredients"] == ["beef", "noodle"]

def test_bare_kcal_number_is_not_a_constraint():
    result = parse_query("I have chicken and rice. 450 kcal.")
    assert result["calorie_target"] is None
    assert result["calorie_max"] is None
    assert result["calorie_min"] is None

def test_range_sets_both_bounds():
    result = parse_query("I have chicken, potato, carrot and onion. Between 400 and 600 calories.")
    assert (result["calorie_min"], result["calorie_max"], result["calorie_target"]) == (400, 600, None)

def test_longest_match_wins_over_substring():
    result = parse_query("Available: tomatoes, onions, garlic, olive oil.")
    assert result["available_ingredients"] == ["tomato", "onion", "garlic", "olive oil"]

def test_plural_exclusion_and_generic_nut():
    result = parse_query("I have oats, banana, milk and honey. Around 350 calories, no nuts.")
    assert result["excluded_ingredients"] == ["nut"]
    assert result["calorie_target"] == 350

def test_tilde_approx_target_is_recognized():
    result = parse_query("i have chicken potato carrot onion garlic. ~500 calories")
    assert result["available_ingredients"] == ["chicken", "potato", "carrot", "onion", "garlic"]
    assert result["calorie_target"] == 500

def test_two_exclusion_cues_in_one_clause():
    result = parse_query("I have salmon and rice. No butter and no cream.")
    assert result["excluded_ingredients"] == ["butter", "cream"]

def test_query_without_pantry_still_parses():
    result = parse_query("I want a low calorie dinner.")
    assert result == {
        "available_ingredients": [],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
 
def test_range_accepts_collapsed_symbol_separator():
    result = parse_query("I have rice. Between 300 & 500 cal.")
    assert (result["calorie_min"], result["calorie_max"]) == (300, 500)

def test_joined_compound_ingredient():
    result = parse_query("I have sweetpotato and spinach.")
    assert result["available_ingredients"] == ["sweet potato", "spinach"]


**`tests/test_approaches.py`**

Approach registry, trace runner, and retrieval approach guards.


In [ ]:
def test_registry_lists_all_lexical_approaches():
    names = [entry["name"] for entry in available_approaches()]
    assert names[:4] == ["rule", "surface-exact", "difflib-0.82", "char-ngram-0.60"]
    # ner is registry-listed regardless of artifact presence (flagged via `available`)
    assert "ner" in names

def test_rule_approach_matches_parse_query_default():
    approach = get_approach("rule")
    for query in (
        "I have chicken, potato, carrot, onion and garlic. Around 500 calories and no cheese.",
        "no milk or butter, just beef",
        "sweetpotato and green onions",
    ):
        assert parse_query(query, vocab, approach.extract) == parse_query(query, vocab)

def test_trace_parse_query_records_spans():
    trace = trace_parse_query("I have eggs. Around 500 calories, no cheese.", get_approach("rule"), vocab)
    assert trace.structured_query == parse_query(
        "I have eggs. Around 500 calories, no cheese.", vocab
    )
    assert trace.calorie_spans, "calorie span must be recorded"
    assert trace.exclusion_spans, "exclusion span must be recorded"
    assert [m.canonical for m in trace.available_trace.matches] == ["egg"]
    assert trace.excluded_trace.ingredients == ["cheese"]

def test_fuzzy_approach_rescues_typos():
    approach = get_approach("difflib-0.82")
    trace = trace_parse_query("I have chiken and tommato.", approach, vocab)
    assert trace.structured_query["available_ingredients"] == ["chicken", "tomato"]
    methods = {m.method for m in trace.available_trace.matches}
    assert "fuzzy-difflib" in methods

def test_ner_approach_requires_artifacts():
    tmp_path = Path(tempfile.mkdtemp())
    missing = NerApproach(model_dir=tmp_path / "nope")
    if not missing.available:
        with pytest.raises(FileNotFoundError):
            missing._load()
    else:  # artifacts present in this checkout — loading must succeed
        missing._load()

def test_p2_approaches_share_candidate_schema():

    approach = TfidfApproach()
    candidates = approach.retrieve(
        {"available_ingredients": ["chicken", "potato"], "excluded_ingredients": []}, top_n=5
    )
    required = {"recipe_id", "title", "canonical_ingredients", "tfidf_score", "total_calories", "calorie_coverage"}
    assert candidates and required.issubset(candidates[0].keys())
    # harness compatibility (evaluate_retriever reads these directly)
    assert approach.recipe_ids is not None
    indices, scores = approach.retrieve_indices(
        {"available_ingredients": ["chicken"], "excluded_ingredients": []}, top_n=3
    )
    assert len(indices) == len(scores) == 3


**`tests/test_pipeline2.py`**


In [ ]:
DUMMY_STRUCTURED_QUERIES = [
    {
        "available_ingredients": ["chicken", "potato", "carrot", "onion", "garlic"],
        "excluded_ingredients": ["cheese"],
        "calorie_target": 500,
        "calorie_min": None,
        "calorie_max": None,
    },
    {
        "available_ingredients": ["egg", "milk", "butter", "flour", "sugar"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": 600,
    },
    {
        "available_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
        "excluded_ingredients": ["milk"],
        "calorie_target": None,
        "calorie_min": 400,
        "calorie_max": 700,
    },
]

def _make_dummy_corpus() -> pd.DataFrame:
    rows = [
        {
            "recipe_id": 0,
            "title": "Chicken Potato Soup",
            "ingredients": ["200 g chicken", "2 potatoes", "1 carrot", "1 onion", "2 cloves garlic", "1 stalk celery"],
            "canonical_ingredients": ["chicken", "potato", "carrot", "onion", "garlic", "celery"],
            "canonical_text": "chicken potato carrot onion garlic celery",
            "directions": ["Boil all ingredients until tender."],
            "total_calories": 482.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 1,
            "title": "Basic Pancakes",
            "ingredients": ["1 cup flour", "1 cup milk", "1 egg", "2 tbsp butter", "2 tbsp sugar", "1 tsp salt"],
            "canonical_ingredients": ["flour", "milk", "egg", "butter", "sugar", "salt"],
            "canonical_text": "flour milk egg butter sugar salt",
            "directions": ["Mix batter and cook on griddle."],
            "total_calories": 520.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 2,
            "title": "Beef Tomato Pasta",
            "ingredients": ["250 g ground beef", "2 tomatoes", "1 onion", "2 cloves garlic", "200 g pasta"],
            "canonical_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
            "canonical_text": "ground beef tomato onion garlic pasta",
            "directions": ["Brown beef, simmer with tomato, toss with pasta."],
            "total_calories": 640.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 3,
            "title": "Salted Water Broth",
            "ingredients": ["4 cups water", "2 tsp salt", "1 tsp pepper"],
            "canonical_ingredients": ["water", "salt", "pepper"],
            "canonical_text": "water salt pepper",
            "directions": ["Boil water with salt and pepper."],
            "total_calories": 0.0,
            "calorie_coverage": 1.0,
        },
    ]
    return pd.DataFrame(rows)

def test_dummy_query_vectorization_and_retrieval() -> None:
    corpus = _make_dummy_corpus()
    vectorizer = RecipeVectorizer(max_features=1000, ngram_range=(1, 2), use_staple_downweight=True)
    recipe_vectors = vectorizer.fit_transform(corpus["canonical_text"].tolist())
    retriever = CandidateRetriever(vectorizer=vectorizer, recipe_vectors=recipe_vectors, corpus=corpus)

    q_vec = vectorizer.transform_query(DUMMY_STRUCTURED_QUERIES[0])
    assert q_vec.shape == (1, recipe_vectors.shape[1])
    assert q_vec.nnz > 0

    candidates_soup = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[0], top_n=3)
    assert len(candidates_soup) == 3
    assert candidates_soup[0]["title"] == "Chicken Potato Soup"
    assert candidates_soup[0]["tfidf_score"] >= candidates_soup[1]["tfidf_score"]

    candidates_pancake = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[1], top_n=3)
    assert candidates_pancake[0]["title"] == "Basic Pancakes"

    candidates_pasta = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[2], top_n=3)
    assert candidates_pasta[0]["title"] == "Beef Tomato Pasta"

def test_staple_downweight_reduces_staple_dominance() -> None:
    corpus = _make_dummy_corpus()
    vec_weighted = RecipeVectorizer(max_features=1000, ngram_range=(1, 2), use_staple_downweight=True)
    mat_weighted = vec_weighted.fit_transform(corpus["canonical_text"].tolist())
    retriever = CandidateRetriever(vectorizer=vec_weighted, recipe_vectors=mat_weighted, corpus=corpus)

    dummy_query = {
        "available_ingredients": ["flour", "egg", "salt", "water", "pepper"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
    candidates = retriever.retrieve(dummy_query, top_n=2)
    assert candidates[0]["title"] == "Basic Pancakes"

def test_pipeline2_end_to_end_with_artifacts_and_evaluation():
    tmp_path = Path(tempfile.mkdtemp())
    corpus_parquet = tmp_path / "recipes.parquet"
    vocab_pkl = tmp_path / "ingredient_vocabulary.pkl"
    vec_pkl = tmp_path / "tfidf_vectorizer.pkl"
    mat_pkl = tmp_path / "recipe_vectors.pkl"

    build_processed_corpus(
        corpus_path=SAMPLE_CSV,
        output_path=corpus_parquet,
        vocab_path=vocab_pkl,
        limit=100,
    )
    fit_corpus_tfidf(
        corpus_parquet=corpus_parquet,
        vectorizer_path=vec_pkl,
        vectors_path=mat_pkl,
        max_features=1000,
        use_staple_downweight=True,
    )

    retriever = CandidateRetriever.from_artifacts(
        corpus_path=corpus_parquet,
        vectorizer_path=vec_pkl,
        vectors_path=mat_pkl,
    )
    candidates = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[1], top_n=10)
    assert len(candidates) == 10
    assert all("tfidf_score" in c and "canonical_ingredients" in c for c in candidates)

    queries = generate_in_memory_queries(retriever.corpus, num_queries=20, seed=42)
    metrics = evaluate_retriever(retriever, queries, ks=(1, 5, 10, 50))
    assert metrics["evaluated_queries"] == 20.0
    assert 0.0 <= metrics["recall@50"] <= 1.0
    assert 0.0 <= metrics["mrr"] <= 1.0


**`tests/test_pipeline3.py`**

Unit and integration tests for Pipeline 3 (Pantry-Aware Multi-Criteria Reranking).  Compatible with both ``pytest`` and standard-library ``python -m unittest``.


In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
ANNOTATED_JSON = REPO_ROOT / "data" / "test" / "annotated_queries.json"

class TestNormalization(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        cls.normalizer = IngredientNormalizer(PROCESSED_DIR)

    def test_plural_folding(self) -> None:
        self.assertEqual(normalize_ingredient("potatoes"), "potato")
        self.assertEqual(normalize_ingredient("tomatoes"), "tomato")
        self.assertEqual(normalize_ingredient("eggs"), "egg")
        self.assertEqual(normalize_ingredient("berries"), "berry")
        self.assertEqual(normalize_ingredient("asparagus"), "asparagus")

    def test_alias_resolution_and_qualifiers(self) -> None:
        self.assertEqual(self.normalizer.canonicalize("scallions"), "green onion")
        self.assertEqual(self.normalizer.canonicalize("capsicum"), "bell pepper")
        self.assertEqual(self.normalizer.canonicalize("white sugar"), "sugar")
        self.assertEqual(self.normalizer.canonicalize("hamburger"), "ground beef")
        self.assertEqual(self.normalizer.canonicalize("fresh minced garlic"), "garlic")
        self.assertEqual(self.normalizer.canonicalize("kosher salt"), "salt")

class TestCoverageAndPantryStaples(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        cls.normalizer = IngredientNormalizer(PROCESSED_DIR)
        cls.staples = PantryStapleWeights.from_csv(
            PROCESSED_DIR / "pantry_staples.csv",
            normalizer=cls.normalizer,
        )

    def test_pantry_staple_weights_loaded_from_csv(self) -> None:
        self.assertAlmostEqual(self.staples.get_weight("water"), 0.05)
        self.assertAlmostEqual(self.staples.get_weight("salt"), 0.15)
        self.assertAlmostEqual(self.staples.get_weight("pepper"), 0.15)
        self.assertAlmostEqual(self.staples.get_weight("cream"), 0.80)
        # Unknown/main ingredients default to 1.0 (data/README.md)
        self.assertAlmostEqual(self.staples.get_weight("chicken"), 1.0)
        self.assertAlmostEqual(self.staples.get_weight("potato"), 1.0)
        self.assertTrue(self.staples.is_staple("salt"))
        self.assertFalse(self.staples.is_staple("chicken"))

    def test_prd_section19_unweighted_coverage_example(self) -> None:
        # PRD §19: U={chicken, potato, carrot, onion, garlic}, R=U+{celery} -> 5/6 = 0.833
        user = ["chicken", "potato", "carrot", "onion", "garlic"]
        recipe = ["chicken", "potato", "carrot", "onion", "garlic", "celery"]
        cov = compute_coverage(
            user,
            recipe,
            staple_weights=self.staples,
            normalizer=self.normalizer,
            use_weights=False,
        )
        self.assertAlmostEqual(cov.unweighted_coverage, 5 / 6, places=4)
        self.assertAlmostEqual(cov.coverage_score, 5 / 6, places=4)
        self.assertAlmostEqual(cov.unweighted_missing_penalty, 1 / 6, places=4)
        self.assertEqual(cov.missing_ingredients, ["celery"])
        self.assertEqual(len(cov.available_matched), 5)

    def test_weighted_coverage_downweights_missing_staples(self) -> None:
        # Missing only salt+water should score much higher than missing celery+cream
        user = ["chicken", "potato"]
        recipe_missing_staples = ["chicken", "potato", "salt", "water"]
        recipe_missing_mains = ["chicken", "potato", "celery", "cream"]

        cov_staples = compute_coverage(
            user, recipe_missing_staples, self.staples, self.normalizer, use_weights=True
        )
        cov_mains = compute_coverage(
            user, recipe_missing_mains, self.staples, self.normalizer, use_weights=True
        )

        # Unweighted coverages are identical (2/4 = 0.5)
        self.assertAlmostEqual(cov_staples.unweighted_coverage, 0.5)
        self.assertAlmostEqual(cov_mains.unweighted_coverage, 0.5)

        # Weighted coverage strongly favors recipe missing only staples
        # staples: (1+1)/(1+1+0.15+0.05) = 2.0 / 2.2 = 0.9091
        # mains  : (1+1)/(1+1+1.0+0.80)  = 2.0 / 3.8 = 0.5263
        self.assertAlmostEqual(cov_staples.weighted_coverage, 2.0 / 2.2, places=4)
        self.assertAlmostEqual(cov_mains.weighted_coverage, 2.0 / 3.8, places=4)
        self.assertGreater(cov_staples.coverage_score, cov_mains.coverage_score)
        self.assertLess(cov_staples.missing_penalty, cov_mains.missing_penalty)
        self.assertEqual(cov_staples.missing_non_staples, [])
        self.assertEqual(set(cov_staples.missing_staples), {"salt", "water"})

    def test_empty_recipe_ingredients_handled_safely(self) -> None:
        cov = compute_coverage(["chicken"], [], self.staples, self.normalizer)
        self.assertEqual(cov.coverage_score, 0.0)
        self.assertEqual(cov.missing_penalty, 0.0)
        self.assertEqual(cov.missing_ingredients, [])

class TestCalorieCompatibility(unittest.TestCase):
    def test_prd_section22_target_examples(self) -> None:
        # PRD §22.2: target=500 -> 490: 0.98, 450: 0.90, 300: 0.60
        c_a = evaluate_calorie_compatibility(490, calorie_target=500)
        c_b = evaluate_calorie_compatibility(450, calorie_target=500)
        c_c = evaluate_calorie_compatibility(300, calorie_target=500)
        c_far = evaluate_calorie_compatibility(1200, calorie_target=500)

        self.assertAlmostEqual(c_a.score, 0.98, places=4)
        self.assertAlmostEqual(c_b.score, 0.90, places=4)
        self.assertAlmostEqual(c_c.score, 0.60, places=4)
        self.assertEqual(c_far.score, 0.0)
        self.assertTrue(c_far.passed_hard_filter)  # target is soft

    def test_max_calorie_hard_filter(self) -> None:
        ok = evaluate_calorie_compatibility(480, calorie_max=500)
        bad = evaluate_calorie_compatibility(650, calorie_max=500)

        self.assertTrue(ok.passed_hard_filter)
        self.assertEqual(ok.score, 1.0)
        self.assertEqual(ok.calorie_distance, 0.0)

        self.assertFalse(bad.passed_hard_filter)
        self.assertAlmostEqual(bad.calorie_distance, 150.0)
        self.assertAlmostEqual(bad.score, 1.0 - 150.0 / 500.0, places=4)

    def test_min_and_range_calorie_constraints(self) -> None:
        inside = evaluate_calorie_compatibility(500, calorie_min=400, calorie_max=600)
        below = evaluate_calorie_compatibility(300, calorie_min=400, calorie_max=600)
        above = evaluate_calorie_compatibility(700, calorie_min=400, calorie_max=600)
        min_only_ok = evaluate_calorie_compatibility(650, calorie_min=600)
        min_only_low = evaluate_calorie_compatibility(450, calorie_min=600)

        self.assertTrue(inside.passed_hard_filter)
        self.assertEqual(inside.score, 1.0)
        self.assertEqual(inside.calorie_distance, 0.0)

        self.assertFalse(below.passed_hard_filter)
        self.assertEqual(below.calorie_distance, 100.0)
        self.assertFalse(above.passed_hard_filter)
        self.assertEqual(above.calorie_distance, 100.0)

        self.assertTrue(min_only_ok.passed_hard_filter)
        self.assertFalse(min_only_low.passed_hard_filter)

    def test_no_constraint_and_unknown_calories(self) -> None:
        none_c = evaluate_calorie_compatibility(500)
        self.assertEqual(none_c.constraint_type, "none")
        self.assertEqual(none_c.score, 1.0)
        self.assertTrue(none_c.passed_hard_filter)

        unk_soft = evaluate_calorie_compatibility(None, calorie_target=500)
        self.assertTrue(unk_soft.passed_hard_filter)
        self.assertAlmostEqual(unk_soft.score, 0.5)

        unk_hard = evaluate_calorie_compatibility(None, calorie_max=500)
        self.assertFalse(unk_hard.passed_hard_filter)

class TestExclusionFilter(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        cls.normalizer = IngredientNormalizer(PROCESSED_DIR)
        cls.ef = ExclusionFilter(normalizer=cls.normalizer)

    def test_exact_and_category_cheese_exclusions(self) -> None:
        # "no cheese" must catch all cheese subtypes even though alias maps cheese->cheddar cheese
        for ing in ("cheese", "cheddar cheese", "mozzarella", "parmesan cheese", "cream cheese", "ricotta cheese"):
            res = self.ef.check(["cheese"], ["chicken", "tomato", ing])
            self.assertTrue(res.violated, msg=f"Expected 'no cheese' to catch {ing!r}")
            self.assertEqual(res.violation_count, 1)

    def test_nut_category_and_false_positive_guards(self) -> None:
        self.assertTrue(self.ef.check(["nut"], ["oats", "banana", "walnuts"]).violated)
        self.assertTrue(self.ef.check(["nut"], ["oats", "banana", "almond"]).violated)
        self.assertTrue(self.ef.check(["nut"], ["oats", "banana", "mixed nuts"]).violated)
        # False-positive guards: nutmeg and butternut squash are NOT nuts
        self.assertFalse(self.ef.check(["nut"], ["oats", "banana", "nutmeg"]).violated)
        self.assertFalse(self.ef.check(["nut"], ["butternut squash", "onion"]).violated)

    def test_butter_and_cream_false_positive_guards(self) -> None:
        self.assertTrue(self.ef.check(["butter"], ["potato", "unsalted butter"]).violated)
        self.assertFalse(self.ef.check(["butter"], ["bread", "peanut butter"]).violated)
        self.assertFalse(self.ef.check(["butter"], ["flour", "buttermilk"]).violated)

        self.assertTrue(self.ef.check(["cream"], ["pasta", "heavy cream"]).violated)
        self.assertFalse(self.ef.check(["cream"], ["egg white", "cream of tartar"]).violated)

class TestMultiCriteriaReranker(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        cls.reranker = MultiCriteriaReranker(processed_dir=PROCESSED_DIR)

    def test_prd_section25_exact_formula(self) -> None:
        # PRD §25 numerical example:
        # 0.30(0.78) + 0.35(0.90) + 0.20(0.96) - 0.15(0.10) = 0.726
        # Construct 10 unweighted ingredients (9 matched, 1 missing -> cov=0.90, miss=0.10)
        # and calorie=480 with target=500 -> cal_score=0.96
        user_ings = ["chicken", "potato", "carrot", "broccoli", "spinach", "zucchini", "celery", "beef", "tomato"]
        recipe_ings = user_ings + ["salmon"]  # 9/10 = 0.90 unweighted
        query = StructuredQuery(
            available_ingredients=user_ings,
            excluded_ingredients=["cheese"],
            calorie_target=500,
        )
        cand = CandidateRecipe(
            recipe_id="prd_ex",
            title="Candidate A",
            canonical_ingredients=recipe_ings,
            tfidf_score=0.78,
            total_calories=480.0,
        )
        cfg = RankingConfig(use_pantry_staple_weights=False, use_weighted_missing_penalty=False)
        scored = self.reranker.score_candidate(cand, query, config=cfg)

        self.assertAlmostEqual(scored.coverage.coverage_score, 0.90, places=4)
        self.assertAlmostEqual(scored.coverage.missing_penalty, 0.10, places=4)
        self.assertAlmostEqual(scored.calorie.score, 0.96, places=4)
        self.assertFalse(scored.exclusion.violated)
        self.assertAlmostEqual(scored.final_score, 0.726, places=4)

    def test_reranker_beats_tfidf_baseline_on_pantry_usability(self) -> None:
        query = StructuredQuery(
            available_ingredients=["chicken", "potato", "carrot", "onion", "garlic"],
            excluded_ingredients=["cheese"],
            calorie_target=500,
        )
        candidates = [
            # Highest TF-IDF, but has excluded cheese!
            CandidateRecipe(
                recipe_id=1,
                title="Cheesy Chicken Bake",
                canonical_ingredients=["chicken", "potato", "cheddar cheese", "cream"],
                tfidf_score=0.95,
                total_calories=510,
            ),
            # High TF-IDF, but needs 5 extra main ingredients and way off calories
            CandidateRecipe(
                recipe_id=2,
                title="Loaded Feast Casserole",
                canonical_ingredients=[
                    "chicken", "potato", "carrot", "bacon", "sour cream",
                    "mushroom", "celery", "broccoli", "pasta",
                ],
                tfidf_score=0.88,
                total_calories=1200,
            ),
            # Slightly lower TF-IDF, but high coverage (only missing celery) and ~500 kcal!
            CandidateRecipe(
                recipe_id=3,
                title="Chicken Potato Soup",
                canonical_ingredients=["chicken", "potato", "carrot", "onion", "garlic", "celery"],
                tfidf_score=0.75,
                total_calories=482,
            ),
        ]

        baseline_res = self.reranker.rerank(candidates, query, top_k=2, config=RankingConfig.ablation_model_a())
        proposed_res = self.reranker.rerank(candidates, query, top_k=2, config=RankingConfig.ablation_model_d())

        self.assertEqual(baseline_res.recommendations[0].recipe.recipe_id, 1)
        self.assertEqual(proposed_res.recommendations[0].recipe.recipe_id, 3)
        self.assertEqual(proposed_res.filtered_out_exclusion, 1)
        self.assertFalse(proposed_res.is_fallback)
        # Verify cheese recipe is excluded
        returned_ids = [r.recipe.recipe_id for r in proposed_res.recommendations]
        self.assertNotIn(1, returned_ids)

    def test_risk6_fallback_relaxes_calorie_first_then_exclusion(self) -> None:
        query = StructuredQuery(
            available_ingredients=["chicken", "rice"],
            excluded_ingredients=["cheese"],
            calorie_max=300,
        )
        candidates = [
            # Violates calorie_max (450 > 300), but respects exclusion
            CandidateRecipe(
                recipe_id="cal_over",
                title="Chicken Rice Bowl",
                canonical_ingredients=["chicken", "rice", "salt"],
                tfidf_score=0.80,
                total_calories=450,
            ),
            # Violates exclusion (cheese), even though calories <= 300
            CandidateRecipe(
                recipe_id="has_cheese",
                title="Cheesy Chicken Rice",
                canonical_ingredients=["chicken", "rice", "cheddar cheese"],
                tfidf_score=0.90,
                total_calories=280,
            ),
        ]

        res = self.reranker.rerank(candidates, query, top_k=1)
        self.assertTrue(res.is_fallback)
        self.assertIn("No exact recommendation found", res.fallback_message or "")
        self.assertEqual(res.relaxed_constraints, ["calorie_bounds"])
        # Tier 1 must pick calorie-over candidate rather than cheese-violating candidate!
        self.assertEqual(res.recommendations[0].recipe.recipe_id, "cal_over")
        self.assertTrue(res.recommendations[0].is_fallback_candidate)

        # Tier 2: if ALL candidates violate exclusions, still returns closest alternative with explicit warning
        res_tier2 = self.reranker.rerank([candidates[1]], query, top_k=1)
        self.assertTrue(res_tier2.is_fallback)
        self.assertIn("excluded_ingredients", res_tier2.relaxed_constraints)
        self.assertEqual(res_tier2.recommendations[0].recipe.recipe_id, "has_cheese")

    def test_explainable_formatting_and_serialization(self) -> None:
        query = StructuredQuery(
            available_ingredients=["chicken", "potato", "carrot", "onion", "garlic"],
            excluded_ingredients=["cheese"],
            calorie_target=500,
            raw_query="I have chicken, potato, carrot, onion and garlic. Around 500 calories. No cheese.",
        )
        cand = CandidateRecipe(
            recipe_id=123,
            title="Chicken Potato Soup",
            canonical_ingredients=["chicken", "potato", "carrot", "onion", "garlic", "celery", "salt"],
            tfidf_score=0.78,
            total_calories=482.0,
            ingredients=["2 chicken breasts", "2 potatoes", "1 carrot", "1 onion", "2 cloves garlic", "1 stalk celery", "1 tsp salt"],
            directions=["Combine all ingredients in a pot.", "Simmer 30 minutes."],
        )
        result = self.reranker.rerank([cand], query, top_k=3)
        card = format_recommendation_card(
            result.recommendations[0],
            excluded_ingredients=query.excluded_ingredients,
        )
        full_text = format_recommendations(result)
        payload = to_dict(result)

        self.assertIn("#1 Chicken Potato Soup", card)
        self.assertIn("Overall Match", card)
        self.assertIn("Ingredient Coverage", card)
        self.assertIn("Estimated Calories  : 482 kcal", card)
        self.assertIn("YOU HAVE:", card)
        self.assertIn("MISSING:", card)
        self.assertIn("celery", card)
        self.assertIn("salt (pantry staple)", card)
        self.assertIn("no cheese detected", card)
        self.assertIn("no cheese detected", full_text)
        self.assertIn("DIRECTIONS:", full_text)
        self.assertEqual(payload["recommendations"][0]["recipe_id"], 123)

    def test_pipeline1_and_pipeline2_raw_dict_contract(self) -> None:
        # Exact output schema of Pipeline 1 parse_query() (pipeline1-ibay)
        p1_dict = {
            "available_ingredients": ["chicken", "potato", "carrot", "onion", "garlic"],
            "excluded_ingredients": ["cheese"],
            "calorie_target": None,
            "calorie_min": None,
            "calorie_max": 600,
        }
        # Exact output schema of Pipeline 2 CandidateRetriever.retrieve() (pipeline3-hakim)
        p2_candidates = [
            {
                "recipe_id": 101,
                "title": "Unmapped Zero-Calorie Recipe",
                "ingredients": ["1 mystery spice"],
                "canonical_ingredients": ["chicken", "potato"],
                "directions": ["Cook."],
                "total_calories": 0.0,
                "calorie_coverage": 0.0,
                "tfidf_score": 0.92,
            },
            {
                "recipe_id": 102,
                "title": "Cheesy Chicken Gratin",
                "ingredients": ["2 chicken breasts", "1 cup cheddar cheese"],
                "canonical_ingredients": ["chicken", "cheddar cheese", "potato"],
                "directions": ["Bake."],
                "total_calories": 520.0,
                "calorie_coverage": 1.0,
                "tfidf_score": 0.90,
            },
            {
                "recipe_id": 103,
                "title": "Simple Chicken Potato Soup",
                "ingredients": ["2 chicken breasts", "2 potatoes", "1 carrot", "1 onion", "2 cloves garlic"],
                "canonical_ingredients": ["chicken", "potato", "carrot", "onion", "garlic", "salt"],
                "directions": ["Boil and simmer.", "Serves 4."],
                "total_calories": 490.0,
                "calorie_coverage": 0.95,
                "tfidf_score": 0.81,
            },
            {
                "recipe_id": 104,
                "title": "NaN Calorie Edge Case",
                "ingredients": ["1 chicken"],
                "canonical_ingredients": ["chicken"],
                "directions": ["Roast."],
                "total_calories": float("nan"),
                "calorie_coverage": 0.0,
                "tfidf_score": 0.70,
            },
        ]

        res = recommend(p2_candidates, p1_dict, top_k=3, reranker=self.reranker)
        # Strict match is ONLY recipe 103 (101 & 104 have unavailable calories under max=600, 102 has cheese)
        self.assertEqual(res.strict_match_count, 1)
        self.assertEqual(res.filtered_out_exclusion, 1)
        self.assertEqual(res.filtered_out_calorie, 2)
        self.assertEqual(res.recommendations[0].recipe.recipe_id, 103)
        self.assertFalse(res.recommendations[0].is_fallback_candidate)

        payload = to_dict(res)
        self.assertAlmostEqual(payload["recommendations"][0]["calorie_coverage"], 0.95)
        self.assertEqual(payload["recommendations"][0]["estimated_calories"], 490.0)

    def test_optional_per_serving_normalization_from_directions(self) -> None:
        query = {
            "available_ingredients": ["chicken", "potato"],
            "excluded_ingredients": [],
            "calorie_target": 500,
            "calorie_min": None,
            "calorie_max": None,
        }
        cand = {
            "recipe_id": 201,
            "title": "Family Roast Chicken",
            "ingredients": ["1 whole chicken", "4 potatoes"],
            "canonical_ingredients": ["chicken", "potato"],
            "directions": ["Roast for 1 hour.", "Serves 4."],
            "total_calories": 2000.0,
            "calorie_coverage": 1.0,
            "tfidf_score": 0.80,
        }
        cfg_total = RankingConfig(normalize_servings_from_directions=False)
        cfg_serving = RankingConfig(normalize_servings_from_directions=True)

        scored_total = self.reranker.score_candidate(cand, query, config=cfg_total)
        scored_serving = self.reranker.score_candidate(cand, query, config=cfg_serving)

        # Whole recipe (2000 vs target 500 -> score 0.0); per serving (2000/4 = 500 -> score 1.0)
        self.assertEqual(scored_total.calorie.estimated_calories, 2000.0)
        self.assertEqual(scored_total.calorie.score, 0.0)
        self.assertEqual(scored_serving.calorie.estimated_calories, 500.0)
        self.assertEqual(scored_serving.calorie.score, 1.0)

class TestCalorieEstimator(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        cls.estimator = RecipeCalorieEstimator(processed_dir=PROCESSED_DIR)

    def test_ingredient_line_parser(self) -> None:
        p1 = parse_ingredient_line("2 Tbsp. butter")
        self.assertAlmostEqual(p1.quantity or 0, 2.0)
        self.assertEqual(p1.unit, "tbsp")
        self.assertEqual(p1.ingredient, "butter")

        p2 = parse_ingredient_line("1 1/2 c. sugar")
        self.assertAlmostEqual(p2.quantity or 0, 1.5)
        self.assertEqual(p2.unit, "cup")
        self.assertEqual(p2.ingredient, "sugar")

        p3 = parse_ingredient_line("1 (16 oz.) can whole cranberry sauce")
        self.assertAlmostEqual(p3.quantity or 0, 16.0)
        self.assertEqual(p3.unit, "oz")
        self.assertIn("cranberry sauce", p3.ingredient.lower())

        p4 = parse_ingredient_line("1/2 tsp. salt")
        self.assertAlmostEqual(p4.quantity or 0, 0.5)
        self.assertEqual(p4.unit, "tsp")
        self.assertEqual(p4.ingredient, "salt")

    def test_extract_servings_and_recipe_estimation(self) -> None:
        self.assertEqual(extract_servings(["Bake 1 hour.", "Serves 16."]), 16)
        self.assertEqual(extract_servings(["Makes 4 to 6 servings."]), 5)

        est = self.estimator.estimate_recipe(
            raw_ingredients=["200 g chicken breast", "200 g potato", "1 tbsp olive oil", "1 tsp salt"],
            directions=["Cook chicken and potatoes in olive oil.", "Serves 2."],
        )
        self.assertIsNotNone(est.total_calories)
        self.assertEqual(est.estimated_servings, 2)
        self.assertEqual(est.servings_source, "parsed")
        # 200g chicken breast (~344) + 200g potato (~154) + 1 tbsp olive oil (~120) + salt (0) ~= 618 kcal
        self.assertGreater(est.total_calories or 0, 450)
        self.assertLess(est.total_calories or 0, 800)
        self.assertAlmostEqual(est.calories_per_serving or 0, (est.total_calories or 0) / 2, delta=0.1)

class TestSampleCorpusIntegration(unittest.TestCase):
    def test_end_to_end_on_sample_recipes_and_annotated_queries(self) -> None:
        self.assertTrue(SAMPLE_CSV.exists(), "recipes_sample.csv must exist")
        self.assertTrue(ANNOTATED_JSON.exists(), "annotated_queries.json must exist")

        normalizer = IngredientNormalizer(PROCESSED_DIR)
        estimator = RecipeCalorieEstimator(processed_dir=PROCESSED_DIR, normalizer=normalizer)
        reranker = MultiCriteriaReranker(processed_dir=PROCESSED_DIR, normalizer=normalizer)

        # Load first 100 sample recipes and estimate calories
        candidates: list[CandidateRecipe] = []
        with SAMPLE_CSV.open(encoding="utf-8", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh)):
                if idx >= 100:
                    break
                raw_ings = ast.literal_eval(row["ingredients"])
                ner_ings = ast.literal_eval(row["NER"])
                directions = ast.literal_eval(row["directions"])
                est = estimator.estimate_recipe(raw_ings, ner_ings, directions, title=row["title"])
                candidates.append(
                    CandidateRecipe(
                        recipe_id=idx,
                        title=row["title"],
                        canonical_ingredients=normalizer.canonicalize_list(ner_ings),
                        tfidf_score=0.5,
                        ingredients=raw_ings,
                        directions=directions,
                        total_calories=est.calories_per_serving,
                    )
                )

        # Verify high calorie estimation coverage on real RecipeNLG sample rows
        with_cals = sum(1 for c in candidates if c.total_calories is not None)
        self.assertGreaterEqual(with_cals, 95)

        # Run q041 from annotated_queries.json ("chicken, potato, carrot, onion, garlic; around 500; no cheese")
        data = json.loads(ANNOTATED_JSON.read_text(encoding="utf-8"))
        q041 = next(q for q in data["queries"] if q["id"] == "q041")
        sq = StructuredQuery.from_dict(q041["expected"], raw_query=q041["query"])
        result = reranker.rerank(candidates, sq, top_k=3)

        self.assertEqual(len(result.recommendations), 3)
        for rec in result.recommendations:
            if not rec.is_fallback_candidate:
                self.assertFalse(rec.exclusion.violated)


In [ ]:
vocab = load_vocabulary()    # pengganti fixture pytest scope-module

In [ ]:
def run_all_tests() -> None:
    """Jalankan seluruh test suite: fungsi test_* bergaya pytest + kelas unittest.TestCase."""
    import traceback

    passed, failed = [], []
    test_fns = [obj for name, obj in list(globals().items())
                if name.startswith("test_") and callable(obj)
                and getattr(obj, "__module__", None) == "__main__"
                and not isinstance(obj, type)]
    for fn in test_fns:
        try:
            fn()
            passed.append(fn.__name__)
        except Exception:
            failed.append((fn.__name__, traceback.format_exc(limit=3)))

    import unittest
    classes = [obj for name, obj in list(globals().items())
               if isinstance(obj, type) and issubclass(obj, unittest.TestCase)
               and obj.__module__ == "__main__"]
    loader = unittest.TestLoader()
    suite = unittest.TestSuite(loader.loadTestsFromTestCase(c) for c in classes)
    result = unittest.TextTestRunner(verbosity=2).run(suite)

    print(f"\npytest-style tests: {len(passed)} passed, {len(failed)} failed")
    for name, tb in failed:
        print(f"  FAIL {name}\n{tb}")
    print(f"unittest tests:     {result.testsRun} run, "
          f"{len(result.failures)} failures, {len(result.errors)} errors")
    ok = not failed and not result.failures and not result.errors
    print(f"\n{'ALL TESTS PASSED' if ok else 'SOME TESTS FAILED'}")
    assert ok, "ada test yang gagal"


run_all_tests()

## 8. Demo End-to-End

Pantry bebas -> Pipeline 1 (parse + trace) -> Pipeline 2 (retrieval TF-IDF; dense bila
tersedia) -> Pipeline 3 (rerank multi-kriteria) -> kartu rekomendasi.


In [ ]:
demo_query = "I have chicken, rice, onion, and garlic. Around 600 calories and no peanuts."

# Pipeline 1: parse + trace
trace = trace_parse_query(demo_query, get_approach("difflib-0.82"), vocab)
sq = trace.structured_query
print("structured query:", sq)
print("matched :", [(m.surface, m.canonical, m.method) for m in trace.available_trace.matches])
print("excluded:", trace.excluded_trace.ingredients)

# Pipeline 2: retrieval kandidat
tfidf_approach = TfidfApproach()
candidates = tfidf_approach.retrieve(sq, top_n=50)
print(f"\nkandidat TF-IDF: {len(candidates)} | teratas: {candidates[0]['title']!r}")

# Pipeline 3: rerank multi-kriteria -> kartu rekomendasi
result = recommend(candidates, sq, top_k=3, processed_dir=DEFAULT_PROCESSED_DIR)
print(format_recommendations(result))

In [ ]:
# Perbandingan retrieval TF-IDF vs dense (bila model sentence-transformers tersedia)
try:
    dense_approach = get_retrieval_approach("dense")
    dense_candidates = dense_approach.retrieve(sq, top_n=50)
    result_dense = recommend(dense_candidates, sq, top_k=3, processed_dir=DEFAULT_PROCESSED_DIR)
    print("=== Dense retrieval (top-3) ===")
    print("\n".join(format_recommendations(result_dense).splitlines()[:12]))
    print("\n=== TF-IDF retrieval (top-3) ===")
    print("\n".join(format_recommendations(result).splitlines()[:12]))
except Exception as exc:  # model/language hub tidak tersedia
    print(f"dense approach dilewati: {exc}")